# New Section

In [ ]:
#!/usr/bin/env python3
"""
AI Stock Analyzer  (A to Z)
---------------------------
Run:   python stock_analyzer.py
Then type a stock name, e.g.  RELIANCE   TCS   "tata motors"   AAPL

Does:
  1. Fundamental analysis  (valuation, profitability, growth, debt, cash flow, analysts)
  2. Technical analysis    (trend, RSI, MACD, Bollinger, ADX, ATR, volume, support/resistance)
  3. News sentiment        (simple keyword based)
  4. Final verdict         (STRONG BUY / BUY / HOLD / SELL / STRONG SELL)
  5. Trade plan            (buy zone, stop loss, 3 targets, trailing stop, position size)
  6. Exit rules            (when to sell)
  7. Optional Claude AI summary (if ANTHROPIC_API_KEY is set)
  8. Chart saved as PNG

Educational tool only. NOT financial advice. Always do your own research.
"""

import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

# ======================= YOUR SETTINGS =======================
CAPITAL = 100000            # your total trading capital (in your currency)
RISK_PER_TRADE_PCT = 1.0    # risk max 1% of capital per trade
ATR_STOP_MULT = 2.0         # stop loss = entry - 2 x ATR
WEIGHT_FUNDAMENTAL = 0.40   # final score weight
WEIGHT_TECHNICAL = 0.60
SAVE_CHART = True
# =============================================================


# ----------------------- helpers -----------------------
def num(x):
    try:
        if x is None:
            return None
        v = float(x)
        return None if (math.isnan(v) or math.isinf(v)) else v
    except Exception:
        return None


def fmt(x, d=2, suffix=""):
    v = num(x)
    return "N/A" if v is None else f"{v:,.{d}f}{suffix}"


def pct(x, d=1):
    v = num(x)
    return "N/A" if v is None else f"{v * 100:.{d}f}%"


def big_money(x, currency):
    v = num(x)
    if v is None:
        return "N/A"
    if currency == "INR":
        return f"Rs {v / 1e7:,.0f} Cr"
    if v >= 1e12:
        return f"{v / 1e12:.2f} T {currency}"
    if v >= 1e9:
        return f"{v / 1e9:.2f} B {currency}"
    return f"{v / 1e6:.1f} M {currency}"


def line(char="=", n=78):
    print(char * n)


def title(text):
    print()
    line()
    print(f"  {text}")
    line()


def resolve_symbol(user_input):
    """Turn 'reliance' / 'tata motors' / 'AAPL' into a valid Yahoo symbol."""
    raw = user_input.strip()
    s = raw.upper().replace(" ", "")
    if "." in s or s.startswith("^"):
        candidates = [s]
    else:
        candidates = [s + ".NS", s + ".BO", s]  # NSE, BSE, then as-is (US etc.)
    for c in candidates:
        try:
            if not yf.Ticker(c).history(period="5d").empty:
                return c
        except Exception:
            pass
    # fall back to name search
    try:
        quotes = yf.Search(raw, max_results=8).quotes
        for q in quotes:
            sym = q.get("symbol", "")
            if sym.endswith(".NS") or sym.endswith(".BO"):
                return sym
        if quotes:
            return quotes[0]["symbol"]
    except Exception:
        pass
    return None


def score_block(rows):
    """rows = list of dicts with pts and mx. Returns 0-100 score."""
    if not rows:
        return 50.0
    total_pts = sum(r["pts"] for r in rows)
    total_max = sum(r["mx"] for r in rows)
    if total_max == 0:
        return 50.0
    return round((total_pts + total_max) / (2 * total_max) * 100, 1)


def print_rows(rows):
    for r in rows:
        sign = "+" if r["pts"] > 0 else ("-" if r["pts"] < 0 else "=")
        print(f"  [{sign}] {r['name']:<24} {r['value']:<16} {r['note']}")


# ----------------------- fundamentals -----------------------
def analyze_fundamentals(info, price):
    rows = []

    def add(name, value, pts, mx, note):
        rows.append({"name": name, "value": value, "pts": pts, "mx": mx, "note": note})

    sector = info.get("sector", "") or ""
    is_financial = "Financial" in sector

    # Valuation
    pe = num(info.get("trailingPE"))
    if pe is not None:
        if pe <= 0:
            add("P/E (trailing)", fmt(pe), -2, 2, "Company is loss making")
        elif pe < 15:
            add("P/E (trailing)", fmt(pe), 2, 2, "Cheap valuation")
        elif pe < 25:
            add("P/E (trailing)", fmt(pe), 1, 2, "Fair valuation")
        elif pe < 40:
            add("P/E (trailing)", fmt(pe), 0, 2, "Slightly expensive")
        else:
            add("P/E (trailing)", fmt(pe), -1, 2, "Expensive, needs high growth")

    fpe = num(info.get("forwardPE"))
    if fpe is not None and pe is not None and pe > 0 and fpe > 0:
        if fpe < pe * 0.9:
            add("Forward P/E", fmt(fpe), 1, 1, "Earnings expected to grow")
        elif fpe > pe * 1.1:
            add("Forward P/E", fmt(fpe), -1, 1, "Earnings expected to fall")

    peg = num(info.get("trailingPegRatio") or info.get("pegRatio"))
    if peg is not None and peg > 0:
        if peg < 1:
            add("PEG ratio", fmt(peg), 2, 2, "Growth is cheap")
        elif peg < 2:
            add("PEG ratio", fmt(peg), 0, 2, "Fairly priced for growth")
        else:
            add("PEG ratio", fmt(peg), -1, 2, "Pricey for growth")

    pb = num(info.get("priceToBook"))
    if pb is not None and pb > 0:
        limit_low, limit_high = (1.5, 3.5) if is_financial else (3, 7)
        if pb < limit_low:
            add("Price / Book", fmt(pb), 1, 1, "Trading near book value")
        elif pb > limit_high:
            add("Price / Book", fmt(pb), -1, 1, "High premium to book")

    # Profitability
    roe = num(info.get("returnOnEquity"))
    if roe is not None:
        if roe > 0.20:
            add("ROE", pct(roe), 2, 2, "Excellent")
        elif roe > 0.15:
            add("ROE", pct(roe), 1, 2, "Good")
        elif roe > 0.08:
            add("ROE", pct(roe), 0, 2, "Average")
        else:
            add("ROE", pct(roe), -2, 2, "Weak")

    roa = num(info.get("returnOnAssets"))
    if roa is not None and not is_financial:
        if roa > 0.10:
            add("ROA", pct(roa), 1, 1, "Efficient asset use")
        elif roa < 0.03:
            add("ROA", pct(roa), -1, 1, "Low asset efficiency")

    pm = num(info.get("profitMargins"))
    if pm is not None:
        if pm > 0.15:
            add("Net profit margin", pct(pm), 2, 2, "High margin business")
        elif pm > 0.08:
            add("Net profit margin", pct(pm), 1, 2, "Decent margin")
        elif pm > 0.03:
            add("Net profit margin", pct(pm), 0, 2, "Thin margin")
        else:
            add("Net profit margin", pct(pm), -1, 2, "Very thin / negative")

    om = num(info.get("operatingMargins"))
    if om is not None and om > 0.20:
        add("Operating margin", pct(om), 1, 1, "Strong operations")
    elif om is not None and om < 0.05:
        add("Operating margin", pct(om), -1, 1, "Weak operations")

    # Growth
    rg = num(info.get("revenueGrowth"))
    if rg is not None:
        if rg > 0.15:
            add("Revenue growth (YoY)", pct(rg), 2, 2, "Fast growth")
        elif rg > 0.08:
            add("Revenue growth (YoY)", pct(rg), 1, 2, "Healthy growth")
        elif rg >= 0:
            add("Revenue growth (YoY)", pct(rg), 0, 2, "Slow growth")
        else:
            add("Revenue growth (YoY)", pct(rg), -2, 2, "Shrinking sales")

    eg = num(info.get("earningsGrowth") or info.get("earningsQuarterlyGrowth"))
    if eg is not None:
        if eg > 0.15:
            add("Earnings growth", pct(eg), 2, 2, "Profits rising fast")
        elif eg > 0:
            add("Earnings growth", pct(eg), 1, 2, "Profits rising")
        else:
            add("Earnings growth", pct(eg), -2, 2, "Profits falling")

    # Financial health
    de = num(info.get("debtToEquity"))  # yfinance gives percent (45 = 0.45x)
    if de is not None and not is_financial:
        if de < 50:
            add("Debt / Equity", fmt(de / 100), 2, 2, "Low debt")
        elif de < 100:
            add("Debt / Equity", fmt(de / 100), 1, 2, "Manageable debt")
        elif de < 200:
            add("Debt / Equity", fmt(de / 100), -1, 2, "High debt")
        else:
            add("Debt / Equity", fmt(de / 100), -2, 2, "Very high debt")

    cr = num(info.get("currentRatio"))
    if cr is not None and not is_financial:
        if cr >= 1.5:
            add("Current ratio", fmt(cr), 1, 1, "Strong liquidity")
        elif cr < 1:
            add("Current ratio", fmt(cr), -1, 1, "Liquidity risk")

    fcf = num(info.get("freeCashflow"))
    if fcf is not None:
        if fcf > 0:
            add("Free cash flow", big_money(fcf, info.get("financialCurrency", "")), 1, 1, "Generates cash")
        else:
            add("Free cash flow", big_money(fcf, info.get("financialCurrency", "")), -1, 1, "Burning cash")

    # Dividend
    div_rate = num(info.get("dividendRate"))
    if div_rate and price:
        dy = div_rate / price
        if dy > 0.02:
            add("Dividend yield", pct(dy), 1, 1, "Pays good dividend")

    # Analysts
    target = num(info.get("targetMeanPrice"))
    if target and price:
        upside = (target - price) / price
        if upside > 0.15:
            add("Analyst target upside", pct(upside), 1, 1, f"Target {fmt(target)}")
        elif upside < -0.05:
            add("Analyst target upside", pct(upside), -1, 1, f"Target {fmt(target)}")
        else:
            add("Analyst target upside", pct(upside), 0, 1, f"Target {fmt(target)}")

    return rows, score_block(rows)


# ----------------------- technicals -----------------------
def compute_indicators(df):
    d = df.copy()
    c, h, l, v = d["Close"], d["High"], d["Low"], d["Volume"]

    for n in (20, 50, 200):
        d[f"SMA{n}"] = c.rolling(n).mean()
    d["EMA20"] = c.ewm(span=20, adjust=False).mean()

    # RSI (Wilder)
    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_g = gain.ewm(alpha=1 / 14, adjust=False).mean()
    avg_l = loss.ewm(alpha=1 / 14, adjust=False).mean()
    rs = avg_g / avg_l.replace(0, np.nan)
    d["RSI"] = 100 - 100 / (1 + rs)

    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    d["MACD"] = ema12 - ema26
    d["MACD_SIG"] = d["MACD"].ewm(span=9, adjust=False).mean()
    d["MACD_HIST"] = d["MACD"] - d["MACD_SIG"]

    # ATR
    prev_c = c.shift()
    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    d["ATR"] = tr.ewm(alpha=1 / 14, adjust=False).mean()

    # Bollinger
    std = c.rolling(20).std()
    d["BB_UP"] = d["SMA20"] + 2 * std
    d["BB_LO"] = d["SMA20"] - 2 * std

    # ADX
    up, down = h.diff(), -l.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=d.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=d.index)
    plus_di = 100 * plus_dm.ewm(alpha=1 / 14, adjust=False).mean() / d["ATR"]
    minus_di = 100 * minus_dm.ewm(alpha=1 / 14, adjust=False).mean() / d["ATR"]
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    d["PLUS_DI"], d["MINUS_DI"] = plus_di, minus_di
    d["ADX"] = dx.ewm(alpha=1 / 14, adjust=False).mean()

    d["VOL20"] = v.rolling(20).mean()
    d["VOL5"] = v.rolling(5).mean()
    return d


def analyze_technicals(d):
    last, prev = d.iloc[-1], d.iloc[-2]
    price = last["Close"]
    rows = []

    def add(name, value, pts, mx, note):
        rows.append({"name": name, "value": value, "pts": pts, "mx": mx, "note": note})

    sma20, sma50, sma200 = num(last["SMA20"]), num(last["SMA50"]), num(last["SMA200"])

    if sma200 is not None:
        if price > sma200:
            add("Price vs 200 DMA", fmt(sma200), 2, 2, "Long-term uptrend")
        else:
            add("Price vs 200 DMA", fmt(sma200), -2, 2, "Long-term downtrend")
    if sma50 is not None:
        if price > sma50:
            add("Price vs 50 DMA", fmt(sma50), 1, 1, "Medium-term bullish")
        else:
            add("Price vs 50 DMA", fmt(sma50), -1, 1, "Medium-term bearish")
    if sma50 is not None and sma200 is not None:
        if sma50 > sma200:
            add("50 DMA vs 200 DMA", "Golden", 1, 1, "50 DMA above 200 DMA")
        else:
            add("50 DMA vs 200 DMA", "Death", -1, 1, "50 DMA below 200 DMA")
    if sma20 is not None:
        if price > sma20:
            add("Price vs 20 DMA", fmt(sma20), 1, 1, "Short-term bullish")
        else:
            add("Price vs 20 DMA", fmt(sma20), -1, 1, "Short-term bearish")

    rsi = num(last["RSI"])
    if rsi is not None:
        if rsi < 30:
            add("RSI (14)", fmt(rsi, 1), 2, 2, "Oversold, bounce likely")
        elif rsi < 45:
            add("RSI (14)", fmt(rsi, 1), 0, 2, "Weak momentum")
        elif rsi <= 65:
            add("RSI (14)", fmt(rsi, 1), 1, 2, "Healthy momentum")
        elif rsi <= 75:
            add("RSI (14)", fmt(rsi, 1), 0, 2, "Strong but getting hot")
        else:
            add("RSI (14)", fmt(rsi, 1), -2, 2, "Overbought, pullback risk")

    macd, sig = num(last["MACD"]), num(last["MACD_SIG"])
    if macd is not None and sig is not None:
        if macd > sig and prev["MACD"] <= prev["MACD_SIG"]:
            add("MACD", "Bull cross", 2, 2, "Fresh bullish crossover")
        elif macd > sig:
            add("MACD", "Bullish", 1, 2, "MACD above signal")
        elif macd < sig and prev["MACD"] >= prev["MACD_SIG"]:
            add("MACD", "Bear cross", -2, 2, "Fresh bearish crossover")
        else:
            add("MACD", "Bearish", -1, 2, "MACD below signal")

    bb_up, bb_lo = num(last["BB_UP"]), num(last["BB_LO"])
    if bb_up is not None and bb_lo is not None:
        if price < bb_lo:
            add("Bollinger Bands", "Below lower", 1, 1, "Stretched down, mean reversion")
        elif price > bb_up:
            add("Bollinger Bands", "Above upper", -1, 1, "Stretched up, may cool off")

    adx = num(last["ADX"])
    if adx is not None:
        bullish_di = last["PLUS_DI"] > last["MINUS_DI"]
        if adx > 25:
            add("ADX trend strength", fmt(adx, 1), 1 if bullish_di else -1, 1,
                "Strong uptrend" if bullish_di else "Strong downtrend")
        else:
            add("ADX trend strength", fmt(adx, 1), 0, 1, "Sideways / weak trend")

    v5, v20 = num(last["VOL5"]), num(last["VOL20"])
    if v5 and v20:
        up_move = price > d["Close"].iloc[-6]
        if v5 > v20 * 1.2 and up_move:
            add("Volume", f"{v5 / v20:.2f}x avg", 1, 1, "Rising price on rising volume")
        elif v5 > v20 * 1.2 and not up_move:
            add("Volume", f"{v5 / v20:.2f}x avg", -1, 1, "Falling price on rising volume")

    support20 = float(d["Low"].tail(20).min())
    support60 = float(d["Low"].tail(60).min())
    resistance20 = float(d["High"].tail(20).max())
    resistance60 = float(d["High"].tail(60).max())
    hi52 = float(d["High"].tail(252).max())
    lo52 = float(d["Low"].tail(252).min())

    levels = {
        "support20": support20, "support60": support60,
        "resistance20": resistance20, "resistance60": resistance60,
        "hi52": hi52, "lo52": lo52,
    }
    return rows, score_block(rows), levels


# ----------------------- news sentiment -----------------------
POSITIVE = ["surge", "jump", "rally", "gain", "profit", "beat", "record", "growth", "upgrade", "buy",
            "strong", "wins", "order", "expansion", "dividend", "bullish", "outperform", "soars", "rises"]
NEGATIVE = ["fall", "drop", "plunge", "loss", "miss", "decline", "downgrade", "sell", "weak", "probe",
            "fraud", "penalty", "lawsuit", "cut", "bearish", "underperform", "slump", "crash", "debt", "fine"]


def news_sentiment(ticker):
    headlines = []
    try:
        for item in (ticker.news or [])[:12]:
            t = item.get("title") or (item.get("content") or {}).get("title")
            if t:
                headlines.append(t)
    except Exception:
        pass
    if not headlines:
        return [], 0, "No news found"
    score = 0
    for h in headlines:
        low = h.lower()
        score += sum(w in low for w in POSITIVE) - sum(w in low for w in NEGATIVE)
    label = "Positive" if score > 1 else ("Negative" if score < -1 else "Neutral")
    return headlines, score, label


# ----------------------- quarterly trend -----------------------
def quarterly_trend(ticker):
    try:
        q = ticker.quarterly_income_stmt
        if q is None or q.empty:
            return None
        out = []
        for col in list(q.columns)[:4]:
            rev = num(q.loc["Total Revenue", col]) if "Total Revenue" in q.index else None
            ni = num(q.loc["Net Income", col]) if "Net Income" in q.index else None
            out.append((str(col.date()), rev, ni))
        return out
    except Exception:
        return None


# ----------------------- trade plan -----------------------
def build_trade_plan(d, levels, verdict, fund_score, tech_score):
    last = d.iloc[-1]
    price = float(last["Close"])
    atr = float(last["ATR"])
    ema20 = float(last["EMA20"])
    rsi = num(last["RSI"]) or 50

    # Entry logic
    overheated = rsi > 72 or price > ema20 + 2.5 * atr
    if overheated:
        entry = ema20
        entry_note = "Price is stretched. Wait for a pullback to the 20 EMA zone."
    else:
        entry = price
        entry_note = "Price is acceptable. Can enter near current market price."

    buy_zone = (ema20 - 0.5 * atr, ema20 + 0.5 * atr)
    breakout = levels["resistance60"] * 1.002

    # Stop loss: ATR stop, or just below swing support if that is tighter but sensible
    sl_atr = entry - ATR_STOP_MULT * atr
    sl_support = levels["support20"] - 0.25 * atr
    stop = sl_atr
    if sl_atr < sl_support < entry - 1.0 * atr:
        stop = sl_support

    risk = entry - stop
    t1, t2, t3 = entry + 1.5 * risk, entry + 2.5 * risk, entry + 4.0 * risk

    risk_amount = CAPITAL * RISK_PER_TRADE_PCT / 100
    qty = int(risk_amount // risk) if risk > 0 else 0
    max_qty_by_capital = int(CAPITAL // entry) if entry > 0 else 0
    qty = max(0, min(qty, max_qty_by_capital))

    trailing = float(d["Close"].tail(22).max()) - 3 * atr  # chandelier exit

    return {
        "price": price, "entry": entry, "entry_note": entry_note,
        "buy_zone": buy_zone, "breakout": breakout,
        "stop": stop, "risk": risk, "risk_pct": risk / entry * 100,
        "t1": t1, "t2": t2, "t3": t3,
        "qty": qty, "capital_used": qty * entry, "risk_amount": risk_amount,
        "trailing": trailing, "atr": atr,
    }


def decide(final_score, tech_score, rsi):
    if final_score >= 70:
        v = "STRONG BUY"
    elif final_score >= 58:
        v = "BUY"
    elif final_score >= 42:
        v = "HOLD / WAIT"
    elif final_score >= 30:
        v = "SELL / AVOID"
    else:
        v = "STRONG SELL"
    # safety overrides
    if v in ("STRONG BUY", "BUY") and tech_score < 40:
        v = "HOLD / WAIT"
    if v in ("STRONG BUY", "BUY") and rsi and rsi > 75:
        v = "WAIT FOR PULLBACK (overbought)"
    return v


# ----------------------- chart -----------------------
def save_chart(d, symbol, plan, verdict):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        x = d.tail(180)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                       gridspec_kw={"height_ratios": [3, 1]})
        ax1.plot(x.index, x["Close"], label="Close", color="black", linewidth=1.4)
        ax1.plot(x.index, x["SMA50"], label="SMA50", color="blue", linewidth=1)
        ax1.plot(x.index, x["SMA200"], label="SMA200", color="red", linewidth=1)
        ax1.fill_between(x.index, x["BB_LO"], x["BB_UP"], color="grey", alpha=0.12)
        ax1.axhline(plan["entry"], color="green", linestyle="--", label=f"Entry {plan['entry']:.2f}")
        ax1.axhline(plan["stop"], color="red", linestyle="--", label=f"Stop {plan['stop']:.2f}")
        for i, t in enumerate((plan["t1"], plan["t2"], plan["t3"]), 1):
            ax1.axhline(t, color="darkgreen", linestyle=":", label=f"T{i} {t:.2f}")
        ax1.set_title(f"{symbol}  |  {verdict}")
        ax1.legend(loc="upper left", fontsize=8, ncol=3)
        ax2.plot(x.index, x["RSI"], color="purple")
        ax2.axhline(70, color="red", linestyle=":")
        ax2.axhline(30, color="green", linestyle=":")
        ax2.set_ylabel("RSI")
        plt.tight_layout()
        fname = f"{symbol.replace('.', '_')}_analysis.png"
        plt.savefig(fname, dpi=110)
        plt.close()
        return fname
    except Exception as e:
        return None


# ----------------------- optional Claude AI summary -----------------------
def ai_summary(report_text):
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return None
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=key)
        msg = client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=900,
            messages=[{
                "role": "user",
                "content": (
                    "You are a careful equity analyst. Below is an automated analysis of a stock. "
                    "Write a short plain-English summary: 1) business quality, 2) key risks, "
                    "3) whether the trade plan looks sensible, 4) what would change the view. "
                    "Be balanced, no guarantees.\n\n" + report_text
                ),
            }],
        )
        return "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    except Exception as e:
        return f"(AI summary skipped: {e})"


# ----------------------- main analysis -----------------------
def analyze(user_input):
    symbol = resolve_symbol(user_input)
    if not symbol:
        print(f"Could not find a stock for '{user_input}'. Try the exact ticker (e.g. RELIANCE, TCS, AAPL).")
        return

    t = yf.Ticker(symbol)
    hist = t.history(period="2y", interval="1d", auto_adjust=True)
    if hist is None or len(hist) < 60:
        print("Not enough price history to analyze this stock.")
        return
    try:
        info = t.info or {}
    except Exception:
        info = {}

    d = compute_indicators(hist)
    price = float(d["Close"].iloc[-1])
    currency = info.get("currency", "")
    report = []  # collect key text for AI summary

    # ---------- Company overview ----------
    title(f"COMPANY OVERVIEW  -  {info.get('longName', symbol)} ({symbol})")
    print(f"  Sector / Industry : {info.get('sector', 'N/A')} / {info.get('industry', 'N/A')}")
    print(f"  Current price     : {fmt(price)} {currency}")
    print(f"  Market cap        : {big_money(info.get('marketCap'), currency)}")
    print(f"  52w High / Low    : {fmt(info.get('fiftyTwoWeekHigh'))} / {fmt(info.get('fiftyTwoWeekLow'))}")
    print(f"  Beta (volatility) : {fmt(info.get('beta'))}")
    print(f"  Insider/Promoter  : {pct(info.get('heldPercentInsiders'))}   "
          f"Institutions: {pct(info.get('heldPercentInstitutions'))}")
    print(f"  Analyst view      : {info.get('recommendationKey', 'N/A')} "
          f"({info.get('numberOfAnalystOpinions', 'N/A')} analysts)")
    summary = info.get("longBusinessSummary")
    if summary:
        print(f"\n  About: {summary[:450]}...")
    report.append(f"Stock: {info.get('longName', symbol)}, sector {info.get('sector')}, price {price:.2f} {currency}")

    # ---------- Fundamentals ----------
    f_rows, f_score = analyze_fundamentals(info, price)
    title(f"FUNDAMENTAL ANALYSIS   Score: {f_score}/100")
    if f_rows:
        print_rows(f_rows)
    else:
        print("  Fundamental data not available for this stock.")
    qt = quarterly_trend(t)
    if qt:
        print("\n  Last quarters (latest first):")
        for dt, rev, ni in qt:
            print(f"    {dt}   Revenue: {big_money(rev, info.get('financialCurrency', currency))}"
                  f"   Net profit: {big_money(ni, info.get('financialCurrency', currency))}")
    report.append("Fundamentals: " + "; ".join(f"{r['name']}={r['value']} ({r['note']})" for r in f_rows))

    # ---------- Technicals ----------
    t_rows, t_score, levels = analyze_technicals(d)
    title(f"TECHNICAL ANALYSIS   Score: {t_score}/100")
    print_rows(t_rows)
    print(f"\n  Support (20d / 60d)    : {fmt(levels['support20'])} / {fmt(levels['support60'])}")
    print(f"  Resistance (20d / 60d) : {fmt(levels['resistance20'])} / {fmt(levels['resistance60'])}")
    print(f"  52w High / Low         : {fmt(levels['hi52'])} / {fmt(levels['lo52'])}"
          f"   (price is {(price / levels['hi52'] - 1) * 100:.1f}% from high)")
    report.append("Technicals: " + "; ".join(f"{r['name']}={r['value']} ({r['note']})" for r in t_rows))

    # ---------- News ----------
    headlines, n_score, n_label = news_sentiment(t)
    title(f"NEWS SENTIMENT   {n_label} (raw {n_score:+d})")
    for h in headlines[:8]:
        print(f"  - {h}")
    if not headlines:
        print("  No recent headlines available.")

    # ---------- Final verdict ----------
    final = WEIGHT_FUNDAMENTAL * f_score + WEIGHT_TECHNICAL * t_score
    if n_label == "Positive":
        final += 2
    elif n_label == "Negative":
        final -= 2
    final = round(max(0, min(100, final)), 1)
    rsi = num(d["RSI"].iloc[-1])
    verdict = decide(final, t_score, rsi)
    plan = build_trade_plan(d, levels, verdict, f_score, t_score)

    title("FINAL RESULT")
    print(f"  Fundamental score : {f_score}/100   (weight {int(WEIGHT_FUNDAMENTAL * 100)}%)")
    print(f"  Technical score   : {t_score}/100   (weight {int(WEIGHT_TECHNICAL * 100)}%)")
    print(f"  News sentiment    : {n_label}")
    print(f"  COMBINED SCORE    : {final}/100")
    print(f"\n  >>>  VERDICT: {verdict}  <<<")

    # ---------- Trade plan ----------
    title("TRADE PLAN  (swing trade, days to weeks)")
    if verdict.startswith("SELL") or verdict.startswith("STRONG SELL"):
        print("  Do NOT buy now. If you already hold, consider exiting or tightening your stop.")
        print(f"  Re-check when price closes above 50 DMA ({fmt(d['SMA50'].iloc[-1])}) with rising volume.")
    else:
        print(f"  When to BUY:")
        print(f"    1) {plan['entry_note']}")
        print(f"    2) Pullback buy zone : {fmt(plan['buy_zone'][0])} to {fmt(plan['buy_zone'][1])}")
        print(f"    3) Breakout buy      : close above {fmt(plan['breakout'])} with volume above average")
        print(f"\n  Entry price      : {fmt(plan['entry'])}")
        print(f"  STOP LOSS        : {fmt(plan['stop'])}   (risk {plan['risk_pct']:.1f}% per share, "
              f"{ATR_STOP_MULT} x ATR={fmt(plan['atr'])})")
        print(f"  Target 1 (1:1.5) : {fmt(plan['t1'])}   -> sell 40%, move stop to entry")
        print(f"  Target 2 (1:2.5) : {fmt(plan['t2'])}   -> sell 30%, trail the rest")
        print(f"  Target 3 (1:4)   : {fmt(plan['t3'])}   -> sell remaining or trail")
        print(f"\n  Position sizing  : capital {CAPITAL:,.0f}, risk {RISK_PER_TRADE_PCT}% = {plan['risk_amount']:,.0f}")
        print(f"  Quantity to buy  : {plan['qty']} shares  (invest about {plan['capital_used']:,.0f})")
        print(f"  Trailing stop now: {fmt(plan['trailing'])}  (highest close 22d - 3 x ATR, only move UP)")

    print("\n  When to SELL (exit rules):")
    print(f"    - Price closes below stop loss {fmt(plan['stop'])}  -> exit full position, no excuses")
    print(f"    - Price hits targets T1/T2/T3 -> book profit in parts as above")
    print(f"    - Daily close below trailing stop -> exit")
    print(f"    - Close below 50 DMA ({fmt(d['SMA50'].iloc[-1])}) after a big rally -> exit swing trade")
    print(f"    - RSI above 75 and then turns down, or fresh MACD bearish crossover -> book profits")
    print(f"    - Bad news: fraud, regulatory action, big profit miss, promoter selling -> reduce/exit")

    print("\n  Key risk notes:")
    print(f"    - Never risk more than {RISK_PER_TRADE_PCT}% of capital on one trade.")
    print("    - Check results/earnings dates before entering; gaps can jump over stop loss.")
    print("    - Market direction matters: avoid new buys when Nifty/S&P is below its 200 DMA.")

    # ---------- chart ----------
    if SAVE_CHART:
        fname = save_chart(d, symbol, plan, verdict)
        if fname:
            print(f"\n  Chart saved: {fname}")

    # ---------- optional AI ----------
    report.append(f"Scores: fundamental {f_score}, technical {t_score}, news {n_label}, combined {final}, verdict {verdict}")
    report.append(f"Plan: entry {plan['entry']:.2f}, stop {plan['stop']:.2f}, targets {plan['t1']:.2f}/{plan['t2']:.2f}/{plan['t3']:.2f}")
    ai = ai_summary("\n".join(report))
    if ai:
        title("AI SUMMARY (Claude)")
        print(ai)

    print()
    line("-")
    print("  Educational tool only. Not financial advice. Markets are risky, you can lose money.")
    line("-")


def main():
    # Ignore hidden arguments that Colab/Jupyter add (e.g. "-f kernel.json")
    args = [a for a in sys.argv[1:] if not a.startswith("-") and not a.endswith(".json")]
    in_notebook = "ipykernel" in sys.modules
    if args and not in_notebook:
        analyze(" ".join(args))
        return
    print("AI Stock Analyzer  (type 'q' to quit)")
    while True:
        name = input("\nEnter stock name / symbol: ").strip()
        if name.lower() in ("q", "quit", "exit", ""):
            break
        try:
            analyze(name)
        except Exception as e:
            print(f"Error while analyzing: {e}")


if __name__ == "__main__":
    main()

# New Section

In [ ]:
# =============================================================================
#  MASTER PROMPT IMPLEMENTATION
#  Automated Portfolio Analyzer  (Indian stock market focused)
#  Target environment : Google Colab (with graceful fallback for local Jupyter)
#
#  HOW TO USE
#  ----------
#  1. Copy this ENTIRE file into a single Google Colab code cell (or a .py file).
#  2. Run it. It will:
#       a) install required dependencies,
#       b) prompt you to manually upload your holdings file (.xlsx/.xls/.csv/.pdf),
#       c) extract + validate + classify holdings,
#       d) collect available market data,
#       e) run per-asset-class analysis engines,
#       f) compute portfolio-level metrics,
#       g) generate ONE comprehensive PDF report covering every holding in full
#          detail (plus optional CSV/Excel companions) and show the download link.
#
#  DESIGN PRINCIPLES (per the master prompt)
#  -----------------------------------------
#  * Financial accuracy and transparent calculations over bold recommendations.
#  * Never fabricate quantities, prices, ratios, estimates or targets. Metrics
#    that cannot be sourced from reliable data are marked "N/A".
#  * Robust execution: a single failed lookup must never crash the run.
#  * Every uploaded row is accounted for; unidentified rows are flagged, not
#    silently dropped.
#  * No hard-coded file paths, filenames, tickers or portfolios.
# =============================================================================

import os
import sys
import io
import re
import csv
import gc
import json
import time
import math
import shutil
import hashlib
import logging
import warnings
import tempfile
import traceback
from datetime import datetime, date
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1. AUTOMATIC DEPENDENCY INSTALLATION
# -----------------------------------------------------------------------------
# Only packages we actually need. We avoid force-upgrading core Colab packages
# (numpy/pandas already present) unless import fails.
def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IS_COLAB = _running_in_colab()

def install_dependencies() -> None:
    """Install / verify required Python packages, handling failures gracefully."""
    # (import_name, pip_name, required_for_colab_only)
    packages = [
        ("pandas", "pandas", False),
        ("numpy", "numpy", False),
        ("openpyxl", "openpyxl", False),
        ("xlrd", "xlrd", False),
        ("lxml", "lxml", False),          # pandas.read_html (HTML-as-xlsx exports)
        ("yfinance", "yfinance", False),
        ("requests", "requests", False),
        ("reportlab", "reportlab", False),
        ("matplotlib", "matplotlib", False),
        ("pypdf", "pypdf", False),
        ("pdfplumber", "pdfplumber", False),
        ("PIL", "pillow", False),
        ("tabulate", "tabulate", False),
        ("tqdm", "tqdm", False),
        ("IPython", "ipython", False),
    ]
    if IS_COLAB:
        packages.append(("pytesseract", "pytesseract", True))

    missing = []
    for import_name, pip_name, _ in packages:
        try:
            __import__(import_name)
        except Exception:
            missing.append((import_name, pip_name))

    if not missing:
        print("[setup] All Python dependencies already available.")
        return

    print(f"[setup] Installing {len(missing)} missing package(s): "
          f"{', '.join(p for _, p in missing)}")
    install_cmd = "pip install --quiet"
    if IS_COLAB:
        install_cmd += " --user"
    for import_name, pip_name in missing:
        try:
            os.system(f"{install_cmd} '{pip_name}'")
        except Exception as e:
            print(f"[setup][WARN] pip install failed for {pip_name}: {e}")
    # Refresh import cache after install.
    import importlib
    for import_name, _ in missing:
        try:
            importlib.import_module(import_name)
        except Exception:
            pass

def ensure_system_ocr() -> bool:
    """
    Attempt to make OCR available for scanned PDFs.
    Returns True if pytesseract + the tesseract binary are usable.
    Never crashes the run when OCR is unavailable.
    """
    try:
        import pytesseract  # noqa: F401
    except Exception:
        return False
    # Is the tesseract binary present? Install it on Colab if apt is available.
    from shutil import which
    if which("tesseract") is None and IS_COLAB:
        os.system("apt-get install -y -q tesseract-ocr > /dev/null 2>&1")
    if which("tesseract") is None:
        return False
    try:
        import pytesseract
        from PIL import Image
        pytesseract.get_tesseract_version()
        return True
    except Exception:
        return False

install_dependencies()

# -----------------------------------------------------------------------------
# Imports that must exist after installation
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 2/3. CONFIGURATION AND LOGGING
# -----------------------------------------------------------------------------
class Config:
    OUTPUT_DIR = "Portfolio_Output"
    CACHE_DIR = ".pa_cache"
    CURRENCY = "INR"
    CURRENCY_SYMBOL = "\u20b9"          # Rupee sign
    GENERATED_ON = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    HTTP_TIMEOUT = 20
    MAX_RETRIES = 3
    # Assessment vocabulary. Signals are derived from the USER'S OWN rules on
    # their own data; they are checklists, not investment recommendations.
    RULE_SIGNALS = ["BUY MORE", "HOLD", "SELL", "INSUFFICIENT DATA"]
    STOCK_ASSESSMENTS = ["BUY CANDIDATE", "HOLD", "WATCHLIST", "REDUCE", "AVOID"]
    FUND_ASSESSMENTS = ["ACCUMULATE", "HOLD", "REVIEW"]
    NA = "N/A"

def setup_logging() -> logging.Logger:
    logger = logging.getLogger("PortfolioAnalyzer")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    sh = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

LOG = setup_logging()

def fmt_inr(value: Any, decimals: int = 2) -> str:
    """Indian currency formatting with the ₹ symbol and lakh/crore-friendly commas."""
    if value is None or (isinstance(value, float) and (math.isnan(value))):
        return Config.NA
    try:
        value = float(value)
    except Exception:
        return str(value)
    sign = "-" if value < 0 else ""
    value = abs(value)
    whole = int(value)
    frac = round(value - whole, decimals)
    # Indian grouping: last 3 digits, then groups of 2.
    s = str(whole)
    if len(s) > 3:
        head, tail = s[:-3], s[-3:]
        parts = []
        while len(head) > 2:
            parts.insert(0, head[-2:])
            head = head[:-2]
        if head:
            parts.insert(0, head)
        grouped = ",".join(parts) + "," + tail
    else:
        grouped = s
    return f"{sign}{Config.CURRENCY_SYMBOL}{grouped}.{int(round(frac * (10 ** decimals))):0{decimals}d}"

def safe_div(numer: Any, denom: Any) -> Optional[float]:
    """Division that returns None instead of raising on invalid/zero denominators."""
    try:
        n = float(numer)
        d = float(denom)
        if d == 0 or math.isnan(n) or math.isnan(d):
            return None
        return n / d
    except Exception:
        return None

def to_float(value: Any) -> Optional[float]:
    """Parse a numeric value from strings that may contain ₹, commas, %, () etc."""
    if value is None:
        return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if isinstance(value, float) and math.isnan(value):
            return None
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"nan", "none", "-", "\u2013", "n/a", "na"}:
        return None
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    s = s.replace(Config.CURRENCY_SYMBOL, "").replace("₹", "")
    s = s.replace(",", "").replace("%", "").replace(" ", "")
    s = s.replace("cr", "").replace("Cr", "").replace("lakh", "").replace("Lakh", "")
    try:
        val = float(s)
        return -val if neg else val
    except Exception:
        return None

# -----------------------------------------------------------------------------
# 4. MANUAL HOLDINGS FILE UPLOAD (Colab native + local fallback)
# -----------------------------------------------------------------------------
@dataclass
class UploadedFile:
    name: str
    data: bytes
    size: int

def _looks_like_notebook(name: str) -> bool:
    return name.lower().endswith((".ipynb", ".py"))

# Only these are definitely NOT holdings data. Everything else is accepted and
# judged by CONTENT (magic bytes), because broker exports often mislabel files
# (an HTML table saved as .xlsx, a CSV renamed to .xls, etc.).
_BLOCKED_EXTS = (".ipynb", ".py", ".png", ".jpg", ".jpeg", ".gif")

def upload_holdings_file() -> List[UploadedFile]:
    """
    Request the holdings file from the user.
    Accepts anything that CONTENT-wise is Excel/CSV/PDF (extension is a hint,
    not a veto). Never interprets notebook args or images as data.
    Returns a list (multiple files are handled by the caller).
    """
    allowed_exts = (".xlsx", ".xls", ".csv", ".pdf")
    print("\n" + "=" * 70)
    print(" STEP 1 / 8 : UPLOAD YOUR HOLDINGS FILE")
    print("=" * 70)
    print("Supported formats: .xlsx, .xls, .csv, .pdf")
    print("(The file type is checked by CONTENT, so a broker export whose name is")
    print(" mislabeled - e.g. an .xlsx that is really a table - is still accepted.)")

    if IS_COLAB:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        if not uploaded:
            raise ValueError("No file was uploaded in Colab. Aborting.")
        results = []
        for name, data in uploaded.items():
            low = name.lower()
            if low.endswith(".json") or _looks_like_notebook(name):
                LOG.warning("Ignoring notebook/code file '%s' (not a holdings file).", name)
                continue
            if low.endswith(_BLOCKED_EXTS):
                LOG.warning("Ignoring file '%s' - that type can not be holdings data.", name)
                continue
            fmt = detect_format(UploadedFile(name=name, data=data, size=len(data)))
            if fmt == "unknown":
                LOG.warning("File '%s' is not recognisable as Excel/CSV/PDF content "
                            "even though its name says so. Its first bytes were: %r",
                            name, bytes(data[:24]))
                continue
            if not low.endswith(allowed_exts):
                LOG.info("File '%s' has an unusual name but its content is a valid "
                         "%s - accepting it.", name, fmt.upper())
            results.append(UploadedFile(name=name, data=data, size=len(data)))
        if not results:
            raise ValueError("Uploaded files were not in a supported holdings format.")
        return results
    else:
        # Local Jupyter fallback: try the widgets file uploader, else a path prompt.
        try:
            import ipywidgets as widgets
            from IPython.display import display
            up = widgets.FileUpload(multiple=True)   # no accept filter: judge by content
            display(up)
            LOG.info("Select your holdings file using the widget above, then re-run "
                     "only if nothing appears. Falling back to a path prompt if the "
                     "widget yields nothing after selection.")
            # Give the user a moment, then read whatever is uploaded.
            import time as _t
            deadline = _t.time() + 60
            while _t.time() < deadline and len(up.value) == 0:
                _t.sleep(1)
            if len(up.value) > 0:
                results = []
                for item in up.value:
                    name = item["name"]
                    data = item["content"]
                    results.append(UploadedFile(name=name, data=data, size=len(data)))
                return results
        except Exception as e:
            LOG.info("File-upload widget unavailable (%s). Using path prompt.", e)
        # Plain prompt fallback.
        path = input("Enter the full path to your holdings file "
                     "(.xlsx/.xls/.csv/.pdf): ").strip().strip('"')
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")
        with open(path, "rb") as f:
            data = f.read()
        return [UploadedFile(name=os.path.basename(path), data=data,
                             size=len(data))]

def resolve_multiple_uploads(files: List[UploadedFile]) -> UploadedFile:
    """
    If several files were uploaded, ask which to analyze (or list them) rather
    than silently merging duplicate holdings.
    """
    if len(files) == 1:
        return files[0]
    print("\nMultiple supported files were uploaded:")
    for i, f in enumerate(files, start=1):
        print(f"  [{i}] {f.name}  ({f.size/1024:.1f} KB)")
    choice = None
    while choice is None:
        raw = input("Enter the number of the file to analyze "
                    "(they are NOT merged automatically): ").strip()
        try:
            idx = int(raw)
            if 1 <= idx <= len(files):
                choice = files[idx - 1]
            else:
                print("Invalid selection, try again.")
        except ValueError:
            print("Please enter a valid number.")
    return choice


# -----------------------------------------------------------------------------
# 5. FILE VALIDATION & DATA EXTRACTION  (xlsx / xls / csv / pdf)
# -----------------------------------------------------------------------------
SUPPORTED_EXTENSIONS = (".xlsx", ".xls", ".csv", ".pdf")

@dataclass
class ExtractedTable:
    """Raw tabular data pulled from the uploaded file, before column mapping."""
    dataframe: pd.DataFrame
    source_format: str
    notes: List[str] = field(default_factory=list)
    has_header: bool = False

def detect_format(file: UploadedFile) -> str:
    """
    Detect file format from CONTENT magic bytes only - the extension is never
    trusted, because broker exports are often mislabeled (HTML tables or CSV
    text saved with an .xlsx name). Returns "unknown" for anything that is not
    readable data (empty file, binary garbage), which callers report clearly.
    """
    data = file.data
    if data is None or (isinstance(data, (bytes, bytearray)) and len(data.strip()) == 0) \
            or (isinstance(data, str) and not data.strip()):
        return "unknown"
    if data[:4] == b"%PDF":
        return "pdf"
    if data[:2] == b"PK":                       # xlsx is a zip container
        return "xlsx"
    if data[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":  # legacy OLE2 (.xls)
        return "xls"
    head = data[:512].lstrip().lower() if isinstance(data, bytes) \
        else str(data)[:512].lstrip().lower()
    if head.startswith((b"<!doctype", b"<html", b"<table", b"<?xml")):
        # HTML table masquerading as a spreadsheet (very common broker export).
        return "html"
    # Delimited text only if it is mostly printable.
    sample = data[:2048] if isinstance(data, bytes) else str(data)[:2048].encode()
    printable = sum(1 for b in sample if 32 <= b < 127 or b in (9, 10, 13))
    if sample and printable / len(sample) > 0.85:
        return "csv"
    return "unknown"

def extract_tables(file: UploadedFile) -> List[ExtractedTable]:
    """
    Return a list of ExtractedTable. Excel/CSV workbooks with several sheets
    (e.g. broker statements with Equity / Mutual Funds / Combined tabs) are
    fully read; duplicate rows across sheets are removed later by ISIN/symbol.
    """
    return _extract_table(file)

def _extract_table(file: UploadedFile) -> List[ExtractedTable]:
    fmt = detect_format(file)
    LOG.info("Detected file format: %s (%s bytes)", fmt, file.size)
    if fmt == "unknown":
        raise ValueError(
            f"The file '{file.name}' could not be recognised as Excel/CSV/PDF "
            "content. Please export the holdings again from the broker as "
            ".xlsx or .csv and upload that file.")
    if fmt == "pdf":
        return [_extract_pdf(file)]
    buf = io.BytesIO(file.data)
    if fmt == "html":
        return _extract_html_tables(file, fmt)
    if fmt == "csv":
        try:
            df = _read_csv_robust(buf)
        except Exception as e:
            raise ValueError(f"Could not read the CSV file: {e}")
        if df is None or df.empty:
            raise ValueError("The uploaded file appears to be empty.")
        return [ExtractedTable(dataframe=df, source_format=fmt)]
    # Excel: read EVERY sheet (broker statements often split Equity / MF / Combined).
    try:
        engine = "openpyxl" if fmt == "xlsx" else "xlrd"
        sheets = pd.read_excel(buf, engine=engine, sheet_name=None, header=None)
    except Exception as e:
        # Mislabelled exports: a file sniffing as zip/OLE2 but not a real
        # workbook. Try HTML tables, then delimited text, before giving up.
        tables = _extract_html_tables(file, fmt, quiet_ok=True)
        if tables:
            LOG.warning("'%s' was named Excel but contained an HTML table - "
                        "parsed it as such.", file.name)
            return tables
        try:
            df = _read_csv_robust(io.BytesIO(file.data))
            if df is not None and not df.empty:
                LOG.warning("'%s' was named Excel but contained delimited text - "
                            "parsed it as CSV.", file.name)
                return [ExtractedTable(dataframe=df, source_format="csv")]
        except Exception:
            pass
        raise ValueError(f"Could not read the {fmt} file: {e}")
    tables: List[ExtractedTable] = []
    for sheet_name, df in sheets.items():
        if df is None or df.empty:
            continue
        tables.append(ExtractedTable(dataframe=df.reset_index(drop=True),
                                     source_format=fmt,
                                     notes=[f"sheet: {sheet_name}"]))
    if not tables:
        raise ValueError("The uploaded workbook appears to be empty.")
    LOG.info("Read %d worksheet(s): %s", len(tables),
             [t.notes[0] for t in tables])
    return tables

def _extract_html_tables(file: UploadedFile, sniffed_as: str,
                         quiet_ok: bool = False) -> List[ExtractedTable]:
    """
    Parse <table> markup saved under a spreadsheet name (classic broker export).
    Uses pandas.read_html (lxml/bs4); returns [] when no tables are found and
    quiet_ok is set, so the caller can try other fallbacks.
    """
    try:
        text = file.data.decode("utf-8-sig", errors="replace") \
            if isinstance(file.data, (bytes, bytearray)) else str(file.data)
        frames = pd.read_html(io.StringIO(text), header=None)
    except Exception as e:
        if quiet_ok:
            return []
        raise ValueError(
            f"The file '{file.name}' contains HTML (a web-page table saved as "
            f"{sniffed_as.upper()}), but it could not be parsed: {e}")
    tables: List[ExtractedTable] = []
    for i, df in enumerate(frames):
        if df is None or df.empty or df.dropna(how="all").empty:
            continue
        tables.append(ExtractedTable(dataframe=df.reset_index(drop=True),
                                     source_format="html",
                                     notes=[f"html table {i + 1}"]))
    if not tables:
        if quiet_ok:
            return []
        raise ValueError(f"No readable table found inside the HTML file '{file.name}'.")
    LOG.info("Parsed %d HTML table(s) from '%s'.", len(tables), file.name)
    return tables

def _read_csv_robust(buf: io.BytesIO) -> pd.DataFrame:
    """Try common delimiters + encodings for CSV holdings exports."""
    raw = buf.getvalue()
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            text = raw.decode(enc)
            break
        except Exception:
            continue
    else:
        text = raw.decode("utf-8", errors="ignore")
    sample = text[:4096]
    delim = ","
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        delim = dialect.delimiter
    except Exception:
        counts = {d: sample.count(d) for d in [",", ";", "\t", "|"]}
        if counts:
            delim = max(counts, key=counts.get)
    from io import StringIO
    return pd.read_csv(StringIO(text), sep=delim, header=None,
                       engine="python", skip_blank_lines=True)

def _extract_pdf(file: UploadedFile) -> ExtractedTable:
    """Extract tables/text from PDF; fall back to OCR for scanned pages."""
    notes: List[str] = []
    pdfplumber = None
    try:
        import pdfplumber
    except Exception as e:
        raise ValueError(f"PDF support unavailable ({e}).")
    buf = io.BytesIO(file.data)
    best_table = None
    text_lines: List[str] = []
    page_count = 0
    try:
        with pdfplumber.open(buf) as pdf:
            page_count = len(pdf.pages)
            for page in pdf.pages:
                tables = page.extract_tables() or []
                for t in tables:
                    df = pd.DataFrame(t[1:], columns=t[0]) if t and t[0] else None
                    if df is not None and len(df) > 0:
                        if best_table is None or len(df) > len(best_table):
                            best_table = df
                txt = page.extract_text() or ""
                if txt.strip():
                    text_lines.append(txt)
    except Exception as e:
        notes.append(f"pdfplumber table/text extraction error: {e}")

    if best_table is not None and len(best_table) >= 1:
        notes.append(f"Extracted a table with {len(best_table)} rows from PDF.")
        return ExtractedTable(dataframe=best_table, source_format="pdf", notes=notes,
                              has_header=True)

    # No embedded table -> try text parse, then OCR if needed.
    joined_text = "\n".join(text_lines)
    if len(joined_text.strip()) < 30 and page_count > 0:
        notes.append("PDF looks scanned (little embedded text). Attempting OCR...")
        ocr_text = _ocr_pdf(file)
        if ocr_text:
            joined_text = ocr_text
            notes.append("OCR text obtained; parsed with best-effort column inference.")
        else:
            notes.append("OCR unavailable/failed. Cannot extract reliable holdings "
                         "from this scanned PDF. Provide a text-based PDF or CSV/Excel.")
            raise ValueError("Scanned PDF without usable OCR. No reliable data extracted.")

    df = _parse_holding_text(joined_text)
    if df is None or df.empty:
        raise ValueError("Could not identify any holdings table in the PDF text.")
    return ExtractedTable(dataframe=df, source_format="pdf", notes=notes)

def _ocr_pdf(file: UploadedFile) -> Optional[str]:
    try:
        import pytesseract
        from PIL import Image
    except Exception:
        return None
    if shutil.which("tesseract") is None:
        return None
    try:
        import pdfplumber
        buf = io.BytesIO(file.data)
        chunks = []
        with pdfplumber.open(buf) as pdf:
            for page in pdf.pages[:6]:                # cap OCR pages for speed
                img = page.to_image(resolution=200).original
                chunks.append(pytesseract.image_to_string(img))
        return "\n".join(chunks)
    except Exception as e:
        LOG.warning("OCR failed: %s", e)
        return None

def _parse_holding_text(text: str) -> Optional[pd.DataFrame]:
    """Best-effort: turn whitespace-aligned text into a header+rows DataFrame."""
    lines = [ln for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2:
        return None
    rows = [re.split(r"\s{2,}|\t", ln.strip()) for ln in lines]
    width = max(len(r) for r in rows)
    if width < 3:
        return None
    rows = [r + [""] * (width - len(r)) for r in rows]
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 6. COLUMN MAPPING, NORMALIZATION & VALIDATION
# -----------------------------------------------------------------------------
# Map canonical field -> list of name fragments seen across Indian brokers.
COLUMN_SYNONYMS: Dict[str, List[str]] = {
    "name": ["name", "security", "scrip name", "stock name", "company",
             "description", "fund name", "scheme name"],
    "symbol": ["symbol", "ticker", "scrip code", "nse code", "bse code",
               "scheme code", "scrip"],
    "isin": ["isin"],
    "quantity": ["quantity", "qty", "units", "no of shares", "shares",
                 "vol", "volume", "holding", "nos"],
    "avg_price": ["avg", "average", "buy price", "purchase price", "cost",
                  "avg price", "average price", "avg cost", "bought"],
    "invested": ["invested", "buy value", "purchase value", "cost value",
                 "amount invested", "total buy", "investment", "bought value",
                 "buy amount"],
    "cmp": ["cmp", "current market price", "last price", "ltp", "market price",
            "current price", "previous closing price", "closing price",
            "price", "nav", "current nav"],
    "current_value": ["current value", "market value", "mv", "valuation",
                      "present value", "current market value", "latest value",
                      "total value", "scheme value"],
    "pnl": ["p/l", "pl", "p l", "profit", "loss", "unrealised",
            "unrealized", "gain", "mtm", "overall gain", "profit/loss"],
    "pnl_pct": ["p/l %", "p l pct", "return", "gain %", "profit %",
                "unrealised %", "unrealized %", "pct", "%"],
    "sector": ["sector"],
    "asset_type": ["instrument type", "asset type", "security type",
                   "type", "category", "kind"],
    "exchange": ["exchange", "exch", "bse/nse"],
    "currency": ["currency", "ccy"],
}

@dataclass
class Holding:
    """Normalized single holding row with full provenance."""
    source_row: int
    raw_name: str
    name: str
    symbol: str
    ISIN: str
    quantity: Optional[float]
    avg_price: Optional[float]
    invested: Optional[float]
    cmp: Optional[float]
    current_value: Optional[float]
    pnl: Optional[float]
    pnl_pct: Optional[float]
    asset_type_raw: str
    exchange: str
    currency: str
    # Filled during processing:
    asset_class: str = "Unidentified"
    sector: str = Config.NA
    broker_sector: str = ""        # sector as supplied by the broker, if any
    sub_category: str = Config.NA
    classification_source: str = Config.NA
    confidence: str = "Low"
    issues: List[str] = field(default_factory=list)
    yf_ticker: str = ""            # resolved yfinance symbol
    metrics: Dict[str, Any] = field(default_factory=dict)
    assessment: str = Config.NA
    assessment_reasons: List[str] = field(default_factory=list)
    analyzed: bool = False
    analysis: Dict[str, Any] = field(default_factory=dict)  # rule tables for the PDF

def _norm_header(s: Any) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9/%.-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s

def _detect_header_row(df: pd.DataFrame) -> int:
    """
    Find the row that looks most like a header. Broker statements put the real
    header far down the sheet (Angel/SmartIQ equity sheets have ~7 metadata +
    6 summary lines before it), so we scan well beyond the top rows and score
    mainly by KNOWN column names - summary labels like "Invested Value" match
    nothing, while "Symbol / ISIN / Quantity Available..." matches ~14.
    """
    best_idx, best_score = 0, -1.0
    for i in range(min(200, len(df))):
        row = df.iloc[i].astype(str).tolist()
        cells = [c for c in row if c and c.lower() not in ("nan", "none", "")]
        if len(cells) < 3:
            continue
        known = 0
        non_numeric = 0
        for c in cells:
            is_num = to_float(c) is not None
            if not is_num:
                non_numeric += 1
            # A header cell must MATCH a synonym as a whole word, so data rows
            # containing words like "finance" or "capital" are not mistaken
            # for headers (\b fails after '%' so '/(\b|$)' is used too).
            nc = _norm_header(c)
            if any(re.search(r"(?:\b|/|%)" + re.escape(frag) + r"(?:\b|/|$)", nc)
                   for frags in COLUMN_SYNONYMS.values() for frag in frags):
                known += 1
        if non_numeric < len(cells) * 0.7:
            continue  # majority must be text, not numbers
        score = known * 3.0 + non_numeric
        if score > best_score:
            best_score, best_idx = score, i
            if known >= 3 and non_numeric / len(cells) >= 0.9:
                break  # confidently a header; stop early on big sheets
    return best_idx

def map_columns(df: pd.DataFrame, has_header: bool = False) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """
    Auto-detect a header row and map synonyms to canonical columns.
    If has_header is True the DataFrame's existing columns are treated as the
    header (used for tables already parsed with a header row, e.g. from PDF).
    Returns (data-with-canonical-cols, mapping dict).
    """
    if has_header:
        header = df.columns.astype(str).tolist()
        data = df.reset_index(drop=True)
        data.columns = [_norm_header(h) for h in header]
    else:
        hdr_idx = _detect_header_row(df)
        header = df.iloc[hdr_idx].astype(str).tolist()
        data = df.iloc[hdr_idx + 1:].reset_index(drop=True)
        data.columns = [ _norm_header(h) for h in header ]

    mapping: Dict[str, int] = {}
    assigned_src: set = set()
    for canon, synonyms in COLUMN_SYNONYMS.items():
        chosen = None
        for src_i, col in enumerate(data.columns):
            if src_i in assigned_src:
                continue
            for syn in synonyms:
                if syn in col:
                    chosen = src_i
                    break
            if chosen is not None:
                break
        if chosen is not None:
            mapping[canon] = chosen
            assigned_src.add(chosen)

    # Build a clean frame keyed by canonical names.
    clean = pd.DataFrame()
    for canon, src_i in mapping.items():
        clean[canon] = data.iloc[:, src_i].values
    for i, col in enumerate(data.columns):
        if i not in assigned_src:
            clean[f"_extra_{col}"] = data.iloc[:, i].values
    return clean, mapping

def build_holdings(clean: pd.DataFrame, mapping: Dict[str, int],
                   source_format: str) -> List[Holding]:
    """Convert the mapped, clean frame into validated Holding objects."""
    holdings: List[Holding] = []
    def g(row: pd.Series, key: str) -> Any:
        return row[key] if key in row.index else None

    for idx, row in clean.iterrows():
        name = _clean_str(g(row, "name"))
        symbol = _clean_str(g(row, "symbol"))
        provided_isin = _clean_str(g(row, "isin")).upper()
        provided_sector = _clean_str(g(row, "sector"))
        # Broker statements often have no separate "name" column: fall back to
        # the symbol (which for MF rows is the full fund name) then the ISIN.
        if not name:
            name = symbol or provided_isin
        if not name and not symbol and not provided_isin:
            continue  # truly empty row

        h = Holding(
            source_row=int(idx) + 1,
            raw_name=name or symbol or provided_isin,
            name=name,
            symbol=symbol,
            ISIN=provided_isin,
            quantity=to_float(g(row, "quantity")),
            avg_price=to_float(g(row, "avg_price")),
            invested=to_float(g(row, "invested")),
            cmp=to_float(g(row, "cmp")),
            current_value=to_float(g(row, "current_value")),
            pnl=to_float(g(row, "pnl")),
            pnl_pct=to_float(g(row, "pnl_pct")),
            asset_type_raw=_clean_str(g(row, "asset_type")),
            exchange=_clean_str(g(row, "exchange")).upper(),
            currency=(_clean_str(g(row, "currency")) or "INR").upper(),
        )
        if provided_sector:
            h.broker_sector = provided_sector
        # If ISIN was only available in the symbol column, capture it.
        if not h.ISIN and re.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}[0-9]", symbol or ""):
            h.ISIN = symbol
        _derive_and_validate(h)
        holdings.append(h)
    return holdings

def _clean_str(v: Any) -> str:
    if v is None:
        return ""
    s = str(v).strip()
    if s.lower() in ("nan", "none", ""):
        return ""
    # Broker "Combined" sheets use a lone dash as a not-applicable placeholder.
    if s in ("-", "\u2013", "\u2014", "--", "n/a", "na", "nil", "none"):
        return ""
    return s

def _derive_and_validate(h: Holding) -> None:
    """
    Fill derived values (invested / current_value / pnl) and validate consistency.
    Missing values are NOT invented; only derived where at least the required
    inputs exist. Distinguishes missing vs genuine zero via issues list.
    """
    # invested = quantity * avg_price  (when not given directly)
    if h.invested is None and h.quantity is not None and h.avg_price is not None:
        h.invested = h.quantity * h.avg_price
        h.classification_source = h.classification_source  # unchanged
    # current_value = quantity * cmp   (when not given directly)
    if h.current_value is None and h.quantity is not None and h.cmp is not None:
        h.current_value = h.quantity * h.cmp
    # pnl = current_value - invested
    if h.pnl is None and h.current_value is not None and h.invested is not None:
        h.pnl = h.current_value - h.invested
    # pnl_pct = pnl / invested * 100
    if h.pnl_pct is None:
        r = safe_div(h.pnl, h.invested)
        h.pnl_pct = (r * 100) if r is not None and h.invested and h.invested > 0 else None

    # Validate quantity/price presence
    if h.quantity is None and h.invested is None:
        h.issues.append("Missing both quantity and invested amount.")
    if h.invested is not None and h.invested < 0:
        h.issues.append("Negative invested amount (short position or data error).")
    if h.current_value is not None and h.invested is not None and h.current_value > 0:
        # Consistency: recomputed pnl vs provided pnl
        if h.pnl is not None:
            recomputed = h.current_value - h.invested
            if abs(recomputed - h.pnl) > max(1.0, abs(recomputed) * 0.02):
                h.issues.append(
                    f"P/L mismatch: file says {fmt_inr(h.pnl)}, "
                    f"recomputed {fmt_inr(recomputed)}.")


# -----------------------------------------------------------------------------
# 7. AUTOMATIC ASSET CLASSIFICATION
# -----------------------------------------------------------------------------
# Primary asset classes (A-E in the master prompt).
CLASS_BANKS_FI = "Banks & Financial Institutions"
CLASS_STOCKS   = "Individual Stocks"
CLASS_ETF      = "ETFs"
CLASS_MF       = "Mutual Funds"
CLASS_UNKNOWN  = "Unidentified Assets"

# Curated keyword/identifier tables. These are transparent, editable heuristics;
# every classification records its source + confidence so users can verify.
ETF_NAME_HINTS = ["etf", "exchange traded fund", "niftybees", "bankbees",
                  "goldbees", "gold etf", "silver etf", "nifty 50 etf",
                  "juniorbees", "itbees", "midcap beees", "shadowlender"]
MF_NAME_HINTS = ["mutual fund", "direct plan", "regular plan", "growth plan",
                 "idcw", "dividend plan", "elss", "flexi cap", "flexicap",
                 "multi cap", "small cap", "mid cap", "large cap", "index fund",
                 "liquid fund", "gilt fund", "fof", "fund of funds", "hybrid",
                 "savings plan", "systematic"]
BANK_FI_HINTS = ["bank", "bajaj finance", "hudco", "nbcc", "housing develop",
                 "nbf", "finance", "financial", "l&t finance", "sundaram",
                 "chothala", "shriram", "muthoot", "manappuram", "credit",
                 "capital", "muthoot finance", "iasl", "repos", "millicap",
                 "iifl", "angel one", "bse", "mcx", "csl", "cent", "small finance"]
INSURANCE_HINTS = ["insurance", "lic", "hdfc life", "icici luna", "birla sun life",
                   "sbi life", "max life", "star health", "care health",
                   "national insurance", "new india insurance", " oriental",
                   "unix", "reinsurance", "general insurance"]

# Sector detection keyword table for individual stocks (name based).
SECTOR_KEYWORDS: Dict[str, List[str]] = {
    "Information Technology": ["infosys", "tcs", "tatko", "wipro", "hcl",
        "tech mahindra", "lTIMINDTREE", "mindtree", "persistent", "cognizant",
        "mpc", " Zensar", "lnt", "page industries", "kfintech", "oracle",
        "simplify", "cummins", "it", "software", "systems", "technologies", "digital"],
    "Banking & NBFC": ["bank", "finance", "housing"],
    "Pharmaceuticals & Healthcare": ["sun pharma", "dr reddy", "cipl"
        "lupin", "aurobindo", "aurolibharam", "divis", "biocon", "glyphin",
        "pharma", "laboratories", "healthcare", "hospital", "diagnostics", "medic"],
    "Automobiles": ["maruti", "mahindra", "m&m", "tata motors", " Bajaj Auto",
        "hero", "tvS", "eicher", "ashok", "auto", "motors", "automotive", "tyres"],
    "FMCG": ["hdfc cbi", "hindustile", "hul", "itc", "nestle", "britania",
        "dabur", "godrej consumer", "colgate", "marico", "tata consumer",
        "emami", "fmcg", "foods", "beverage"],
    "Metals & Mining": ["tata steel", "jsw", "jsw steel", "vedanta", "hindalco",
        "national alum", "sAIL", "rashtriy", "jindal", "zinc", "copper",
        "metal", "steel", "mineral", "mining"],
    "Energy & Oil": ["reliance", "ongc", "gail", "indian oil", "bpcl", "hpcl",
        "coal india", "nTPC", "power", "energy", "petroleum", "renewable", "solar"],
    "Chemicals": ["srf", "pidilite", "deepak nitrite", "tata chemical", "uflex",
        "chemical", "fertilizer", "chromite", "aromatic"],
    "Infrastructure & Construction": ["l&t", "larisen", "ncc", "shapoorji",
        "irb", "dhandos", "roads", "infra", "construction", "cement", "ultratech",
        "grasim", "ambuja", "dalmia"],
    "Telecom": ["reliance jio", "bharti airtel", "vodafone idea", "vi", "telecom"],
    "Real Estate": ["dlf", "godrej prop", "sobha", "prestige", "brigade",
        "embassy", "realty", "properties", "realestate"],
    "Consumer Discretionary": ["titan", "nykaa", "avenue", "trendra", "page",
        "fashion", "retail", "jewel", "luxury"],
    "Defence & Aerospace": ["hindustan aeronautics", "hal", "beML", "cochin ship",
        "mazagon", "bharat dynamics", "defence", "aerospace"],
    "IT Services / Consulting": ["accureyt", "caltas", "tata consultancy"],
    "Agriculture & Rural": ["escorts", "icra", "rural", "agri", "seed"],
    "Transportation & Logistics": ["concor", "ship", "logistics", "container",
        "transport", "airline", "indigo", "interglobe"],
    "Media & Entertainment": ["saregama", "pvr", "inoxia", "media", "entertain",
        "cinema", "network"],
    "Electronics & EMS": ["kaynes", " Dixon", " Amber", "Syrma", "electronic",
        "semiconductor", "delta autowave"],
    "Services": ["jbm", "tata Elxsi", "unitech", "mspl", "moschip"],
    "Diversified": ["tata", "godrej", "mahindra"],
}

# A small, transparent symbol->sector map for common Indian large caps so we do
# not rely purely on fuzzy name matching. Extend freely.
KNOWN_SYMBOL_SECTOR: Dict[str, str] = {
    "RELIANCE": "Energy & Oil", "TCS": "Information Technology",
    "HDFCBANK": "Banking & NBFC", "ICICIBANK": "Banking & NBFC",
    "INFY": "Information Technology", "SBIN": "Banking & NBFC",
    "BHARTIARTL": "Telecom", "ITC": "FMCG", "LT": "Infrastructure & Construction",
    "KOTAKBANK": "Banking & NBFC", "AXISBANK": "Banking & NBFC",
    "BAJFINANCE": "Banking & NBFC", "ASIANPAINT": "Consumer Discretionary",
    "MARUTI": "Automobiles", "HCLTECH": "Information Technology",
    "SUNPHARMA": "Pharmaceuticals & Healthcare", "TITAN": "Consumer Discretionary",
    "UMIHDFC": "Banking & NBFC", "WIPRO": "Information Technology",
    "NESTLEIND": "FMCG", "TATAMOTORS": "Automobiles",
}

def _contains(hay: str, needle: str) -> bool:
    return needle.strip().lower() in (hay or "").lower()

def _prettify_sector(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().title()

def classify_holding(h: Holding) -> None:
    """
    Assign primary asset class + sector/sub-category with source + confidence.

    For Indian markets these signals are authoritative and checked first:
      * ISIN prefix:  INF -> AMFI scheme (ETF or mutual fund),
                      INE -> exchange-listed security (equity / bond / InvIT).
      * broker 'Sector' column: 'ETF' -> exchange-traded fund; financial-
                      services sectors -> Banks & NBFC; otherwise it is used
                      directly as the sector label (more reliable than fuzzy
                      name matching).
    General files without these columns still fall back to keyword heuristics.
    """
    hay = f"{h.name} {h.symbol} {h.asset_type_raw}".lower()
    type_hint = (h.asset_type_raw or "").lower()
    bsector = (h.broker_sector or "").strip().lower()
    isin = (h.ISIN or "").upper()
    is_fund_isin = isin.startswith("INF")
    is_security_isin = isin.startswith("INE")

    # --- ETF detection (broker tag is authoritative for this broker format) ---
    looks_etf = (
        bsector == "etf" or type_hint == "etf"
        or "bees" in (h.symbol or "").lower()
        or (h.symbol or "").lower().endswith("ietf")
        or re.search(r"(^|[^a-z])etf([^a-z]|$)", hay) is not None
        or any(_contains(hay, k) for k in
               ["goldbees", "niftybees", "juniorbees", "itbees", "bankbees"])
    )

    if looks_etf:
        _set_class(h, CLASS_ETF, _etf_flavour(hay),
                   "broker sector = ETF" if bsector == "etf" else "ETF name/symbol token",
                   "High" if (bsector == "etf" or is_fund_isin) else "Medium")
        h.sector = "ETF"
    elif is_fund_isin:
        # INF scheme that is not an ETF -> mutual fund.
        _set_class(h, CLASS_MF, _mf_subcategory(hay + " " + type_hint),
                   "ISIN prefix INF (AMFI scheme)", "High")
    elif _contains(hay, "mutual fund") or any(
            k in type_hint for k in ["mutual", "hybrid", "elss", "idcw"]):
        _set_class(h, CLASS_MF, _mf_subcategory(hay + " " + type_hint),
                   "asset_type/name heuristic", "Medium")
    elif bsector and bsector != "etf":
        # Security with a broker sector label -> trust it.
        if bsector in ("financial services", "banking", "bank", "nbfc",
                        "insurance", "housing finance"):
            sub = ("Insurance"
                   if any(_contains(hay, k) for k in INSURANCE_HINTS)
                   else _bank_sub(hay))
            _set_class(h, CLASS_BANKS_FI, sub, "broker sector (financials)", "High")
        else:
            _set_class(h, CLASS_STOCKS, Config.NA, "broker sector (equity)", "High")
        h.sector = _prettify_sector(bsector)
    else:
        _classify_by_keywords(h, hay, type_hint)
        return

    _flag_unknown(h)

def _classify_by_keywords(h: Holding, hay: str, type_hint: str) -> None:
    """Fallback classification for files lacking ISIN/broker-sector columns."""
    if "etf" in type_hint:
        _set_class(h, CLASS_ETF, _etf_flavour(hay), "asset_type column", "High")
    elif any(k in type_hint for k in ["mutual", "mf", "scheme", "fund"]):
        _set_class(h, CLASS_MF, _mf_subcategory(hay), "asset_type column", "High")
    elif any(k in type_hint for k in ["equity", "stock", "share", "common"]):
        _set_class(h, CLASS_STOCKS, Config.NA, "asset_type column", "High")
    elif any(_contains(hay, k) for k in MF_NAME_HINTS):
        _set_class(h, CLASS_MF, _mf_subcategory(hay), "name heuristic", "Medium")
    elif any(_contains(hay, k) for k in INSURANCE_HINTS):
        _set_class(h, CLASS_BANKS_FI, "Insurance", "name heuristic", "Medium")
    elif any(_contains(hay, k) for k in BANK_FI_HINTS):
        _set_class(h, CLASS_BANKS_FI, _bank_sub(hay), "name heuristic", "Medium")
    else:
        _set_class(h, CLASS_STOCKS, Config.NA, "default (equity-like)", "Low")

    if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI):
        sym_key = (h.symbol or "").upper().replace(" ", "")
        if sym_key in KNOWN_SYMBOL_SECTOR:
            h.sector = KNOWN_SYMBOL_SECTOR[sym_key]
            if h.classification_source == "default (equity-like)":
                h.classification_source = "known-symbol sector map"
                h.confidence = "Medium"
        else:
            h.sector = _detect_sector(hay)
    _flag_unknown(h)

def _flag_unknown(h: Holding) -> None:
    if h.asset_class == CLASS_STOCKS and h.sector in (Config.NA, "") and h.confidence == "Low":
        h.asset_class = CLASS_UNKNOWN
        h.issues.append("Could not confidently identify asset type/sector.")

def _set_class(h: Holding, cls: str, sub: str, source: str, conf: str):
    h.asset_class = cls
    h.sub_category = sub
    h.classification_source = source
    h.confidence = conf

def _detect_sector(hay: str) -> str:
    for sector, kws in SECTOR_KEYWORDS.items():
        for kw in kws:
            if _contains(hay, kw):
                return sector
    return Config.NA

def _bank_sub(hay: str) -> str:
    if "insurance" in hay:
        return "Insurance"
    if any(k in hay for k in ["small finance", "sfb"]):
        return "Small Finance Bank"
    if any(k in hay for k in ["housing", "hudco", "home fin"]):
        return "Housing Finance"
    if any(k in hay for k in ["nbfc", "finance", "credit", "capital", "loan"]):
        return "NBFC"
    return "Bank"

def _etf_flavour(hay: str) -> str:
    if "gold" in hay:
        return "Gold ETF"
    if "silver" in hay:
        return "Silver ETF"
    if any(k in hay for k in ["us", "nasdaq", "s&p", "sp500", "global"]):
        return "International / US ETF"
    if "bank" in hay:
        return "Sectoral ETF (Bank)"
    if "nifty 50" in hay or "niftybees" in hay:
        return "Broad-market ETF (Nifty 50)"
    if any(k in hay for k in ["midcap", "mid cap", "juniorbees", "small"]):
        return "Mid/Small-cap ETF"
    if "it" in hay or "tech" in hay:
        return "Thematic ETF (IT/Tech)"
    return "ETF"

def _mf_subcategory(hay: str) -> str:
    for key, label in [
        ("elss", "ELSS"), ("flexi", "Flexi-cap"), ("multi", "Multi-cap"),
        ("small cap", "Small-cap"), ("mid cap", "Mid-cap"),
        ("large cap", "Large-cap"), ("index", "Index Fund"),
        ("liquid", "Liquid Fund"), ("gilt", "Gilt / Duration Fund"),
        ("debt", "Debt Fund"), ("hybrid", "Hybrid Fund"),
        ("fof", "Fund of Funds"), ("fund of fund", "Fund of Funds"),
        ("nifty 50", "Index Fund"), ("sensex", "Index Fund"),
    ]:
        if key in hay:
            plan = "Direct" if "direct" in hay else ("Regular" if "regular" in hay else Config.NA)
            opt = "Growth" if "growth" in hay else ("IDCW" if any(k in hay for k in
                  ["idcw", "dividend"]) else Config.NA)
            return f"{label} ({plan}, {opt})".replace("(, )", "").replace("N/A", "Unknown")
    return "Mutual Fund"


# -----------------------------------------------------------------------------
# 8. FINANCIAL DATA COLLECTION  (yfinance where suitable + caching + retries)
# -----------------------------------------------------------------------------
# IMPORTANT DISCLOSURE:
#   yfinance provides *delayed / historical* data and does NOT reliably cover
#   Indian mutual-fund NAVs, NSE/BSE filings, shareholding, or bank-specific
#   fundamentals (NPA, CASA, NIM, etc.). Anything not available is marked N/A.
#   We never invent numbers.
class DataCollector:
    def __init__(self):
        self.cache: Dict[str, Any] = {}
        self.source_timestamps: Dict[str, str] = {}
        self.failures: List[str] = []
        self.uses_network = False
        try:
            import yfinance as yf
            self.yf = yf
            self.has_yf = True
        except Exception as e:
            self.yf = None
            self.has_yf = False
            LOG.warning("yfinance unavailable (%s); market data will be N/A.", e)
        os.makedirs(Config.CACHE_DIR, exist_ok=True)

    # ---- ticker resolution ------------------------------------------------
    def resolve_yf_ticker(self, h: Holding) -> str:
        """
        Map a holding to a yfinance symbol. Indian equities trade on NSE and use
        '.NS' suffix. We only append it to plausible alphabetic tickers; we never
        guess on mutual funds (yf lacks most Indian MF schemes).
        """
        sym = (h.symbol or h.name or "").strip().upper()
        sym = re.sub(r"\s+", "", sym)
        # Already suffixed?
        if sym.endswith(".NS") or sym.endswith(".BO"):
            return sym
        # Drop fund/plan noise for MF: no reliable yf ticker.
        if h.asset_class == CLASS_MF:
            return ""
        # NSE broker symbols sometimes carry an exchange/series suffix
        # (TATAGOLD-E, SILVERLINE-X, IRBINVIT-IV, TATSILV-E). Strip it.
        base = sym.split("-")[0]
        # A usable NSE ticker: starts with a letter, then alphanumerics
        # (e.g. NIFTYBEES, NEXT50IETF, ICICIB22, IRCTC).
        if re.fullmatch(r"[A-Z][A-Z0-9]{1,13}", base):
            return f"{base}.NS"
        return ""

    def _cache_key(self, kind: str, ident: str) -> str:
        raw = f"{kind}:{ident}"
        return hashlib.md5(raw.encode()).hexdigest()

    def _cached(self, key: str, ttl_sec: int = 6 * 3600) -> Optional[Any]:
        path = os.path.join(Config.CACHE_DIR, key + ".json")
        if not os.path.exists(path):
            return None
        if time.time() - os.path.getmtime(path) > ttl_sec:
            return None
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return None

    def _store_cache(self, key: str, obj: Any) -> None:
        try:
            path = os.path.join(Config.CACHE_DIR, key + ".json")
            with open(path, "w", encoding="utf-8") as f:
                json.dump(obj, f, default=str)
        except Exception:
            pass

    def fetch_price_history(self, ticker: str, period: str = "1y") -> Optional[pd.DataFrame]:
        if not self.has_yf or not ticker:
            return None
        key = self._cache_key("hist", f"{ticker}:{period}")
        cached = self._cached(key)
        if cached is not None:
            return pd.DataFrame(cached)
        for attempt in range(1, Config.MAX_RETRIES + 1):
            try:
                self.uses_network = True
                tk = self.yf.Ticker(ticker)
                hist = tk.history(period=period)
                if hist is None or hist.empty:
                    return None
                out = hist.reset_index()[["Date", "Close", "Volume"]].copy()
                out["Date"] = out["Date"].astype(str)
                self.cache[ticker] = out
                self.source_timestamps[ticker] = str(out["Date"].iloc[-1])
                self._store_cache(key, out.to_dict(orient="records"))
                return out
            except Exception as e:
                LOG.debug("history attempt %d failed for %s: %s", attempt, ticker, e)
                time.sleep(min(2 ** attempt, 6))
        self.failures.append(f"price-history:{ticker}")
        return None

    def fetch_info(self, ticker: str) -> Dict[str, Any]:
        """Fetch yfinance .info / .fast_info; returns {} on failure. N/A-friendly."""
        if not self.has_yf or not ticker:
            return {}
        key = self._cache_key("info", ticker)
        cached = self._cached(key, ttl_sec=6 * 3600)
        if cached is not None:
            return cached
        for attempt in range(1, Config.MAX_RETRIES + 1):
            try:
                self.uses_network = True
                tk = self.yf.Ticker(ticker)
                info = {}
                try:
                    info.update(tk.info or {})
                except Exception:
                    pass
                # Merge fast_info primitives.
                try:
                    fi = dict(tk.fast_info) if tk.fast_info else {}
                    info.setdefault("lastPrice", fi.get("lastPrice"))
                    info.setdefault("currency", fi.get("currency"))
                except Exception:
                    pass
                # Only JSON-serializable subset cached.
                slim = {k: v for k, v in info.items()
                        if isinstance(v, (int, float, str, bool)) or v is None}
                self._store_cache(key, slim)
                return info
            except Exception as e:
                LOG.debug("info attempt %d failed for %s: %s", attempt, ticker, e)
                time.sleep(min(2 ** attempt, 6))
        self.failures.append(f"info:{ticker}")
        return {}

    def fetch_financials(self, ticker: str) -> Dict[str, pd.DataFrame]:
        """
        Fetch multi-year annual statements (balance sheet, income statement,
        cash flow) from yfinance. In-memory cache per run; empty dict on
        failure. These let us COMPUTE ratios like a Screener-style report
        instead of trusting (often missing) pre-computed .info fields.
        """
        if not self.has_yf or not ticker:
            return {}
        if not hasattr(self, "fin_cache"):
            self.fin_cache: Dict[str, Dict[str, pd.DataFrame]] = {}
        if ticker in self.fin_cache:
            return self.fin_cache[ticker]
        out: Dict[str, pd.DataFrame] = {}
        try:
            self.uses_network = True
            tk = self.yf.Ticker(ticker)
            for name, attr in (("bs", "balance_sheet"), ("fin", "income_stmt"),
                               ("cf", "cashflow")):
                try:
                    df = getattr(tk, attr)
                    if df is not None and not df.empty:
                        out[name] = df
                except Exception:
                    pass
        except Exception as e:
            LOG.debug("financials fetch failed for %s: %s", ticker, e)
        self.fin_cache[ticker] = out
        if not out:
            self.failures.append(f"financials:{ticker}")
        return out


# -----------------------------------------------------------------------------
# 9. SEPARATE ANALYSIS ENGINES  (do NOT share one scoring formula)
# -----------------------------------------------------------------------------
# ---- statement-mining helpers (Screener-style ratios from raw statements) ----
def _clean_num(x: Any) -> Optional[float]:
    try:
        v = float(x)
        return None if math.isnan(v) else v
    except Exception:
        return None

def _series_row(df: Optional[pd.DataFrame], names: List[str]) -> List[Optional[float]]:
    """Values of the first matching line item, most-recent period first."""
    if df is None or getattr(df, "empty", True):
        return []
    for nm in names:
        try:
            if nm in df.index:
                row = df.loc[nm]
                if isinstance(row, pd.DataFrame):   # duplicated line-item labels
                    row = row.iloc[0]
                return [_clean_num(x) for x in row.tolist()]
        except Exception:
            continue
    return []

def _paired_pct(a: List[Optional[float]], b: List[Optional[float]]) -> List[Optional[float]]:
    out = []
    for i in range(min(len(a), len(b))):
        if a[i] is None or not b[i]:
            out.append(None)
        else:
            out.append(a[i] / b[i] * 100.0)
    return out

def _avg(vals: List[Optional[float]]) -> Optional[float]:
    vs = [v for v in vals if v is not None]
    return (sum(vs) / len(vs)) if vs else None

def _last(vals: List[Optional[float]]) -> Optional[float]:
    for v in vals:
        if v is not None:
            return v
    return None

def _oldest(vals: List[Optional[float]]) -> Optional[float]:
    for v in reversed(vals):
        if v is not None:
            return v
    return None

def _cagr(vals: List[Optional[float]]) -> Tuple[Optional[float], int]:
    """(growth % per year, periods) between latest and oldest non-null value."""
    v0, vn = _last(vals), _oldest(vals)
    n = len(vals) - 1
    if v0 is None or vn is None or n < 1 or v0 <= 0 or vn <= 0:
        return (None, n)
    return (((v0 / vn) ** (1.0 / n) - 1.0) * 100.0, n)

def _yoy(vals: List[Optional[float]]) -> Optional[float]:
    """Latest vs previous period growth, percent."""
    if len(vals) < 2 or vals[0] is None or not vals[1]:
        return None
    return (vals[0] / vals[1] - 1.0) * 100.0

def compute_fundamentals(fs: Dict[str, pd.DataFrame]) -> Dict[str, Any]:
    """
    Compute multi-year ratios and growth rates from raw annual statements
    (balance sheet 'bs', income statement 'fin', cash flow 'cf'). Anything the
    statements cannot support stays None -> shown as N/A, never guessed.
    """
    fin, bs, cf = fs.get("fin"), fs.get("bs"), fs.get("cf")
    revenue = _series_row(fin, ["Total Revenue", "Operating Revenue"])
    ebit    = _series_row(fin, ["Operating Income"])
    dep     = _series_row(fin, ["Reconciled Depreciation",
                               "Depreciation And Amortization",
                               "Depreciation & Amortization"])
    ni      = _series_row(fin, ["Net Income", "Net Income Common Stockholders"])
    eps     = _series_row(fin, ["Basic EPS", "Diluted Normalized EPS"])
    intexp  = _series_row(fin, ["Interest Expense"])
    equity  = _series_row(bs, ["Stockholders Equity",
                               "Total Equity Gross Minority Interest",
                               "Common Stock Equity"])
    debt    = _series_row(bs, ["Total Debt", "Long Term Debt", "LongTerm Debt"])
    cfo     = _series_row(cf, ["Operating Cash Flow",
                               "Total Cash From Operating Activities"])
    capex   = _series_row(cf, ["Capital Expenditure", "Capital Expenditures"])

    ebitda: List[Optional[float]] = []
    for i in range(min(len(ebit), len(dep)) if dep else len(ebit)):
        if ebit[i] is None:
            ebitda.append(None)
        else:
            ebitda.append(ebit[i] + abs(dep[i] if i < len(dep) and dep[i] else 0.0))

    roce: List[Optional[float]] = []
    for i in range(min(len(ebit), len(equity), len(debt)) if debt else min(len(ebit), len(equity))):
        d = debt[i] if i < len(debt) and debt[i] else 0.0
        ce = (equity[i] or 0.0) + d if i < len(equity) else None
        roce.append(None if (ebit[i] is None or not ce) else ebit[i] / ce * 100.0)

    de_ratio: List[Optional[float]] = []
    for i in range(min(len(debt), len(equity))):
        e = equity[i]
        de_ratio.append(None if (debt[i] is None or not e) else debt[i] / e)

    icov: List[Optional[float]] = []
    for i in range(min(len(ebit), len(intexp))):
        ie = abs(intexp[i]) if intexp[i] is not None else None
        if ebit[i] is None:
            icov.append(None)
        elif not ie:
            icov.append(None)   # no interest expense -> coverage not meaningful
        else:
            icov.append(ebit[i] / ie)

    fcf: List[Optional[float]] = []
    for i in range(min(len(cfo), len(capex))):
        if cfo[i] is None:
            fcf.append(None)
        else:
            fcf.append(cfo[i] - abs(capex[i] or 0.0))

    roe = _paired_pct(ni, equity)
    ebitda_m = _paired_pct(ebitda, revenue)
    net_m = _paired_pct(ni, revenue)
    cf_m = _paired_pct(cfo, revenue)
    return {
        "revenue": revenue, "ebitda": ebitda, "ni": ni, "eps": eps,
        "cfo": cfo, "capex": capex, "equity": equity, "debt": debt,
        "fcf": fcf, "roe": roe, "roce": roce,
        "ebitda_margin": ebitda_m, "net_margin": net_m, "cf_margin": cf_m,
        "debt_equity": de_ratio, "interest_cov": icov,
        "roe_avg": _avg(roe), "roce_avg": _avg(roce),
        "ebitda_margin_avg": _avg(ebitda_m), "net_margin_avg": _avg(net_m),
        "cf_margin_avg": _avg(cf_m),
        "fcf_last": _last(fcf), "de_last": _last(de_ratio),
        "icov_last": _last(icov),
        "rev_cagr": _cagr(revenue), "ebitda_cagr": _cagr(ebitda),
        "eps_cagr": _cagr(eps), "ni_cagr": _cagr(ni),
        "rev_yoy": _yoy(revenue), "ni_yoy": _yoy(ni),
        "cfo_yoy": _yoy(cfo), "fcf_yoy": _yoy(fcf),
        "years": max(len(revenue), len(ni), len(roe), len(roce)),
    }


# -----------------------------------------------------------------------------
# 6-STEP RULE ENGINE (modelled on the user's Screener-style checklist)
# -----------------------------------------------------------------------------
SIG_PASS, SIG_FAIL, SIG_WATCH, SIG_NA = "PASS", "FAIL", "WATCH", "N/A"
_ACTION = {SIG_PASS: "BUY MORE", SIG_FAIL: "HOLD", SIG_WATCH: "HOLD", SIG_NA: "VERIFY"}

def _rule(check: str, rule: str, value: str, signal: str) -> Dict[str, str]:
    return {"check": check, "rule": rule, "value": value,
            "signal": signal, "action": _ACTION.get(signal, "HOLD")}

def _pc(v: Optional[float], dp: int = 1) -> str:
    return Config.NA if v is None else f"{v:.{dp}f}%"

def _xx(v: Optional[float], dp: int = 2) -> str:
    return Config.NA if v is None else f"{v:.{dp}f}x"

def _cr(v: Optional[float]) -> str:
    return Config.NA if v is None else f"{Config.CURRENCY_SYMBOL}{v:,.0f} cr"

def _sig(cond: bool, na: bool = False) -> str:
    if na:
        return SIG_NA
    return SIG_PASS if cond else SIG_FAIL

def build_stock_rules(fund: Dict[str, Any], info: Dict[str, Any],
                      pr: Dict[str, Any], is_financial: bool = False) -> Dict[str, Any]:
    """
    Build the four scored steps. Every check shows the value and a PASS/FAIL/
    WATCH/N/A signal. Checks that cannot be computed from available data are
    marked N/A and EXCLUDED from the score (never guessed). For Banks/NBFCs the
    Debt/Equity and Interest-coverage rules are not applicable (N/A).
    """
    def num(v):
        return v if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v)) else None

    mc_cr = (info.get("marketCap") or 0) / 1e7 if num(info.get("marketCap")) else None
    roce_avg, roe_avg = fund.get("roce_avg"), fund.get("roe_avg")
    roce_now, roe_now = _last(fund.get("roce", [])), _last(fund.get("roe", []))
    fcf = fund.get("fcf_last")
    de = fund.get("de_last")
    icov = fund.get("icov_last")
    rev_c = fund.get("rev_cagr", (None, 0))[0]
    eb_c = fund.get("ebitda_cagr", (None, 0))[0]
    eps_c = fund.get("eps_cagr", (None, 0))[0]
    rev_y, ni_y, cfo_y = fund.get("rev_yoy"), fund.get("ni_yoy"), fund.get("cfo_yoy")
    pe, pb = num(info.get("trailingPE")), num(info.get("priceToBook"))
    pfcf = (mc_cr / (fcf / 1e7)) if (fcf and mc_cr) else None
    years = fund.get("years", 0)

    # STEP 1 - quality & risk
    s1 = [
        _rule("Market cap", ">= " + _cr(1000), _cr(mc_cr),
              _sig(bool(mc_cr and mc_cr >= 1000), mc_cr is None)),
        _rule("Promoter holding", "> 50% (not freely available)",
              "N/A (needs BSE/NSE shareholding filing)", SIG_NA),
        _rule("Promoter pledge", "< 15% / ideally 0%",
              "N/A (needs latest filing)", SIG_NA),
        _rule("Debt to equity", "< 1.5" if not is_financial else "N/A for banks/NBFCs",
              (_pc(None) if is_financial else _xx(de)),
              SIG_NA if is_financial else _sig(bool(de is not None and de < 1.5), de is None)),
        _rule("Interest coverage", "> 3" if not is_financial else "N/A for banks/NBFCs",
              _xx(icov), SIG_NA if is_financial else _sig(bool(icov and icov > 3), icov is None)),
    ]
    # STEP 2 - profitability
    s2 = [
        _rule("5y avg ROCE", "> 15%", f"{_pc(roce_avg)} (now {_pc(roce_now)})",
              _sig(bool(roce_avg and roce_avg > 15), roce_avg is None)),
        _rule("5y avg ROE", "> 12%", f"{_pc(roe_avg)} (now {_pc(roe_now)})",
              _sig(bool(roe_avg and roe_avg > 12), roe_avg is None)),
        _rule("EBITDA margin", "> 18%", _pc(_last(fund.get("ebitda_margin", []))),
              _sig(bool(_last(fund.get("ebitda_margin", [])) and _last(fund.get("ebitda_margin", [])) > 18),
                   _last(fund.get("ebitda_margin", [])) is None)),
        _rule("Net profit margin", "> 10%", _pc(_last(fund.get("net_margin", []))),
              _sig(bool(_last(fund.get("net_margin", [])) and _last(fund.get("net_margin", [])) > 10),
                   _last(fund.get("net_margin", [])) is None)),
        _rule("Cash-flow margin", "> 10%", _pc(_last(fund.get("cf_margin", []))),
              _sig(bool(_last(fund.get("cf_margin", [])) and _last(fund.get("cf_margin", [])) > 10),
                   _last(fund.get("cf_margin", [])) is None)),
        _rule("Free cash flow", "Positive", _cr(fcf / 1e7 if fcf else None),
              _sig(bool(fcf and fcf > 0), fcf is None)),
    ]
    # STEP 3 - growth
    triangle_ok = None
    if rev_y is not None and ni_y is not None and cfo_y is not None:
        triangle_ok = (rev_y > 0 and ni_y > 0 and cfo_y > 0)
    s3 = [
        _rule(f"Revenue growth ({years}y CAGR)", "> 10%", _pc(rev_c),
              _sig(bool(rev_c and rev_c > 10), rev_c is None)),
        _rule(f"EBITDA growth ({years}y CAGR)", "> 12%", _pc(eb_c),
              _sig(bool(eb_c and eb_c > 12), eb_c is None)),
        _rule(f"EPS growth ({years}y CAGR)", "> 10%", _pc(eps_c),
              _sig(bool(eps_c and eps_c > 10), eps_c is None)),
        _rule("Profit triangle", "Revenue, Profit & Cash flow all grow",
              (f"Rev {_pc(rev_y)}, Profit {_pc(ni_y)}, CFO {_pc(cfo_y)} (YoY)"),
              _sig(bool(triangle_ok), triangle_ok is None)),
    ]
    # STEP 4 - price / valuation
    s4 = [
        _rule("P/E vs sector", "No big premium to sector",
              f"P/E {_xx(pe)} (sector P/E not in free data)",
              SIG_WATCH if pe else SIG_NA),
        _rule("P/B vs sector", "No big premium to sector",
              f"P/B {_xx(pb)} supported by ROE {_pc(roe_now)}",
              SIG_WATCH if pb else SIG_NA),
        _rule("Price to free cash flow", "< 25", _xx(pfcf, 1),
              _sig(bool(pfcf and pfcf < 25), pfcf is None)),
        _rule("Promoter selling", "No continuous selling",
              "N/A (needs shareholding history)", SIG_NA),
        _rule("Red flags", "<= 2 red flags",
              "N/A (needs Tickertape/Screener screen)", SIG_NA),
    ]
    steps = [("Step 1: Eliminating low quality and risk", s1),
             ("Step 2: Is it profitable?", s2),
             ("Step 3: Is it growing?", s3),
             ("Step 4: Is the price right?", s4)]

    allrows = [r for _, rows in steps for r in rows]
    n_pass = sum(1 for r in allrows if r["signal"] == SIG_PASS)
    n_fail = sum(1 for r in allrows if r["signal"] == SIG_FAIL)
    n_watch = sum(1 for r in allrows if r["signal"] == SIG_WATCH)
    n_na = sum(1 for r in allrows if r["signal"] == SIG_NA)
    applicable = n_pass + n_fail + n_watch
    step1_applicable = [r for r in s1 if r["signal"] != SIG_NA]
    step1_all_pass = bool(step1_applicable) and all(
        r["signal"] == SIG_PASS for r in step1_applicable)
    ratio = safe_div(n_pass, applicable)
    if applicable < 8 or ratio is None:
        signal = "INSUFFICIENT DATA"
    elif ratio >= 0.85 and step1_all_pass:
        signal = "BUY MORE"
    elif ratio >= 0.55:
        signal = "HOLD"
    else:
        signal = "SELL"
    return {"steps": steps, "pass": n_pass, "fail": n_fail, "watch": n_watch,
            "na": n_na, "applicable": applicable, "ratio_pct": (ratio * 100 if ratio else None),
            "signal": signal, "step1_all_pass": step1_all_pass}


# Shared helper: transparent scoring from available metrics only.
class BaseEngine:
    def __init__(self, collector: DataCollector):
        self.collector = collector

    def _na(self, h: Holding, key: str, val: Any) -> Any:
        h.metrics[key] = val if val not in (None, "") else Config.NA
        return h.metrics[key]

    def price_and_returns(self, h: Holding) -> Dict[str, Any]:
        """Common market-data block: latest price, ranges, moving averages."""
        out: Dict[str, Any] = {}
        hist = self.collector.fetch_price_history(h.yf_ticker) if h.yf_ticker else None
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        last = info.get("regularMarketPrice") or info.get("previousClose") or \
            info.get("lastPrice")
        if last is None and hist is not None and len(hist):
            last = float(hist["Close"].iloc[-1])
        out["last_price"] = last
        out["price_as_of"] = self.collector.source_timestamps.get(h.yf_ticker, Config.NA)
        if hist is not None and len(hist):
            closes = hist["Close"].astype(float)
            out["wk52_high"] = float(closes.max())
            out["wk52_low"] = float(closes.min())
            out["sma_50"] = float(closes.tail(50).mean()) if len(closes) >= 50 else None
            out["sma_200"] = float(closes.tail(200).mean()) if len(closes) >= 200 else None
            ret = safe_div(last, closes.iloc[0]) if closes.iloc[0] else None
            out["period_return_pct"] = ((ret - 1) * 100) if ret else None
            # Simple annualised volatility from daily returns.
            dr = closes.pct_change().dropna()
            out["volatility_pct"] = float(dr.std() * math.sqrt(252) * 100) if len(dr) > 5 else None
            # Rolling horizon returns computed from the same price series.
            dts = pd.to_datetime(hist["Date"].astype(str))
            for label, yrs in (("ret_1y", 1), ("ret_3y", 3), ("ret_5y", 5)):
                out[label] = None
                if last:
                    target = dts.iloc[-1] - pd.Timedelta(days=int(365.25 * yrs))
                    sel = dts[dts <= target]
                    if len(sel):
                        base = float(closes.loc[sel.index[-1]])
                        if base:
                            out[label] = (safe_div(last, base) - 1) * 100 if safe_div(last, base) else None
        else:
            out["wk52_high"] = out["wk52_low"] = out["sma_50"] = \
                out["sma_200"] = out["period_return_pct"] = out["volatility_pct"] = None
        for label in ("ret_1y", "ret_3y", "ret_5y"):
            out.setdefault(label, None)
        return out


class IndividualStockEngine(BaseEngine):
    """Fundamental + technical scoring for non-financial companies."""
    def analyze(self, h: Holding) -> None:
        m = {}
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        fs = self.collector.fetch_financials(h.yf_ticker) if h.yf_ticker else {}
        fund = compute_fundamentals(fs) if fs else {}
        m["Sector"] = h.sector
        m["Last Price"] = pr["last_price"]
        m["52W High"] = pr["wk52_high"]
        m["52W Low"] = pr["wk52_low"]
        m["SMA50"] = pr["sma_50"]
        m["SMA200"] = pr["sma_200"]
        m["Period Return %"] = pr["period_return_pct"]
        m["Volatility % (ann.)"] = pr["volatility_pct"]
        m["1Y Return %"] = pr["ret_1y"]
        m["3Y Return %"] = pr["ret_3y"]
        m["5Y Return %"] = pr["ret_5y"]
        # Valuation / quality from yfinance info (all optional, N/A if absent).
        m["P/E"] = info.get("trailingPE")
        m["P/B"] = info.get("priceToBook")
        m["Dividend Yield %"] = (info.get("dividendYield", 0) * 100) if info.get("dividendYield") else None
        m["Market Cap"] = info.get("marketCap")
        # --- Computed fundamentals from raw annual statements (preferred) ------
        m["ROCE % (5y avg)"] = fund.get("roce_avg")
        m["ROCE % (latest)"] = _last(fund.get("roce", []))
        m["ROE % (5y avg)"] = fund.get("roe_avg")
        m["ROE % (latest)"] = _last(fund.get("roe", []))
        m["EBITDA Margin %"] = _last(fund.get("ebitda_margin", []))
        m["Net Profit Margin %"] = _last(fund.get("net_margin", []))
        m["Cash-Flow Margin %"] = _last(fund.get("cf_margin", []))
        m["Debt/Equity"] = fund.get("de_last")
        m["Interest Coverage"] = fund.get("icov_last")
        m["Free Cash Flow"] = fund.get("fcf_last")
        m["Revenue CAGR %"] = fund.get("rev_cagr", (None, 0))[0]
        m["EBITDA CAGR %"] = fund.get("ebitda_cagr", (None, 0))[0]
        m["EPS CAGR %"] = fund.get("eps_cagr", (None, 0))[0]
        # Fall back to yfinance .info if a computed value is missing.
        if m["ROE % (latest)"] is None:
            m["ROE % (latest)"] = (info.get("returnOnEquity", 0) * 100) if info.get("returnOnEquity") else None
        if m["Free Cash Flow"] is None:
            m["Free Cash Flow"] = info.get("freeCashflow")
        # Metrics not obtainable from free data for India (honest N/A):
        for k in ["Promoter Holding %", "Pledged %", "Receivables Days",
                  "Inventory Turnover", "Auditor Remarks"]:
            m.setdefault(k, Config.NA)
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = (
            "yfinance price + annual financial statements (delayed/historical); "
            "ratios computed. Promoter/pledge & sector P/E need BSE/NSE/Screener - N/A."
            if h.yf_ticker else "no market data")
        # Build the 6-step rule engine result when we have fundamentals/statements.
        if fund and (info or pr.get("last_price")):
            rules = build_stock_rules(fund, info, pr, is_financial=False)
            h.analysis["rules"] = rules
            h.assessment = rules["signal"]
            h.assessment_reasons = self._rule_reasons(rules)
        else:
            h.assessment, h.assessment_reasons = self._score(h, m)
        h.analyzed = True

    def _rule_reasons(self, rules: Dict[str, Any]) -> List[str]:
        out = [f"Rule-based score: {rules['pass']} pass / {rules['fail']} fail / "
               f"{rules['watch']} watch / {rules['na']} N/A "
               f"(signal: {rules['signal']})."]
        for title, rows in rules["steps"]:
            fails = [r["check"] for r in rows if r["signal"] == SIG_FAIL]
            if fails:
                out.append(f"{title.split(':')[0]} fails: {', '.join(fails)}")
        if rules["na"] >= 6:
            out.append("Several checks are N/A (promoter/pledge, sector P/E, red flags) "
                       "because that data is not in the free source; verify on "
                       "BSE/NSE filings or Screener before acting.")
        return out


    def _score(self, h: Holding, m: Dict[str, Any]) -> Tuple[str, List[str]]:
        reasons: List[str] = []
        score = 0
        # Each rule only fires when the underlying metric is available.
        def num(v):
            return v if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v)) else None
        pe = num(m.get("P/E")); roe = num(m.get("ROE % (latest)"))
        growth = num(m.get("EPS CAGR %"))
        dte = num(m.get("Debt/Equity")); prc = num(m.get("Last Price"))
        sma200 = num(m.get("SMA200")); margin = num(m.get("Net Profit Margin %"))
        vol = num(m.get("Volatility % (ann.)"))

        if roe is not None:
            if roe >= 18: score += 2; reasons.append(f"Strong ROE {roe:.1f}%")
            elif roe >= 12: score += 1; reasons.append(f"Decent ROE {roe:.1f}%")
            elif roe < 8: score -= 1; reasons.append(f"Weak ROE {roe:.1f}%")
        if growth is not None:
            if growth >= 15: score += 2; reasons.append(f"Earnings growing {growth:.1f}%")
            elif growth < 0: score -= 2; reasons.append(f"Earnings declining {growth:.1f}%")
        if margin is not None and margin >= 15:
            score += 1; reasons.append(f"Healthy net margin {margin:.1f}%")
        if dte is not None:
            if dte <= 0.6: score += 1; reasons.append(f"Low leverage (D/E {dte:.2f})")
            elif dte > 1.5: score -= 2; reasons.append(f"High leverage (D/E {dte:.2f})")
        if pe is not None:
            if 0 < pe < 20: score += 1; reasons.append(f"Reasonable P/E {pe:.1f}")
            elif pe > 60: score -= 1; reasons.append(f"Rich P/E {pe:.1f}")
        if prc is not None and sma200 is not None:
            if prc >= sma200: score += 1; reasons.append("Price above 200-DMA (trend intact)")
            else: score -= 1; reasons.append("Price below 200-DMA (weak trend)")
        if vol is not None and vol > 40:
            reasons.append(f"Elevated volatility {vol:.0f}%")
        if h.pnl_pct is not None and h.pnl_pct < -25:
            reasons.append("Deep unrealised loss - check if temporary vs fundamental")

        # Data-sufficiency guard.
        available = sum(1 for k in ["P/E","ROE % (latest)","EPS CAGR %","Debt/Equity","Last Price"]
                        if num(m.get(k)) is not None)
        if available < 2:
            return "WATCHLIST", ["Insufficient reliable data for a firm assessment. "
                                 "Metrics marked N/A; not forced into BUY/SELL.", *reasons]
        if score >= 5: return "BUY CANDIDATE", reasons
        if score >= 2: return "HOLD", reasons
        if score >= 0: return "WATCHLIST", reasons
        if score >= -2: return "REDUCE", reasons
        return "AVOID", reasons


class BankNBFCEngine(BaseEngine):
    """Banking/NBFC-specific metrics. Most are NOT in yfinance -> honest N/A."""
    def analyze(self, h: Holding) -> None:
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        fs = self.collector.fetch_financials(h.yf_ticker) if h.yf_ticker else {}
        fund = compute_fundamentals(fs) if fs else {}
        m = {
            "Sector": h.sector or "Banking & NBFC",
            "Last Price": pr["last_price"],
            "52W High": pr["wk52_high"], "52W Low": pr["wk52_low"],
            "SMA50": pr["sma_50"], "SMA200": pr["sma_200"],
            "Volatility % (ann.)": pr["volatility_pct"],
            "1Y Return %": pr["ret_1y"], "3Y Return %": pr["ret_3y"],
            "5Y Return %": pr["ret_5y"],
            "Period Return %": pr["period_return_pct"],
            "Market Cap": info.get("marketCap"),
            "P/E": info.get("trailingPE"),
            "P/B": info.get("priceToBook"),
            "Dividend Yield %": (info.get("dividendYield", 0) * 100) if info.get("dividendYield") else None,
            "ROE % (5y avg)": fund.get("roe_avg"),
            "ROE % (latest)": _last(fund.get("roe", [])),
            "Revenue CAGR %": fund.get("rev_cagr", (None, 0))[0],
            "EPS CAGR %": fund.get("eps_cagr", (None, 0))[0],
            # Bank-specific metrics that require authoritative Indian filings:
            "NIM %": Config.NA, "Gross NPA %": Config.NA, "Net NPA %": Config.NA,
            "Provision Coverage %": Config.NA, "Capital Adequacy (CAR) %": Config.NA,
            "CET1 %": Config.NA, "ROA %": Config.NA, "CASA Ratio %": Config.NA,
            "Cost-to-Income %": Config.NA, "Credit Growth %": Config.NA,
            "Deposit Growth %": Config.NA, "Slippages %": Config.NA,
            "Promoter Holding %": Config.NA, "Pledged %": Config.NA,
        }
        if m["ROE % (latest)"] is None:
            m["ROE % (latest)"] = (info.get("returnOnEquity", 0) * 100) if info.get("returnOnEquity") else None
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = ("yfinance price + annual statements (valuation, ROE, "
                                     "growth). Bank fundamentals (NPA, NIM, CAR, CASA) "
                                     "require RBI/company filings - marked N/A.")
        reasons = ["Bank/NBFC balance-sheet metrics (NPA, NIM, CAR, CASA) are not "
                   "available from the current data source and are shown as N/A.",
                   "Debt/Equity and interest-coverage rules are NOT applicable to "
                   "banks/NBFCs, so they are excluded from the score."]
        pb = m["P/B"] if isinstance(m["P/B"], (int, float)) else None
        roe = m["ROE % (latest)"] if isinstance(m["ROE % (latest)"], (int, float)) else None
        score = 0
        if roe is not None:
            score += 1 if roe >= 14 else (-1 if roe < 9 else 0)
            reasons.append(f"ROE {roe:.1f}%")
        if pb is not None:
            if 0 < pb < 1.5: score += 1; reasons.append(f"P/B {pb:.2f} (not expensive)")
            elif pb > 4: score -= 1; reasons.append(f"P/B {pb:.2f} (premium)")
        if pr["last_price"] and pr["sma_200"] and pr["last_price"] < pr["sma_200"]:
            reasons.append("Trading below 200-DMA")
        assess = "HOLD" if score >= 1 else ("WATCHLIST" if score >= 0 else "REDUCE")
        if fund and (info or pr["last_price"]):
            rules = build_stock_rules(fund, info, pr, is_financial=True)
            h.analysis["rules"] = rules
            reasons.insert(0, f"Rule-based score: {rules['pass']} pass / {rules['fail']} fail / "
                              f"{rules['watch']} watch / {rules['na']} N/A (signal: {rules['signal']}).")
            assess = rules["signal"]
        h.assessment = assess
        h.assessment_reasons = reasons
        h.analyzed = True


class InsuranceEngine(BaseEngine):
    def analyze(self, h: Holding) -> None:
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        fs = self.collector.fetch_financials(h.yf_ticker) if h.yf_ticker else {}
        fund = compute_fundamentals(fs) if fs else {}
        m = {
            "Sector": h.sector or "Insurance",
            "Last Price": pr["last_price"], "52W High": pr["wk52_high"],
            "52W Low": pr["wk52_low"], "SMA50": pr["sma_50"], "SMA200": pr["sma_200"],
            "Volatility % (ann.)": pr["volatility_pct"],
            "1Y Return %": pr["ret_1y"], "3Y Return %": pr["ret_3y"],
            "5Y Return %": pr["ret_5y"],
            "Period Return %": pr["period_return_pct"],
            "Market Cap": info.get("marketCap"),
            "P/E": info.get("trailingPE"), "P/B": info.get("priceToBook"),
            "Dividend Yield %": (info.get("dividendYield", 0) * 100) if info.get("dividendYield") else None,
            "ROE % (5y avg)": fund.get("roe_avg"),
            "ROE % (latest)": _last(fund.get("roe", [])),
            "Revenue CAGR %": fund.get("rev_cagr", (None, 0))[0],
            "EPS CAGR %": fund.get("eps_cagr", (None, 0))[0],
            # Insurance-specific (need company filings / IRDAI data):
            "Solvency Ratio": Config.NA, "Persistency %": Config.NA,
            "Embedded Value": Config.NA, "Value of New Business (VNB)": Config.NA,
            "Product Mix": Config.NA,
            "Promoter Holding %": Config.NA, "Pledged %": Config.NA,
        }
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = ("yfinance price + annual statements (valuation, ROE, "
                                     "growth). Insurance metrics (solvency, persistency, "
                                     "EV, VNB) need company/IRDAI filings - marked N/A.")
        h.assessment = "WATCHLIST"
        h.assessment_reasons = [
            "Insurance-specific metrics (solvency, persistency, EV, VNB margin) are "
            "unavailable from the current source and are shown as N/A.",
            "Firm assessment deferred until authoritative insurance data is added.",
        ]
        if fund and (info or pr["last_price"]):
            rules = build_stock_rules(fund, info, pr, is_financial=True)
            h.analysis["rules"] = rules
            h.assessment_reasons.insert(
                0, f"Rule-based score on generally-available data: {rules['pass']} pass / "
                   f"{rules['fail']} fail / {rules['watch']} watch / {rules['na']} N/A "
                   f"(signal: {rules['signal']}). Insurance checks are still N/A.")
            h.assessment = rules["signal"]
        h.analyzed = True


class ETFEngine(BaseEngine):
    def analyze(self, h: Holding) -> None:
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        m = {
            "ETF Flavour": h.sub_category,
            "Last Price": pr["last_price"], "52W High": pr["wk52_high"],
            "52W Low": pr["wk52_low"], "SMA50": pr["sma_50"], "SMA200": pr["sma_200"],
            "Period Return %": pr["period_return_pct"],
            "Volatility % (ann.)": pr["volatility_pct"],
            "1Y Return %": pr["ret_1y"], "3Y Return %": pr["ret_3y"],
            "5Y Return %": pr["ret_5y"],
            "Expense Ratio %": info.get("annualReportExpenseRatio"),
            "Total Assets (AUM)": info.get("totalAssets"),
            "NAV (yfinance)": info.get("navPrice") or info.get("previousClose"),
            # Need fund factsheet / AMFI data:
            "Tracking Error": Config.NA, "Tracking Difference": Config.NA,
            "Bid-Ask Spread": Config.NA,
            "Premium/Discount to NAV %": Config.NA, "Underlying Index": Config.NA,
            "Top Holdings": Config.NA, "Geographic Exposure": Config.NA,
        }
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = ("yfinance (price/vol/AUM/NAV where present; 1Y/3Y/5Y "
                                     "returns computed from price history). Tracking error, "
                                     "premium to NAV and holdings need the factsheet - N/A.")
        reasons = ["ETF is a diversified instrument; no arbitrary price target applied."]
        if isinstance(m["Expense Ratio %"], (int, float)):
            er = m["Expense Ratio %"]
            reasons.append(f"Expense ratio {er*100:.2f}%" if er < 1 else
                           f"Expense ratio {er:.2f}%")
        if pr["last_price"] and pr["sma_200"] and pr["last_price"] >= pr["sma_200"]:
            reasons.append("Trading above long-term average - accumulation can continue")
        else:
            reasons.append("Below long-term average - consider staggered accumulation")
        h.assessment = "ACCUMULATE" if "above long-term average" in " ".join(reasons) else "HOLD"
        h.assessment_reasons = reasons
        h.analyzed = True


class MutualFundEngine(BaseEngine):
    def analyze(self, h: Holding) -> None:
        # yfinance does not carry Indian MF scheme performance; be honest.
        m = {
            "Scheme Category": h.sub_category,
            "AMC": Config.NA, "Plan (Direct/Regular)": Config.NA,
            "Option (Growth/IDCW)": Config.NA, "Benchmark": Config.NA,
            "Fund Manager": Config.NA, "Expense Ratio %": Config.NA,
            "AUM": Config.NA,
            "1Y Return %": h.pnl_pct if h.invested and h.invested > 0 and h.pnl_pct is not None else Config.NA,
            "3Y Return %": Config.NA, "5Y Return %": Config.NA,
            "Rolling Returns": Config.NA, "Standard Deviation": Config.NA,
            "Sharpe Ratio": Config.NA, "Sortino Ratio": Config.NA,
            "Max Drawdown %": Config.NA, "Turnover Ratio": Config.NA,
            "Top Holdings": Config.NA, "Concentration": Config.NA,
            "Overlap": Config.NA, "Style Drift": Config.NA,
            "Exit Load": Config.NA,
        }
        if isinstance(h.sub_category, str) and ("Debt" in h.sub_category or "Gilt" in h.sub_category
                                                or "Liquid" in h.sub_category):
            m["Credit Quality"] = Config.NA
            m["Modified Duration"] = Config.NA
            m["Yield to Maturity %"] = Config.NA
            m["Interest-rate Risk"] = Config.NA
        h.metrics = m
        h.metrics["_data_source"] = ("No reliable free source in this environment for "
                                     "Indian MF factsheet data (AMFI/AMC). Only the "
                                     "user-supplied P/L is shown; all fund metrics N/A.")
        h.assessment = "REVIEW"
        h.assessment_reasons = [
            "Fund-level metrics (Sharpe, rolling returns, benchmark relative, expense "
            "ratio, holdings) are unavailable from the current data source (marked N/A).",
            "No accumulation/HOLD verdict is forced without benchmark & risk data.",
            "Recommended action: review once AMFI/AMC factsheet data is connected.",
        ]
        h.analyzed = True


class AnalysisOrchestrator:
    def __init__(self, collector: DataCollector):
        self.collector = collector
        self.stock = IndividualStockEngine(collector)
        self.bank = BankNBFCEngine(collector)
        self.ins = InsuranceEngine(collector)
        self.etf = ETFEngine(collector)
        self.mf = MutualFundEngine(collector)

    def analyze(self, h: Holding) -> None:
        try:
            if h.asset_class == CLASS_STOCKS:
                self.stock.analyze(h)
            elif h.asset_class == CLASS_BANKS_FI:
                # Insurance is a sub-type of financials with its own engine.
                if str(h.sub_category).lower().startswith("insurance"):
                    self.ins.analyze(h)
                else:
                    self.bank.analyze(h)
            elif h.asset_class == CLASS_ETF:
                self.etf.analyze(h)
            elif h.asset_class == CLASS_MF:
                self.mf.analyze(h)
            else:
                h.assessment = Config.NA
                h.assessment_reasons = ["Unidentified asset - not analyzed. "
                                        "Please verify symbol/ISIN."]
                h.analyzed = False
        except Exception as e:
            LOG.warning("Analysis failed for %s: %s", h.name, e)
            h.issues.append(f"Analysis error: {e}")
            h.assessment = Config.NA
            h.analyzed = False


# -----------------------------------------------------------------------------
# 10. PORTFOLIO-LEVEL CALCULATIONS & RISK ANALYSIS
# -----------------------------------------------------------------------------
@dataclass
class PortfolioSummary:
    total_invested: float = 0.0
    total_current_value: float = 0.0
    unrealized_pnl: float = 0.0
    unrealized_return_pct: Optional[float] = None
    n_holdings: int = 0
    n_identified: int = 0
    n_analyzed: int = 0
    n_failed_lookups: int = 0
    asset_allocation: Dict[str, float] = field(default_factory=dict)
    sector_allocation: Dict[str, float] = field(default_factory=dict)
    financial_sector_exposure_pct: float = 0.0
    domestic_pct: float = 0.0
    international_pct: float = 0.0
    top_holdings: List[Tuple[str, float, float]] = field(default_factory=list)
    concentration_top5_pct: float = 0.0
    duplicate_exposures: List[str] = field(default_factory=list)
    dividend_income: Optional[float] = None
    valuation_notes: List[str] = field(default_factory=list)

def effective_value(h: Holding) -> Tuple[float, bool]:
    """
    Return (current_value, used_invested_fallback).
    Never invents a market price: if no current value / cmp is known we fall back
    to the invested amount (flat assumption) and flag it.
    """
    if h.current_value is not None:
        return h.current_value, False
    if h.cmp is not None and h.quantity is not None:
        return h.cmp * h.quantity, False
    if h.invested is not None:
        return h.invested, True
    return 0.0, True

def compute_portfolio(holdings: List[Holding]) -> PortfolioSummary:
    ps = PortfolioSummary()
    ps.n_holdings = len(holdings)
    total_inv = 0.0
    total_mv = 0.0
    alloc_by_class: Dict[str, float] = {}
    alloc_by_sector: Dict[str, float] = {}
    per_holding_mv: List[Tuple[str, float]] = []
    seen_symbols: Dict[str, int] = {}
    fallback_count = 0
    div_income = 0.0
    div_known = False

    for h in holdings:
        mv, used_fallback = effective_value(h)
        if used_fallback and h.current_value is None:
            fallback_count += 1
            h.issues.append("Current market value unavailable; used invested amount "
                            "as a flat (0% return) proxy in portfolio totals.")
        inv = h.invested if h.invested is not None else 0.0
        total_inv += inv
        total_mv += mv
        alloc_by_class[h.asset_class] = alloc_by_class.get(h.asset_class, 0.0) + mv
        if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI) and h.sector != Config.NA:
            alloc_by_sector[h.sector] = alloc_by_sector.get(h.sector, 0.0) + mv
        per_holding_mv.append((h.name or h.symbol, mv))
        key = (h.symbol or h.name or "").upper().replace(" ", "")
        if key:
            seen_symbols[key] = seen_symbols.get(key, 0) + 1
        # Dividend income (only when we actually have a yield + value).
        dy = h.metrics.get("Dividend Yield %") if h.metrics else None
        if isinstance(dy, (int, float)) and mv:
            div_income += mv * dy / 100.0
            div_known = True
        if h.asset_class != CLASS_UNKNOWN:
            ps.n_identified += 1
        if h.analyzed:
            ps.n_analyzed += 1

    ps.total_invested = total_inv
    ps.total_current_value = total_mv
    ps.unrealized_pnl = total_mv - total_inv
    r = safe_div(ps.unrealized_pnl, total_inv)
    ps.unrealized_return_pct = (r * 100) if r is not None else None

    # Allocation percentages (guard denominator).
    for k, v in alloc_by_class.items():
        pct = safe_div(v, total_mv)
        ps.asset_allocation[k] = round(pct * 100, 2) if pct is not None else 0.0
    for k, v in alloc_by_sector.items():
        pct = safe_div(v, total_mv)
        ps.sector_allocation[k] = round(pct * 100, 2) if pct is not None else 0.0

    # Financial-sector exposure.
    fin_mv = sum(v for k, v in alloc_by_class.items() if k == CLASS_BANKS_FI)
    fin_mv += sum(v for k, v in alloc_by_sector.items() if k == "Banking & NBFC")
    p = safe_div(fin_mv, total_mv)
    ps.financial_sector_exposure_pct = round(p * 100, 2) if p is not None else 0.0

    # Domestic vs international (based on classification sub-flavours).
    intl_mv = 0.0
    for h in holdings:
        mv, _ = effective_value(h)
        sub = str(h.sub_category).lower()
        if any(t in sub for t in ["international", "us", "global"]) or \
           (h.currency and h.currency != "INR"):
            intl_mv += mv
    p_intl = safe_div(intl_mv, total_mv)
    ps.international_pct = round(p_intl * 100, 2) if p_intl is not None else 0.0
    ps.domestic_pct = round(100 - ps.international_pct, 2)

    # Top holdings & concentration.
    per_holding_mv.sort(key=lambda x: x[1], reverse=True)
    top5_mv = 0.0
    for i, (nm, mv) in enumerate(per_holding_mv):
        pct = safe_div(mv, total_mv)
        ps.top_holdings.append((nm, mv, round(pct * 100, 2) if pct else 0.0))
        if i < 5:
            top5_mv += mv
    p_top5 = safe_div(top5_mv, total_mv)
    ps.concentration_top5_pct = round(p_top5 * 100, 2) if p_top5 is not None else 0.0

    # Duplicate exposure (same symbol appearing multiple times).
    ps.duplicate_exposures = [k for k, c in seen_symbols.items() if c > 1]

    ps.dividend_income = round(div_income, 2) if div_known else None

    if fallback_count:
        ps.valuation_notes.append(
            f"{fallback_count} holding(s) had no current market value; invested "
            f"amount used as proxy (totals are conservative, not live valuations).")
    ps.n_failed_lookups = len(set(_COLLECTOR.failures)) if _COLLECTOR else 0
    return ps


# -----------------------------------------------------------------------------
# 11. CHART GENERATION
# -----------------------------------------------------------------------------
def generate_charts(holdings: List[Holding], ps: PortfolioSummary,
                    out_dir: str) -> List[str]:
    """Create PNG charts, returning file paths. Never crashes report generation."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    charts: List[str] = []

    def pie(data: Dict[str, float], title: str, fname: str):
        data = {k: v for k, v in data.items() if v and v > 0}
        if not data:
            return
        try:
            fig, ax = plt.subplots(figsize=(7, 5))
            labels = list(data.keys())
            sizes = list(data.values())
            ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90,
                   textprops={"fontsize": 8})
            ax.set_title(title)
            path = os.path.join(out_dir, fname)
            fig.tight_layout()
            fig.savefig(path, dpi=130)
            plt.close(fig)
            charts.append(path)
        except Exception as e:
            LOG.warning("Chart '%s' failed: %s", title, e)

    def bar(items: List[Tuple[str, float]], title: str, fname: str, color="#2a6f97"):
        items = [it for it in items if it[1] and it[1] > 0][:12]
        if not items:
            return
        try:
            fig, ax = plt.subplots(figsize=(8, 5))
            names = [i[0] for i in items]
            vals = [i[1] for i in items]
            ax.barh(names[::-1], vals[::-1], color=color)
            ax.set_title(title)
            ax.set_xlabel("Value (" + Config.CURRENCY_SYMBOL + ")")
            ax.tick_params(axis="y", labelsize=8)
            path = os.path.join(out_dir, fname)
            fig.tight_layout()
            fig.savefig(path, dpi=130)
            plt.close(fig)
            charts.append(path)
        except Exception as e:
            LOG.warning("Chart '%s' failed: %s", title, e)

    pie(ps.asset_allocation, "Asset-Class Allocation (%)", "chart_asset_allocation.png")
    pie(ps.sector_allocation, "Sector Allocation (Stocks & Financials)",
        "chart_sector_allocation.png")
    bar(ps.top_holdings, "Top Holdings by Market Value", "chart_top_holdings.png")

    # Per-holding historical price charts (limited to those with data).
    hist_dir = os.path.join(out_dir, "price_charts")
    os.makedirs(hist_dir, exist_ok=True)
    made = 0
    for h in holdings:
        if made >= 20:
            break
        hist = _COLLECTOR.cache.get(h.yf_ticker) if _COLLECTOR else None
        if hist is None or len(hist) < 20:
            continue
        try:
            fig, ax = plt.subplots(figsize=(8, 4))
            closes = hist["Close"].astype(float).values
            ax.plot(range(len(closes)), closes, linewidth=1.2, color="#1b4965")
            ax.set_title(f"{h.name or h.symbol} - price history (delayed/historical)")
            ax.set_ylabel("Price (" + Config.CURRENCY_SYMBOL + ")")
            ax.set_xlabel("trading days ->")
            path = os.path.join(hist_dir, f"price_{re.sub(r'[^A-Za-z0-9]','_', (h.symbol or h.name or 'h'))}.png")
            fig.tight_layout()
            fig.savefig(path, dpi=120)
            plt.close(fig)
            charts.append(path)
            _CHARTS_INDEX.setdefault(h.symbol or h.name, []).append(path)
            made += 1
        except Exception as e:
            LOG.debug("Price chart failed for %s: %s", h.name, e)

    # Risk / return scatter (volatility vs period return).
    try:
        pts = []
        for h in holdings:
            vol = h.metrics.get("Volatility % (ann.)")
            ret = h.metrics.get("Period Return %")
            if isinstance(vol, (int, float)) and isinstance(ret, (int, float)):
                pts.append((h.name or h.symbol, vol, ret))
        if pts:
            fig, ax = plt.subplots(figsize=(7.5, 5.5))
            for nm, v, r in pts:
                ax.scatter(v, r, s=45, alpha=0.8)
                ax.annotate(nm[:14], (v, r), fontsize=7,
                            xytext=(3, 3), textcoords="offset points")
            ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
            ax.set_xlabel("Annualised Volatility % (risk)")
            ax.set_ylabel("Historical Period Return %")
            ax.set_title("Risk vs Return (where data available)")
            path = os.path.join(out_dir, "chart_risk_return.png")
            fig.tight_layout()
            fig.savefig(path, dpi=130)
            plt.close(fig)
            charts.append(path)
    except Exception as e:
        LOG.warning("Risk/return chart failed: %s", e)

    LOG.info("Generated %d chart image(s).", len(charts))
    return charts


# Module-level handle used by chart/portfolio helpers (set in main()).
_COLLECTOR: Optional[DataCollector] = None


# -----------------------------------------------------------------------------
# 12. PROFESSIONAL PDF REPORT GENERATION  (ReportLab)
# -----------------------------------------------------------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph,
                                Spacer, Table, TableStyle, Image, PageBreak)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

PDF_FONT = "Helvetica"
PDF_FONT_BOLD = "Helvetica-Bold"
RUPEE_OK = False          # base Helvetica has no ₹ glyph
_FONT_RESOLVED = False

def _resolve_fonts() -> None:
    """
    Use a Unicode TrueType font (DejaVu Sans, shipped with matplotlib) so the ₹
    symbol and other glyphs render. Falls back to Helvetica + 'Rs.' text.
    """
    global PDF_FONT, PDF_FONT_BOLD, RUPEE_OK, _FONT_RESOLVED
    if _FONT_RESOLVED:
        return
    _FONT_RESOLVED = True
    try:
        from matplotlib import font_manager as fm
        regular = bold = None
        try:
            for p in fm.findSystemFonts():
                n = os.path.basename(p).lower()
                if not n.endswith(".ttf"):
                    continue
                if n in ("dejavusans.ttf", "notosansregular.ttf", "latha.ttf"):
                    regular = regular or p
                if n in ("dejavusans-bold.ttf", "notosansbold.ttf"):
                    bold = p
        except Exception:
            pass
        if not regular:
            cand = fm.findfont("DejaVu Sans")
            regular = cand if os.path.exists(cand) else None
        if not regular:
            return
        pdfmetrics.registerFont(TTFont("UnicodeSans", regular))
        PDF_FONT = "UnicodeSans"
        if bold and os.path.exists(bold):
            pdfmetrics.registerFont(TTFont("UnicodeSans-Bold", bold))
            PDF_FONT_BOLD = "UnicodeSans-Bold"
            pdfmetrics.registerFontFamily(
                "UnicodeSans", normal="UnicodeSans", bold="UnicodeSans-Bold",
                italic="UnicodeSans", boldItalic="UnicodeSans-Bold")
        RUPEE_OK = True
    except Exception as e:
        LOG.debug("Unicode PDF font unavailable, using Rs. fallback: %s", e)

import re as _re_amp
def _fix_amp(text: Any) -> str:
    """Escape lone '&' and stray '<' so ReportLab's mini-XML parser does not choke
    on real company names / rule texts (e.g. 'ARE&M', '< 1.5'), while preserving
    intended entities (&#8226;, &#9744;) and tags (<b>, <br/>)."""
    s = str(text)
    s = _re_amp.sub(r"&(?!(?:#\d+;|\w+;))", "&amp;", s)
    s = _re_amp.sub(r"<(?![a-zA-Z/!])", "&lt;", s)
    if not RUPEE_OK and "\u20b9" in s:
        s = s.replace("\u20b9", "Rs.")   # no Unicode font -> avoid blank boxes
    return s

def _plain(text: Any) -> str:
    """Canvas (non-XML) text: only fix the rupee glyph, never escape markup."""
    s = str(text)
    if not RUPEE_OK and "\u20b9" in s:
        s = s.replace("\u20b9", "Rs.")
    return s

class NumberedCanvas:
    """Mixin-free page numbering via a custom DocTemplate."""

class ReportDoc(BaseDocTemplate):
    def __init__(self, filename, title, **kw):
        super().__init__(filename, pagesize=A4,
                         leftMargin=16*mm, rightMargin=16*mm,
                         topMargin=18*mm, bottomMargin=18*mm, title=title, **kw)
        self.doc_title = title
        frame = Frame(self.leftMargin, self.bottomMargin,
                      self.width, self.height, id="main")
        self.addPageTemplates([PageTemplate(id="t", frames=[frame],
                             onPage=self._decorate)])

    def _decorate(self, canvas, doc):
        canvas.saveState()
        # Header
        canvas.setFont(PDF_FONT_BOLD, 9)
        canvas.setFillColor(colors.HexColor("#1b4965"))
        canvas.drawString(doc.leftMargin, A4[1] - 12*mm, "Portfolio Analyzer")
        canvas.setFont(PDF_FONT, 8)
        canvas.setFillColor(colors.grey)
        canvas.drawRightString(A4[0] - doc.rightMargin, A4[1] - 12*mm,
                               _plain(self.doc_title))
        canvas.setStrokeColor(colors.HexColor("#cfd8dc"))
        canvas.line(doc.leftMargin, A4[1] - 14*mm, A4[0] - doc.rightMargin, A4[1] - 14*mm)
        # Footer with page number
        canvas.setFont(PDF_FONT, 8)
        canvas.setFillColor(colors.grey)
        canvas.drawString(doc.leftMargin, 10*mm,
                          _plain(f"Generated {Config.GENERATED_ON} | Not investment advice"))
        canvas.drawRightString(A4[0] - doc.rightMargin, 10*mm, f"Page {doc.page}")
        canvas.restoreState()

class PDFBuilder:
    BRAND = colors.HexColor("#1b4965")
    LIGHT = colors.HexColor("#e8f1f5")

    def __init__(self, title: str, fname: str, out_dir: str):
        _resolve_fonts()
        self.font, self.font_bold = PDF_FONT, PDF_FONT_BOLD
        self.styles = getSampleStyleSheet()
        self.h1 = ParagraphStyle("h1", parent=self.styles["Heading1"],
                                 fontName=self.font_bold, textColor=self.BRAND,
                                 fontSize=16, spaceAfter=6)
        self.h2 = ParagraphStyle("h2", parent=self.styles["Heading2"],
                                 fontName=self.font_bold, textColor=self.BRAND,
                                 fontSize=12, spaceBefore=8, spaceAfter=4)
        self.body = ParagraphStyle("body", parent=self.styles["BodyText"],
                                   fontName=self.font, fontSize=9, leading=12,
                                   alignment=TA_LEFT)
        self.small = ParagraphStyle("small", parent=self.styles["BodyText"],
                                    fontName=self.font, fontSize=7.5, leading=9.5,
                                    textColor=colors.grey)
        self.title_txt = title
        self.path = os.path.join(out_dir, fname)
        self.story: List[Any] = []

    def title_block(self, subtitle=""):
        self.story.append(Paragraph(_fix_amp(self.title_txt), self.h1))
        if subtitle:
            self.story.append(Paragraph(_fix_amp(subtitle), self.body))
        self.story.append(Paragraph(f"Generated: {Config.GENERATED_ON}", self.small))
        self.story.append(Spacer(1, 6))

    def para(self, text): self.story.append(Paragraph(_fix_amp(text), self.body))
    def heading(self, text): self.story.append(Paragraph(_fix_amp(text), self.h2))
    def spacer(self, h=6): self.story.append(Spacer(1, h))
    def pagebreak(self): self.story.append(PageBreak())

    def table(self, rows: List[List[Any]], col_widths: Optional[List[float]] = None,
              font_size: float = 7.5):
        if not rows:
            return
        # Wrap cells as Paragraphs so long text wraps within page width.
        def cell(v, style):
            return Paragraph(_fix_amp(v), style)
        hdr_style = ParagraphStyle("th", parent=self.body, fontSize=font_size,
                                   leading=font_size+2, textColor=colors.white,
                                   fontName=self.font_bold)
        td_style = ParagraphStyle("td", parent=self.body, fontSize=font_size,
                                  leading=font_size+2, fontName=self.font)
        wrapped = []
        for ri, row in enumerate(rows):
            st = hdr_style if ri == 0 else td_style
            wrapped.append([cell(c, st) for c in row])
        avail = A4[0] - 32*mm
        if col_widths is None:
            ncol = len(rows[0])
            col_widths = [avail / ncol] * ncol
        else:
            total = sum(col_widths)
            col_widths = [w / total * avail for w in col_widths]
        t = Table(wrapped, colWidths=col_widths, repeatRows=1, hAlign="LEFT")
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), self.BRAND),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, self.LIGHT]),
            ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#b0bec5")),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3),
            ("TOPPADDING", (0, 0), (-1, -1), 2),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ]))
        self.story.append(t)
        self.spacer(8)

    def image(self, path: str, max_w_mm=170, max_h_mm=95):
        if not path or not os.path.exists(path):
            return
        try:
            from PIL import Image as PILImage
            with PILImage.open(path) as im:
                w, h = im.size
            scale = min((max_w_mm*mm)/w, (max_h_mm*mm)/h)
            self.story.append(Image(path, width=w*scale, height=h*scale))
            self.spacer(8)
        except Exception as e:
            LOG.debug("Image embed failed %s: %s", path, e)

    def disclaimer(self):
        self.spacer(6)
        self.story.append(Paragraph(
            "DISCLAIMER: This report is an automated, data-driven analysis for "
            "research and educational purposes only. It is NOT investment advice, "
            "and NOT a recommendation to buy/sell. Data may be delayed, historical "
            "or incomplete; unavailable metrics are shown as N/A. Verify against "
            "authoritative sources before acting.", self.small))

    def build(self):
        doc = ReportDoc(self.path, self.title_txt)
        doc.build(self.story)
        return self.path


def _fmt_metric(v: Any) -> str:
    if v is None:
        return Config.NA
    if isinstance(v, float):
        if abs(v) >= 1e7:
            return fmt_inr(v)
        return f"{v:,.2f}"
    if isinstance(v, int):
        return f"{v:,}"
    return str(v)

def generate_master_pdf(holdings: List[Holding], ps: PortfolioSummary,
                        charts: List[str], out_dir: str) -> List[str]:
    """
    Produce ONE comprehensive PDF: portfolio summary + risk + a full A-to-Z
    analysis section for every single holding (rule tables, all metrics, chart,
    decision table and monitoring). Returns [path], or [] on failure.
    """
    os.makedirs(out_dir, exist_ok=True)
    try:
        b = PDFBuilder("Consolidated Portfolio Analysis Report",
                       "Portfolio_Analysis_Report.pdf", out_dir)
        b.title_block(
            f"Single-file detailed report | {ps.n_holdings} holding(s) | "
            f"{ps.n_identified} identified | {ps.n_analyzed} analysed | Indian market")
        _exec_section(b, holdings, ps, charts)
        _risk_section(b, holdings, ps)
        _all_holdings_register(b, holdings, ps)

        order = [CLASS_STOCKS, CLASS_BANKS_FI, CLASS_ETF, CLASS_MF, CLASS_UNKNOWN]
        grouped: List[Holding] = []
        for cls in order:
            grouped += sorted([h for h in holdings if h.asset_class == cls],
                              key=lambda x: (x.current_value or x.invested or 0),
                              reverse=True)
        grouped += [h for h in holdings if h.asset_class not in order]
        b.pagebreak()
        b.heading("Part II - Detailed analysis, holding by holding")
        b.para("Each holding gets its own section: position data, the rule-based "
               "scorecard (Steps 1-4) where fundamentals are available, every metric "
               "collected (including N/A items), the decision table, what would change "
               "the signal, and a monitoring checklist. Sections are ordered by asset "
               "class, then by market value.")
        for idx, h in enumerate(grouped, 1):
            try:
                _holding_section(b, h, ps, idx)
            except Exception as e:
                LOG.warning("Detail section failed for %s: %s", h.name, e)
                b.pagebreak()
                b.heading(f"Holding {idx}: {_short(h.name or h.symbol)}")
                b.para(f"This section could not be rendered: {e}")
        path = b.build()
        LOG.info("Generated 1 comprehensive PDF with %d holding section(s).", len(grouped))
        return [path]
    except Exception as e:
        LOG.error("Master PDF generation failed: %s", e)
        return []

def _slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")[:40] or "holding"

def _short(s: str) -> str:
    s = str(s)
    return s if len(s) <= 22 else s[:20] + ".."

def _all_holdings_register(b: PDFBuilder, holdings: List[Holding], ps: PortfolioSummary):
    """One register listing EVERY uploaded row so nothing is unaccounted for."""
    b.pagebreak()
    b.heading("Part I (cont.) - Register of all uploaded holdings")
    rows = [["#", "Symbol", "Name", "Class", "Sector / Category", "Qty",
             "Invested", "Value", "P/L %", "Weight %", "Assessment", "Conf."]]
    for i, h in enumerate(holdings, 1):
        w = safe_div(h.current_value, ps.total_current_value)
        rows.append([i, h.symbol or Config.NA, _short(h.name), _short(h.asset_class),
                     _short(h.sector if h.sector != Config.NA else h.sub_category),
                     _fmt_metric(h.quantity), fmt_inr(h.invested),
                     fmt_inr(h.current_value),
                     (f"{h.pnl_pct:.1f}" if h.pnl_pct is not None else Config.NA),
                     (f"{w*100:.1f}" if w is not None else Config.NA),
                     h.assessment, h.confidence])
    b.table(rows, col_widths=[3, 11, 16, 11, 14, 8, 11, 11, 6, 7, 9, 6], font_size=6.5)
    b.para("Every row from the uploaded file appears here. Holdings that could not be "
           "identified are listed under 'Unidentified' with their data issues instead "
           "of being guessed.")

_STEP_NOTE = {
    "Step 1: Eliminating low quality and risk":
        "Filters weed out risky companies before spending time on analysis. "
        "Debt/equity and interest coverage are NOT applicable to banks and NBFCs "
        "(deposits and bonds are their raw material). Promoter holding and pledging "
        "come from BSE/NSE shareholding filings, which the free data source does not "
        "provide - those rows stay N/A.",
    "Step 2: Is it profitable?":
        "ROCE/ROE averages and margins are COMPUTED from the last available annual "
        "statements (yfinance). Check ROE together with ROCE: high ROE on low ROCE "
        "means the return is running on debt. Cash-flow margin and free cash flow show "
        "whether profits are real cash.",
    "Step 3: Is it growing?":
        "CAGRs use every annual period the data source supplies (usually 4-5 years). "
        "The profit triangle needs revenue, profit AND cash flow to rise together; if "
        "profit rises without cash flow it is only paper profit.",
    "Step 4: Is the price right?":
        "Valuation last, and only last. Sector P/E and P/B comparisons need a sector "
        "multiple source (Screener/Sharpely) that is not in the free feed, so those "
        "rows are marked WATCH with the stock's own multiple shown - compare it "
        "against peers before acting.",
}

def _change_conditions(rules: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    """Concrete, data-derived conditions that would flip the signal either way."""
    upgrade: List[str] = []
    for _, rows in rules["steps"]:
        for r in rows:
            if r["signal"] == SIG_FAIL:
                upgrade.append(f"{r['check']}: {r['rule']} (now {r['value']})")
    upgrade.append("Quarterly results confirm margins/profit recovering while "
                   "quality checks stay intact")
    downgrade = [
        "Any Step 1 check turning FAIL (leverage up, coverage down, pledge appearing)",
        "Promoter pledging above 15% or continuous promoter selling in filings",
        "ROCE below 15% or interest coverage below 3",
        "Revenue declining two consecutive quarters",
        "More than 2 red flags on a Screener/Tickertape screen",
    ]
    return upgrade[:8], downgrade

def _instrument_rule_rows(h: Holding, m: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Rule rows for instruments where the 6-step equity rules do NOT apply
    (ETFs, mutual funds, unclassified rows). Honest N/A for anything the free
    data cannot supply.
    """
    def num(v):
        return v if isinstance(v, (int, float)) else None
    rows: List[Dict[str, str]] = []
    if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI):
        # Equity/bank holding whose statements could not be fetched (offline run,
        # unresolved ticker or failed lookup): report it without inventing a score.
        rows.append(_rule("Row accounted for", "Every uploaded row is reported",
                          f"Row {h.source_row}: {h.raw_name or h.symbol or '(blank)'}",
                          SIG_PASS))
        rows.append(_rule("Market data & annual statements",
                          "Needed to score Steps 1-4 of the rule card",
                          "N/A - no data returned for this symbol (lookup failed, "
                          "unresolved ticker or offline run). Re-run with internet "
                          "access; nothing was guessed.", SIG_NA))
    elif h.asset_class == CLASS_ETF:
        px, sma = num(m.get("Last Price")), num(m.get("SMA200"))
        rows.append(_rule("Trend vs 200-DMA", "At/above long-term average",
                          (f"{_fmt_metric(px)} vs SMA200 {_fmt_metric(sma)}"),
                          _sig(bool(px and sma and px >= sma), not (px and sma))))
        r1, r3, r5 = num(m.get("1Y Return %")), num(m.get("3Y Return %")), num(m.get("5Y Return %"))
        have = [v for v in (r1, r3, r5) if v is not None]
        rows.append(_rule("1Y / 3Y / 5Y returns", "Positive over the holding horizon",
                          f"1Y {_pc(r1)}, 3Y {_pc(r3)}, 5Y {_pc(r5)}",
                          _sig(bool(have) and all(v > 0 for v in have), not have)))
        vol = num(m.get("Volatility % (ann.)"))
        rows.append(_rule("Volatility", "Roughly matches an index instrument",
                          _pc(vol), SIG_WATCH if vol else SIG_NA))
        er = num(m.get("Expense Ratio %"))
        rows.append(_rule("Expense ratio", "Lower is better for an ETF",
                          _pc(er, 2) if er is not None else Config.NA,
                          SIG_NA if er is None else SIG_PASS))
        aum = num(m.get("Total Assets (AUM)"))
        rows.append(_rule("AUM / liquidity", "Large enough to trade tight",
                          _cr(aum / 1e7 if aum else None),
                          SIG_NA if aum is None else SIG_PASS))
        for k, lbl in (("Tracking Error", "vs the index it copies"),
                       ("Premium/Discount to NAV %", "Trade price vs NAV"),
                       ("Underlying Index", "Know exactly what you own"),
                       ("Top Holdings", "Overlap with direct stock holdings")):
            rows.append(_rule(k, lbl, "N/A (fund factsheet not in the free data)", SIG_NA))
    elif h.asset_class == CLASS_MF:
        rows.append(_rule("Position recorded", "Units & invested value present",
                          f"{Config.CURRENCY_SYMBOL}{_fmt_metric(h.quantity)} units at avg "
                          f"{_fmt_metric(h.avg_price)}",
                          SIG_PASS if h.quantity else SIG_FAIL))
        rows.append(_rule("Broker valuation", "Current value / P&L from the file",
                          f"{fmt_inr(h.current_value)} ({_pc(h.pnl_pct)})",
                          SIG_PASS if h.current_value else SIG_NA))
        rows.append(_rule("Live NAV", "Independent NAV check",
                          "N/A (no free NAV feed for Indian schemes here)", SIG_NA))
        for k, lbl in (("1Y/3Y/5Y & rolling returns", "Scheme performance vs benchmark"),
                       ("Expense Ratio %", "Direct cheaper than regular"),
                       ("AUM / fund manager / AMC", "Size, stability, manager tenure"),
                       ("Sharpe / Std dev / Max drawdown", "Risk-adjusted behaviour"),
                       ("Top holdings / overlap", "Concentration and duplication"),
                       ("Exit load", "Cost of exiting early")):
            rows.append(_rule(k, lbl, "N/A (AMFI/AMC factsheet required)", SIG_NA))
    else:
        rows.append(_rule("Row accounted for", "Every uploaded row is reported",
                          f"Row {h.source_row}: {h.raw_name or h.symbol or '(blank)'}",
                          SIG_PASS))
        rows.append(_rule("Identification", "Symbol/ISIN resolvable to a real instrument",
                          ("; ".join(h.issues) or "Not matched by name, ISIN or symbol"),
                          SIG_FAIL))
        rows.append(_rule("Valuation", "Price and returns",
                          "N/A (instrument not identified - nothing was guessed)", SIG_NA))
    rows.append(_rule("Position P/L", "Review any deep unrealised loss",
                      _pc(h.pnl_pct),
                      SIG_PASS if (h.pnl_pct is None or h.pnl_pct >= -25) else SIG_WATCH))
    # Fund instruments are not 'bought more' on a rule pass - use neutral actions.
    fund_action = {SIG_PASS: "OK / CONTINUE", SIG_FAIL: "REVIEW", SIG_WATCH: "MONITOR",
                   SIG_NA: "VERIFY"}
    for r in rows:
        r["action"] = fund_action.get(r["signal"], "MONITOR")
    return rows

def _holding_section(b: PDFBuilder, h: Holding, ps: PortfolioSummary, idx: int):
    """
    One complete A-to-Z section per holding, laid out like the user's 6-step
    demo: snapshot -> rule score -> Step 1..4 tables -> decision table -> what
    would change the signal -> every collected metric -> data source and
    limitations -> chart -> monitoring checklist.
    """
    m = h.metrics
    rules = h.analysis.get("rules")
    weight = safe_div(h.current_value, ps.total_current_value)
    cat = h.sector if h.sector != Config.NA else h.sub_category
    b.pagebreak()
    b.heading(f"Holding {idx}: {h.name or h.symbol} ({h.symbol or Config.NA})")
    b.para(f"{h.asset_class} | {cat} | ISIN {h.ISIN or Config.NA} | classified via "
           f"{h.classification_source} (confidence {h.confidence}) | uploaded row "
           f"{h.source_row} | currency {h.currency}")

    def mv(key):
        return _fmt_metric(m.get(key, Config.NA))
    mc = m.get("Market Cap")
    mc_cr = (mc / 1e7) if isinstance(mc, (int, float)) else None
    b.heading("Snapshot")
    b.table([["Metric", "Value", "Metric", "Value"],
             ["Latest price", mv("Last Price"), "Market cap", _cr(mc_cr)],
             ["52-week high / low", f"{mv('52W High')} / {mv('52W Low')}",
              "P/E | P/B", f"{mv('P/E')} | {mv('P/B')}"],
             ["Dividend yield", mv("Dividend Yield %"), "SMA50 / SMA200",
              f"{mv('SMA50')} / {mv('SMA200')}"],
             ["1Y / 3Y / 5Y return",
              f"{mv('1Y Return %')} / {mv('3Y Return %')} / {mv('5Y Return %')}",
              "Volatility (ann.)", mv("Volatility % (ann.)")],
             ["Quantity / units", _fmt_metric(h.quantity),
              "Avg cost / CMP", f"{_fmt_metric(h.avg_price)} / {_fmt_metric(h.cmp)}"],
             ["Invested", fmt_inr(h.invested), "Current value", fmt_inr(h.current_value)],
             ["Unrealised P/L",
              (f"{fmt_inr(h.pnl)} ({h.pnl_pct:.1f}%)" if h.pnl_pct is not None
               else fmt_inr(h.pnl)),
              "Portfolio weight", f"{weight*100:.2f}%" if weight is not None else Config.NA],
             ], col_widths=[21, 29, 21, 29], font_size=7)

    if rules:
        b.heading("Rule-based score and overall signal")
        pct_txt = f" ({rules['ratio_pct']:.0f}%)" if rules["ratio_pct"] is not None else ""
        b.table([["Score from your rules (N/A checks are excluded)", "Overall signal"],
                 [f"{rules['pass']} pass, {rules['fail']} fail, {rules['watch']} watch, "
                  f"{rules['na']} N/A  ->  {rules['pass']} of {rules['applicable']} "
                  f"applicable checks passed{pct_txt}", rules["signal"]]],
                col_widths=[74, 26], font_size=7.5)
        n = 0
        for title, rrows in rules["steps"]:
            b.heading(title)
            b.para(_STEP_NOTE.get(title, ""))
            rows = [["#", "Check", "Your rule", "Data & note (computed here)",
                     "Signal", "Action"]]
            for r in rrows:
                n += 1
                rows.append([n, r["check"], r["rule"], r["value"], r["signal"], r["action"]])
            b.table(rows, col_widths=[4, 16, 18, 40, 8, 9], font_size=6.8)
    else:
        rows = _instrument_rule_rows(h, m)
        if rows:
            b.heading("Rule checks for this instrument")
            if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI):
                b.para("The 6-step equity rule card could not be scored for this "
                       "holding because no market data / annual statements were "
                       "returned. The row is still fully reported - it is NOT dropped "
                       "and no number is invented for it.")
            else:
                b.para("The equity 6-step rules (ROCE, promoter holding, debt/equity, "
                       "margins) do NOT apply to this asset class - it is analysed with "
                       "its own checks, and anything the free data cannot supply stays "
                       "N/A instead of being guessed.")
            trows = [["#", "Check", "Rule / what to look for", "Data & note",
                      "Status", "Action"]]
            for i, r in enumerate(rows, 1):
                trows.append([i, r["check"], r["rule"], r["value"], r["signal"], r["action"]])
            b.table(trows, col_widths=[4, 16, 20, 38, 8, 9], font_size=6.8)

    b.heading("Final decision table")
    label = _short(h.name or h.symbol)
    if rules:
        pct = rules["ratio_pct"]
        pct_s = f"{pct:.0f}%" if pct is not None else Config.NA
        s1_note = ("all applicable Step 1 checks pass" if rules["step1_all_pass"]
                   else "Step 1 is not fully clean")
        b.table([["Rule (edit to your own style)", f"Result for {label}"],
                 ["BUY MORE if >= 85% of applicable checks pass and Step 1 is clean",
                  ("Met: " if rules["signal"] == "BUY MORE" else "Not met: ") +
                  f"{rules['pass']} of {rules['applicable']} pass ({pct_s}), {s1_note}"],
                 ["HOLD if >= 55% of applicable checks pass",
                  ("Met: " if rules["signal"] == "HOLD" else "Not met: ") +
                  f"{rules['pass']} of {rules['applicable']} pass ({pct_s})"],
                 ["SELL if < 55% pass or any Step 1 check fails",
                  ("Met: " if rules["signal"] == "SELL" else "Not met: ") +
                  f"{rules['fail']} failed check(s), {s1_note}"],
                 ["FINAL SIGNAL", rules["signal"]],
                 ["Why", "; ".join(h.assessment_reasons) or Config.NA]],
                col_widths=[48, 52], font_size=7)
        up, down = _change_conditions(rules)
        b.heading("What would change the signal")
        b.table([["Move to BUY MORE if", "Move to SELL if"],
                 ["<br/>".join(f"&#9642; {u}" for u in up),
                  "<br/>".join(f"&#9642; {d}" for d in down)]],
                col_widths=[50, 50], font_size=7)
    else:
        b.table([["Outcome", f"Result for {label}"],
                 ["Assessment from the " + str(h.asset_class) + " engine", h.assessment],
                 ["Why", "; ".join(h.assessment_reasons) or
                  "No assessment possible - the row could not be identified, so no "
                  "data was invented for it."]],
                col_widths=[40, 60], font_size=7)

    b.heading("Every metric collected (A to Z)")
    metric_rows = [["Metric", "Value", "Metric", "Value"]]
    items = [(k, v) for k, v in m.items() if not str(k).startswith("_")]
    half = (len(items) + 1) // 2
    for i in range(half):
        left = items[i]
        right = items[i + half] if i + half < len(items) else ("", "")
        metric_rows.append([left[0], _fmt_metric(left[1]), right[0], _fmt_metric(right[1])])
    if len(items):
        b.table(metric_rows, col_widths=[28, 22, 28, 22], font_size=6.8)
    else:
        b.para("No metrics available for this holding.")
    b.para(f"Data source: {m.get('_data_source', Config.NA)} | prices are "
           f"delayed/historical, not real-time.")
    if h.issues:
        b.heading("Data quality notes")
        for i in h.issues:
            b.para(f"&#8226; {i}")
    price_img = None
    for c in _CHARTS_INDEX.get(h.symbol or h.name, []):
        price_img = c
        break
    b.heading("Historical price chart")
    if price_img and os.path.exists(price_img):
        b.image(price_img)
    else:
        b.para("No price chart for this holding (no market data source matched it).")
    b.heading("Monitoring checklist")
    b.para(_monitoring_checklist(h))
    b.spacer(4)
    b.disclaimer()

def _monitoring_checklist(h: Holding) -> str:
    if h.asset_class == CLASS_MF:
        items = ["Compare vs benchmark & category on rolling 3/5Y returns",
                 "Track expense ratio, fund-manager changes, style drift",
                 "Review AUM changes and liquidity for debt schemes"]
    elif h.asset_class == CLASS_ETF:
        items = ["Monitor tracking error vs index", "Watch premium/discount to NAV & spreads",
                 "Check AUM/liquidity and overlap with direct holdings"]
    elif h.asset_class == CLASS_BANKS_FI:
        items = ["Track GNPA/NNPA, NIM, CASA, provisions & CAR in quarterly filings",
                 "Watch credit/deposit growth and slippages",
                 "Review management/ownership changes and regulatory actions"]
    else:
        items = ["Track revenue, profit, margin & ROE trends quarterly",
                 "Watch debt, cash flows, promoter pledging & governance",
                 "Reassess valuation vs earnings growth and peer multiples"]
    return "<br/>".join(f"&#9744; {i}" for i in items)

# Populated in main(): maps holding symbol/name -> its price chart path(s).
_CHARTS_INDEX: Dict[str, List[str]] = {}


def _exec_section(b: PDFBuilder, holdings, ps: PortfolioSummary, charts):
    """Part I of the single report: portfolio-level numbers and allocation."""
    b.heading("Part I - Portfolio summary")
    b.heading("Headline Numbers")
    b.table([["Metric", "Value"],
             ["Total holdings", ps.n_holdings],
             ["Identified holdings", ps.n_identified],
             ["Analyzed holdings", ps.n_analyzed],
             ["Failed data lookups", ps.n_failed_lookups],
             ["Total invested", fmt_inr(ps.total_invested)],
             ["Current market value", fmt_inr(ps.total_current_value)],
             ["Unrealised P/L", fmt_inr(ps.unrealized_pnl)],
             ["Unrealised return %", (f"{ps.unrealized_return_pct:.2f}%"
                 if ps.unrealized_return_pct is not None else Config.NA)],
             ["Dividend income (est.)", (fmt_inr(ps.dividend_income)
                 if ps.dividend_income is not None else Config.NA)],
             ], col_widths=[45, 55])
    b.heading("Allocation")
    b.image(_find_chart(charts, "chart_asset_allocation.png"))
    b.image(_find_chart(charts, "chart_sector_allocation.png"))
    b.heading("Asset-Class Allocation")
    b.table([["Asset Class", "Value", "Weight %"]] +
            [[k, fmt_inr(sum(h.current_value or h.invested or 0 for h in holdings if h.asset_class == k)),
              ps.asset_allocation.get(k, 0)] for k in ps.asset_allocation],
            col_widths=[45, 30, 25])
    b.heading("Top Holdings & Concentration")
    b.image(_find_chart(charts, "chart_top_holdings.png"))
    rows = [["#", "Holding", "Value", "Weight %"]]
    for i, (nm, mv, pct) in enumerate(ps.top_holdings[:10], 1):
        rows.append([i, nm, fmt_inr(mv), pct])
    b.table(rows, col_widths=[8, 50, 25, 17])
    b.para(f"Top-5 concentration: {ps.concentration_top5_pct:.1f}% | "
           f"Financial-sector exposure: {ps.financial_sector_exposure_pct:.1f}% | "
           f"Domestic: {ps.domestic_pct:.1f}% / International: {ps.international_pct:.1f}%")
    if ps.duplicate_exposures:
        b.para(f"Possible duplicate exposures: {', '.join(ps.duplicate_exposures)}")
    if ps.valuation_notes:
        b.heading("Valuation Notes / Limitations")
        for n in ps.valuation_notes:
            b.para(f"&#8226; {n}")
    b.heading("How this report is built (method & data honesty)")
    for t in [
        "Each asset class is analysed with its OWN engine - individual stocks, "
        "banks/NBFCs/insurance, ETFs and mutual funds never share one scoring formula.",
        "The rule card implements Steps 1-4 (20 checks) exactly as written in the "
        "sample review. Steps 5 and 6 of the notebook method are not defined yet - "
        "they are left blank instead of being invented; add them and the score expands.",
        "Equity rule scores are computed from the last available annual statements "
        "(revenue, EBITDA, net profit, equity, debt, cash flow) pulled by yfinance, "
        "plus prices/valuation multiples. Checks the free data cannot support "
        "(promoter holding, pledging, sector P/E, red flags, bank NPA/CAR, scheme "
        "factsheets) are printed as N/A and EXCLUDED from the score - never guessed.",
        "No ETF/mutual-fund underlying holdings are expanded into the stock sections, "
        "so nothing is double counted; fund-level overlap needs factsheet holdings.",
        "Prices are delayed/historical, not real-time. Every calculation shown can be "
        "reproduced from the printed inputs.",
        "This is a self-study checklist built from your own rules and publicly "
        "available numbers. It is NOT investment advice and no forced BUY/SELL "
        "target is issued; verify with BSE/NSE filings and a SEBI-registered advisor "
        "before acting.",
    ]:
        b.para(f"&#8226; {t}")
    if _COLLECTOR and _COLLECTOR.failures:
        b.heading("Failed / unverified lookups")
        for f in sorted(set(_COLLECTOR.failures))[:30]:
            b.para(f"&#8226; {f}")

def _risk_section(b: PDFBuilder, holdings, ps: PortfolioSummary):
    b.heading("Risk, concentration & diversification")
    b.heading("Concentration")
    b.table([["Risk Metric", "Value"],
             ["Top-5 holdings weight %", f"{ps.concentration_top5_pct:.1f}%"],
             ["Largest single holding %", (f"{ps.top_holdings[0][2]:.1f}%" if ps.top_holdings else Config.NA)],
             ["Financial-sector exposure %", f"{ps.financial_sector_exposure_pct:.1f}%"],
             ["International exposure %", f"{ps.international_pct:.1f}%"],
             ["Number of distinct asset classes", len([k for k,v in ps.asset_allocation.items() if v>0])],
             ], col_widths=[55, 45])
    b.para("Concentration guidance (transparent, not a rule): a single holding above "
           "~10-15% or top-5 above ~50% increases idiosyncratic risk. Review against "
           "your own risk tolerance.")
    b.heading("Duplicate / Overlapping Exposure (symbol-level)")
    if ps.duplicate_exposures:
        b.para("The following symbols appear more than once across uploaded rows. Note: "
               "underlying holdings of ETFs/MFs are NOT auto-expanded here to avoid "
               "double counting - overlap between an ETF/MF and direct stocks requires "
               "factsheet holdings data (shown as N/A when unavailable).")
        b.table([["Symbol appearing multiple times"]] +
                [[s] for s in ps.duplicate_exposures], col_widths=[100])
    else:
        b.para("No exact duplicate symbols detected among uploaded rows.")
    b.image(_find_chart(_ALL_CHARTS, "chart_risk_return.png"))
    b.heading("Data-Driven Risk Observations")
    obs = []
    vols = [h.metrics.get("Volatility % (ann.)") for h in holdings
            if isinstance(h.metrics.get("Volatility % (ann.)"), (int, float))]
    if vols:
        hi = [h for h in holdings if isinstance(h.metrics.get("Volatility % (ann.)"), (int,float))
              and h.metrics["Volatility % (ann.)"] > 35]
        if hi:
            obs.append(f"{len(hi)} holding(s) show high annualised volatility (>35%): "
                       + ", ".join(_short(h.name or h.symbol) for h in hi[:8]))
    losses = [h for h in holdings if h.pnl_pct is not None and h.pnl_pct < -25]
    if losses:
        obs.append(f"{len(losses)} holding(s) are down >25% (temporary weakness vs "
                   "fundamental change should be assessed per report).")
    below = [h for h in holdings if isinstance(h.metrics.get("SMA200"), (int,float))
             and isinstance(h.metrics.get("Last Price"), (int,float))
             and h.metrics["Last Price"] < h.metrics["SMA200"]]
    if below:
        obs.append(f"{len(below)} holding(s) trading below their 200-DMA.")
    if not obs:
        obs.append("Insufficient market data to derive volatility/trend risk flags.")
    for o in obs:
        b.para(f"&#8226; {o}")

def _find_chart(chart_list, base):
    for c in chart_list:
        if os.path.basename(c) == base:
            return c
    return None

_ALL_CHARTS: List[str] = []


# -----------------------------------------------------------------------------
# 13. CSV / EXCEL EXPORTS
# -----------------------------------------------------------------------------
def export_tabular(holdings: List[Holding], ps: PortfolioSummary,
                   out_dir: str) -> List[str]:
    files: List[str] = []
    os.makedirs(out_dir, exist_ok=True)

    # Portfolio_Summary.csv
    summary_df = pd.DataFrame([{
        "Total Holdings": ps.n_holdings, "Identified": ps.n_identified,
        "Analyzed": ps.n_analyzed, "Failed Lookups": ps.n_failed_lookups,
        "Total Invested": ps.total_invested, "Current Value": ps.total_current_value,
        "Unrealized P/L": ps.unrealized_pnl, "Return %": ps.unrealized_return_pct,
        "Top5 Concentration %": ps.concentration_top5_pct,
        "Financial Sector %": ps.financial_sector_exposure_pct,
        "Domestic %": ps.domestic_pct, "International %": ps.international_pct,
    }])
    p = os.path.join(out_dir, "Portfolio_Summary.csv"); summary_df.to_csv(p, index=False); files.append(p)

    # Holdings_Classification.csv
    cls_df = pd.DataFrame([{
        "Source Row": h.source_row, "Uploaded Name": h.raw_name, "Symbol/ISIN": h.symbol,
        "Asset Class": h.asset_class, "Sector": h.sector, "Sub-Category": h.sub_category,
        "Classification Source": h.classification_source, "Confidence": h.confidence,
        "Issues": "; ".join(h.issues),
    } for h in holdings])
    p = os.path.join(out_dir, "Holdings_Classification.csv"); cls_df.to_csv(p, index=False); files.append(p)

    # Financial_Metrics.csv
    metric_rows = []
    for h in holdings:
        row = {"Holding": h.name or h.symbol, "Asset Class": h.asset_class,
               "Assessment": h.assessment}
        for k, v in h.metrics.items():
            if k.startswith("_"):
                continue
            row[k] = v
        metric_rows.append(row)
    met_df = pd.DataFrame(metric_rows)
    p = os.path.join(out_dir, "Financial_Metrics.csv"); met_df.to_csv(p, index=False); files.append(p)

    # Portfolio_Allocation.csv
    alloc_rows = [{"Category": k, "Type": "AssetClass", "Weight %": v}
                  for k, v in ps.asset_allocation.items()]
    alloc_rows += [{"Category": k, "Type": "Sector", "Weight %": v}
                   for k, v in ps.sector_allocation.items()]
    al_df = pd.DataFrame(alloc_rows)
    p = os.path.join(out_dir, "Portfolio_Allocation.csv"); al_df.to_csv(p, index=False); files.append(p)

    # Analysis_Results.xlsx (multi-sheet)
    try:
        xlsx = os.path.join(out_dir, "Analysis_Results.xlsx")
        with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
            summary_df.to_excel(xw, sheet_name="Summary", index=False)
            cls_df.to_excel(xw, sheet_name="Classification", index=False)
            met_df.to_excel(xw, sheet_name="Metrics", index=False)
            al_df.to_excel(xw, sheet_name="Allocation", index=False)
            pd.DataFrame([{"Holding": n, "Value": v, "Weight%": w}
                          for n, v, w in ps.top_holdings]).to_excel(
                xw, sheet_name="TopHoldings", index=False)
        files.append(xlsx)
    except Exception as e:
        LOG.warning("Excel export failed (%s); CSV exports still available.", e)
    return files


# -----------------------------------------------------------------------------
# 14. MAIN ORCHESTRATION
# -----------------------------------------------------------------------------
def _process_holdings(file: UploadedFile) -> List[Holding]:
    tables = extract_tables(file)
    holdings: List[Holding] = []
    any_mapped = False
    for table in tables:
        for n in table.notes:
            LOG.info("Extraction note: %s", n)
        clean, mapping = map_columns(table.dataframe, has_header=table.has_header)
        if not mapping:
            continue  # sheet without recognizable holdings columns (e.g. notes)
        any_mapped = True
        rows = build_holdings(clean, mapping, table.source_format)
        LOG.info("Sheet %s: mapped %d column(s), extracted %d holding row(s).",
                 table.notes[0] if table.notes else "?", len(mapping), len(rows))
        holdings.extend(rows)
    if not any_mapped:
        raise ValueError("Could not map any recognizable columns in the file.")
    if not holdings:
        raise ValueError(
            "Columns were recognized, but no data rows were found under them. "
            "Check that the sheet actually lists holdings below its header row.")
    holdings = _dedupe_holdings(holdings)
    for h in holdings:
        classify_holding(h)
    return holdings

def _dedupe_holdings(holdings: List[Holding]) -> List[Holding]:
    """
    Broker workbooks often contain an 'Equity' sheet, a 'Mutual Funds' sheet and
    a 'Combined' sheet that repeats both. De-duplicate by ISIN (falling back to
    symbol/name) keeping the first occurrence so nothing is double counted.
    """
    seen: set = set()
    out: List[Holding] = []
    dupes = 0
    for h in holdings:
        key = ("ISIN", h.ISIN) if h.ISIN else ("SYM", (h.symbol or h.name or "").upper())
        if key in seen:
            dupes += 1
            continue
        seen.add(key)
        out.append(h)
    if dupes:
        LOG.info("Removed %d duplicate row(s) across worksheets (matched by ISIN/symbol).",
                 dupes)
    return out

def run_analysis(file: UploadedFile) -> Dict[str, Any]:
    global _COLLECTOR, _ALL_CHARTS
    print("\n" + "=" * 70)
    print(f" Processing: {file.name}")
    print("=" * 70)

    print("\n[2/8] Extracting & validating holdings...")
    holdings = _process_holdings(file)
    print(f"      Extracted {len(holdings)} holding(s).")

    print("\n[3/8] Resolving market-data tickers & collecting data...")
    _COLLECTOR = DataCollector()
    for h in holdings:
        h.yf_ticker = _COLLECTOR.resolve_yf_ticker(h)

    print("\n[4/8] Running per-asset-class analysis engines...")
    orch = AnalysisOrchestrator(_COLLECTOR)
    for i, h in enumerate(holdings, 1):
        orch.analyze(h)
        if i % 5 == 0 or i == len(holdings):
            print(f"      analyzed {i}/{len(holdings)} ...")

    print("\n[5/8] Computing portfolio-level metrics...")
    ps = compute_portfolio(holdings)

    print("\n[6/8] Generating charts...")
    out_dir = os.path.abspath(Config.OUTPUT_DIR)
    os.makedirs(out_dir, exist_ok=True)
    _ALL_CHARTS = generate_charts(holdings, ps, out_dir)

    print("\n[7/8] Generating the single comprehensive PDF report...")
    pdfs = generate_master_pdf(holdings, ps, _ALL_CHARTS, out_dir)

    print("\n[8/8] Exporting CSV/Excel companions (optional downloads)...")
    tabular = export_tabular(holdings, ps, out_dir)

    return {"holdings": holdings, "summary": ps, "pdfs": pdfs,
            "tabular": tabular, "charts": _ALL_CHARTS,
            "report_pdf": pdfs[0] if pdfs else None,
            "out_dir": out_dir}

def print_execution_summary(result: Dict[str, Any]) -> None:
    ps: PortfolioSummary = result["summary"]
    holdings = result["holdings"]
    unidentified = [h for h in holdings if h.asset_class == CLASS_UNKNOWN]
    print("\n" + "=" * 70)
    print(" EXECUTION SUMMARY")
    print("=" * 70)
    print(f" Holdings uploaded           : {ps.n_holdings}")
    print(f" Successfully identified     : {ps.n_identified}")
    print(f" Analyzed                    : {ps.n_analyzed}")
    print(f" Failed data lookups         : {ps.n_failed_lookups}")
    print(f" Unidentified / ambiguous    : {len(unidentified)}")
    if unidentified:
        print("   -> " + ", ".join(_short(h.raw_name) for h in unidentified[:15]))
    print(f" Total invested              : {fmt_inr(ps.total_invested)}")
    print(f" Current market value        : {fmt_inr(ps.total_current_value)}")
    print(f" Unrealised P/L              : {fmt_inr(ps.unrealized_pnl)} "
          f"({(f'{ps.unrealized_return_pct:.2f}%' if ps.unrealized_return_pct is not None else 'N/A')})")
    print(f" Comprehensive PDF report    : {len(result['pdfs'])} file "
          f"(Portfolio_Analysis_Report.pdf)")
    print(f" Supporting files (CSV/XLSX) : {len(result['tabular'])}")
    print(f" Charts                      : {len(result['charts'])}")
    print(f" Output directory            : {result['out_dir']}")
    counts: Dict[str, int] = {}
    for h in holdings:
        counts[h.assessment] = counts.get(h.assessment, 0) + 1
    if counts:
        print(" Signal distribution (from the rule engines):")
        for k in sorted(counts, key=lambda x: -counts[x]):
            print(f"   - {k}: {counts[k]}")
    ruled = sum(1 for h in holdings if h.analysis.get("rules"))
    print(f" Holdings scored on the 6-step rule card: {ruled} "
          f"(the rest are funds/unidentified - they use their own checks)")
    if ps.valuation_notes:
        print("\n Limitations:")
        for n in ps.valuation_notes:
            print("   - " + n)

def provide_downloads(result: Dict[str, Any]) -> None:
    print("\n" + "=" * 70)
    print(" DOWNLOAD LINKS")
    print("=" * 70)
    pdf = result.get("report_pdf")
    if pdf and os.path.exists(pdf):
        print(f" Single comprehensive PDF report: {pdf} "
              f"({os.path.getsize(pdf):,} bytes)")
    else:
        print("[WARN] The PDF report could not be produced - check the log above.")
    if IS_COLAB:
        try:
            from google.colab import files as cf
            if pdf and os.path.exists(pdf):
                cf.download(pdf)
                print(" [Colab] PDF download triggered. Supporting CSV/Excel files "
                      "are in:", result["out_dir"])
        except Exception as e:
            print(" [Colab] Automatic download unavailable:", e)
            print("         Use the Files panel / files.download manually on:", pdf)
    else:
        print(" Local run: open the folder above to view the report.")

def main() -> Optional[Dict[str, Any]]:
    LOG.info("Portfolio Analyzer starting. Colab=%s", IS_COLAB)
    try:
        ocr_ok = ensure_system_ocr()
        LOG.info("OCR available for scanned PDFs: %s", ocr_ok)
        files = upload_holdings_file()
        chosen = resolve_multiple_uploads(files)
        result = run_analysis(chosen)
        print_execution_summary(result)
        provide_downloads(result)
        return result
    except (ValueError, FileNotFoundError) as e:
        LOG.error("Input problem: %s", e)
        print("\nPlease upload a valid holdings file (.xlsx/.xls/.csv/.pdf) and re-run.")
    except Exception as e:
        LOG.error("Unexpected error: %s", e)
        traceback.print_exc()
    return None


# -----------------------------------------------------------------------------
# 16. SELF-TESTS (safe, offline-friendly). Executed only via run_self_tests().
# -----------------------------------------------------------------------------
def make_sample_holdings_df() -> pd.DataFrame:
    """
    A realistic broker-style holdings table for testing (NOT used in live runs).
    Mirrors the columns of an Indian broker 'Combined Holdings Statement' export:
    metadata rows at the top, then a header row, and rows that mix equities,
    ETFs and mutual funds (ISIN prefix INE vs INF drives classification).
    """
    header = ["Symbol", "ISIN", "Sector", "Instrument Type", "Quantity Available",
              "Quantity Discrepant", "Quantity Long Term", "Quantity Pledged (Margin)",
              "Quantity Pledged (Loan)", "Average Price", "Previous Closing Price",
              "Unrealized P&L", "Unrealized P&L Pct.", "Exchange", "Currency"]
    rows = [
        ["HDFCBANK", "INE040A01019", "FINANCIAL SERVICES", "-", 40, 0, 0, 0, 0, 1450.5, 1700.2, 9908.0, 17.1, "NSE", "INR"],
        ["RELIANCE", "INE002A01018", "ENERGY", "-", 10, 0, 0, 0, 0, 2400.0, 2950.0, 5500.0, 22.9, "NSE", "INR"],
        ["TCS", "INE467B01029", "SOFTWARE SERVICES", "-", 5, 0, 0, 0, 0, 3300.0, 3900.0, 3000.0, 18.2, "NSE", "INR"],
        ["SUNPHARMA", "INE044A01036", "PHARMACEUTICALS", "-", 12, 0, 0, 0, 0, 1000.0, 1650.0, 7800.0, 65.0, "NSE", "INR"],
        ["M&M", "INE101A01026", "AUTOMOBILE", "-", 3, 0, 0, 0, 0, 2842.9, 2796.2, -140.1, -1.6, "NSE", "INR"],
        ["NIFTYBEES", "INF204K01E23", "ETF", "-", 100, 0, 0, 0, 0, 250.0, 290.0, 4000.0, 16.0, "NSE", "INR"],
        ["ITBEES", "INF204K01E23", "ETF", "-", 800, 0, 0, 0, 0, 34.11, 31.84, -1816.07, -6.65, "NSE", "INR"],
        ["TATAGOLD-E", "INF002K01B23", "ETF", "-", 400, 0, 0, 0, 0, 14.69, 14.42, -110.14, -1.87, "NSE", "INR"],
        ["HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN", "INF179K01WA6", "-",
         "Hybrid - Dynamic Asset Allocation/Balanced Advantage", 19.166, 0, 0, 0, 0, 553.04, 546.08, -133.31, -1.25, "AMFI", "INR"],
        ["JIOBLACKROCK FLEXI CAP FUND - DIRECT PLAN", "INF22M001093", "-",
         "Equity - Flexi Cap", 146.002, 0, 0, 0, 0, 10.27, 9.51, -110.95, -7.39, "AMFI", "INR"],
        ["TOTALLY UNKNOWN INSTRUMENT", "XYZ123", "", "-", 10, 0, 0, 0, 0, 100.0, 90.0, -100.0, -10.0, "", "INR"],
    ]
    meta = [["Client ID", "TEST123"], ["Combined Holdings Statement as on 2026-10-12"],
            ["Summary"], ["Invested Value", 500000], ["Present Value", 520000]]
    return pd.DataFrame(meta + [header] + rows, dtype=object)

def make_broker_workbook_bytes() -> bytes:
    """Build a multi-sheet broker-style .xlsx in memory (Equity/MF/Combined).
    The Combined sheet repeats both, to verify ISIN-based de-duplication."""
    eq = [
        ["Symbol", "ISIN", "Sector", "Quantity Available", "Quantity Discrepant",
         "Quantity Long Term", "Quantity Pledged (Margin)", "Quantity Pledged (Loan)",
         "Average Price", "Previous Closing Price", "Unrealized P&L", "Unrealized P&L Pct."],
        ["HDFCBANK", "INE040A01019", "FINANCIAL SERVICES", 40, 0, 0, 0, 0, 1450.5, 1700.2, 9908.0, 17.1],
        ["RELIANCE", "INE002A01018", "ENERGY", 10, 0, 0, 0, 0, 2400.0, 2950.0, 5500.0, 22.9],
        ["NIFTYBEES", "INF204K01E23", "ETF", 100, 0, 0, 0, 0, 250.0, 290.0, 4000.0, 16.0],
    ]
    mf = [
        ["Symbol", "ISIN", "Instrument Type", "Quantity Available", "Quantity Discrepant",
         "Quantity Pledged (Margin)", "Quantity Pledged (Loan)", "Average Price",
         "Previous Closing Price", "Unrealized P&L", "Unrealized P&L Pct."],
        ["JIOBLACKROCK FLEXI CAP FUND - DIRECT PLAN", "INF22M001093",
         "Equity - Flexi Cap", 146.002, 0, 0, 0, 10.27, 9.51, -110.95, -7.39],
    ]
    combined = [
        ["Symbol", "ISIN", "Sector", "Instrument Type", "Quantity Available",
         "Quantity Discrepant", "Quantity Long Term", "Quantity Pledged (Margin)",
         "Quantity Pledged (Loan)", "Average Price", "Previous Closing Price",
         "Unrealized P&L", "Unrealized P&L Pct."],
        ["HDFCBANK", "INE040A01019", "FINANCIAL SERVICES", "-", 40, 0, 0, 0, 0, 1450.5, 1700.2, 9908.0, 17.1],
        ["RELIANCE", "INE002A01018", "ENERGY", "-", 10, 0, 0, 0, 0, 2400.0, 2950.0, 5500.0, 22.9],
        ["NIFTYBEES", "INF204K01E23", "ETF", "-", 100, 0, 0, 0, 0, 250.0, 290.0, 4000.0, 16.0],
        ["JIOBLACKROCK FLEXI CAP FUND - DIRECT PLAN", "INF22M001093", "-",
         "Equity - Flexi Cap", 146.002, 0, 0, 0, 0, 10.27, 9.51, -110.95, -7.39],
    ]
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        pd.DataFrame(eq, dtype=object).to_excel(xw, sheet_name="Equity", index=False, header=False)
        pd.DataFrame(mf, dtype=object).to_excel(xw, sheet_name="Mutual Funds", index=False, header=False)
        pd.DataFrame(combined, dtype=object).to_excel(xw, sheet_name="Combined", index=False, header=False)
    return buf.getvalue()

def make_deep_header_workbook_bytes() -> bytes:
    """
    Replica of the REAL Angel/SmartIQ export layout: each sheet starts with a
    client/statement/summary block, so the header row sits at row 14 (Equity)
    and row 12 (Mutual Funds). Regression test for header detection.
    """
    equity_block = [
        ["", "Client ID", "OOX650"],
        ["", "Equity Holdings Statement as on 2026-10-12"],
        ["", "Summary"],
        ["", "Invested Value", 520775.9917],
        ["", "Present Value", 488827.03],
        ["", "Unrealized P&L", -31948.9617],
        ["", "Unrealized P&L Pct.", -6.1349],
    ]
    eq_header = ["", "Symbol", "ISIN", "Sector", "Quantity Available",
                 "Quantity Discrepant", "Quantity Long Term",
                 "Quantity Pledged (Margin)", "Quantity Pledged (Loan)",
                 "Average Price", "Previous Closing Price", "Unrealized P&L",
                 "Unrealized P&L Pct."]
    eq = equity_block + [eq_header] + [
        ["", "HDFCBANK", "INE040A01034", "FINANCIAL SERVICES", 40, 0, 0, 0, 0, 798.635, 707.25, -3655.4, -11.4426],
        ["", "HCLTECH", "INE860A01027", "SOFTWARE SERVICES", 5, 0, 0, 0, 0, 1275.75, 1216.7, -295.25, -4.6286],
        ["", "AUTOIETF", "INF109KC10V2", "ETF", 140, 0, 0, 0, 0, 26.4429, 25.89, -77.4, -2.0908],
    ]
    mf_block = [
        ["", "Client ID", "OOX650"],
        ["", "Mutual Funds Holdings Statement as on 2026-10-12"],
        ["", "Summary"],
        ["", "Invested Value", 79196.0912],
        ["", "Present Value", 78100.5143],
        ["", "Unrealized P&L", -1095.5768],
        ["", "Unrealized P&L Pct.", -1.3834],
    ]
    mf_header = ["", "Symbol", "ISIN", "Instrument Type", "Quantity Available",
                 "Quantity Discrepant", "Quantity Pledged (Margin)",
                 "Quantity Pledged (Loan)", "Average Price",
                 "Previous Closing Price", "Unrealized P&L", "Unrealized P&L Pct."]
    mf = mf_block + [mf_header] + [
        ["", "HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN", "INF179K01WA6",
         "Hybrid - Dynamic Asset Allocation/Balanced Advantage", 19.166, 0, 0, 0, 553.0437, 546.088, -133.312, -1.2577],
        ["", "TATA DIGITAL INDIA FUND - DIRECT PLAN", "INF277K01Z77",
         "Equity - Sectoral/Thematic", 206.783, 0, 0, 0, 48.3574, 46.2504, -435.6877, -4.3571],
    ]
    combined = [
        ["", "Client ID", "OOX650"],
        ["", "Combined Holdings Statement as on 2026-10-12"],
        ["", "Summary"],
        ["", "Invested Value", 599972.0829],
        ["", "Present Value", 566927.5443],
        ["", "Unrealized P&L", -33044.5385],
        ["", "Unrealized P&L Pct.", -5.5077],
    ] + [
        ["", "Symbol", "ISIN", "Sector", "Instrument Type", "Quantity Available",
         "Quantity Discrepant", "Quantity Long Term", "Quantity Pledged (Margin)",
         "Quantity Pledged (Loan)", "Average Price", "Previous Closing Price",
         "Unrealized P&L", "Unrealize P&L Pct."],
        ["", "HDFCBANK", "INE040A01034", "FINANCIAL SERVICES", "-", 40, 0, 0, 0, 0, 798.635, 707.25, -3655.4, -11.4426],
        ["", "HCLTECH", "INE860A01027", "SOFTWARE SERVICES", "-", 5, 0, 0, 0, 0, 1275.75, 1216.7, -295.25, -4.6286],
        ["", "AUTOIETF", "INF109KC10V2", "ETF", "-", 140, 0, 0, 0, 0, 26.4429, 25.89, -77.4, -2.0908],
        ["", "HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN", "INF179K01WA6", "-",
         "Hybrid - Dynamic Asset Allocation/Balanced Advantage", 19.166, 0, "-", 0, 0, 553.0437, 546.088, -133.312, -1.2577],
        ["", "TATA DIGITAL INDIA FUND - DIRECT PLAN", "INF277K01Z77", "-",
         "Equity - Sectoral/Thematic", 206.783, 0, "-", 0, 0, 48.3574, 46.2504, -435.6877, -4.3571],
    ]
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        pd.DataFrame(eq, dtype=object).to_excel(xw, sheet_name="Equity", index=False, header=False)
        pd.DataFrame(mf, dtype=object).to_excel(xw, sheet_name="Mutual Funds", index=False, header=False)
        pd.DataFrame(combined, dtype=object).to_excel(xw, sheet_name="Combined", index=False, header=False)
    return buf.getvalue()

def run_self_tests() -> bool:
    """
    Run offline unit tests over the pipeline with sample data.
    These exercise: column mapping, derivation, classification, calculations,
    PDF + export generation. No live network is required (analysis engines fall
    back to N/A when yfinance data is absent).
    """
    global _COLLECTOR
    print("\n" + "=" * 70)
    print(" RUNNING SELF-TESTS (sample data, offline-safe)")
    print("=" * 70)
    ok = True
    try:
        # 1) to_float parsing
        assert to_float("\u20b91,234.50") == 1234.5
        assert to_float("(500)") == -500
        assert to_float("abc") is None
        assert to_float("") is None
        print(" [PASS] numeric parsing / missing-vs-zero")

        # 2) fmt_inr Indian grouping
        assert fmt_inr(1234567) == "\u20b912,34,567.00", fmt_inr(1234567)
        print(" [PASS] Indian currency formatting")

        # 3) extraction + mapping from a broker-style CSV byte stream
        df = make_sample_holdings_df()
        csv_bytes = df.to_csv(index=False, header=False).encode("utf-8")
        up = UploadedFile("sample.csv", csv_bytes, len(csv_bytes))
        tables = extract_tables(up)
        table = tables[0]
        clean, mapping = map_columns(table.dataframe, has_header=table.has_header)
        assert "symbol" in mapping and "isin" in mapping and "quantity" in mapping
        assert "cmp" in mapping and "sector" in mapping
        print(" [PASS] column auto-mapping:", list(mapping.keys()))

        holdings = build_holdings(clean, mapping, "csv")
        assert len(holdings) == 11, len(holdings)
        print(f" [PASS] built {len(holdings)} holdings")

        # 3b) file type is judged by CONTENT, not by the name
        html_bytes = ("<html><body><table><tr><td>SYMBOL</td><td>ISIN</td></tr>"
                      "<tr><td>TESTCO</td><td>INE000A01000</td></tr></table></body></html>"
                      ).encode("utf-8")
        assert detect_format(UploadedFile("x.html", html_bytes, len(html_bytes))) == "html"
        # Broker-style mislabel: HTML content saved under an .xlsx name.
        mis = UploadedFile("holdings-OOX650 (5).xlsx", html_bytes, len(html_bytes))
        assert detect_format(mis) == "html"
        t_html = extract_tables(mis)
        assert t_html and t_html[0].source_format == "html"
        assert len(t_html[0].dataframe) == 2
        assert detect_format(UploadedFile("junk.csv", bytes(range(200)), 200)) == "unknown"
        assert detect_format(UploadedFile("empty.xlsx", b"", 0)) == "unknown"
        assert detect_format(UploadedFile("a.csv", csv_bytes, len(csv_bytes))) == "csv"
        print(" [PASS] content-based file-type detection (mislabelled/HTML/unknown)")

        # 4) classification coverage - every row accounted for & broker-aware
        for h in holdings:
            classify_holding(h)
            assert h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI, CLASS_ETF,
                                     CLASS_MF, CLASS_UNKNOWN)
        classes = {h.asset_class for h in holdings}
        by = {h.asset_class for h in holdings}
        assert CLASS_ETF in by and CLASS_MF in by and CLASS_BANKS_FI in by and CLASS_STOCKS in by
        assert CLASS_UNKNOWN in by, "unknown row should be flagged, not dropped"
        assert len(holdings) == 11, "every uploaded row must be accounted for"
        print(" [PASS] asset classification:", by)

        # 5) analysis engines run without network and don't crash
        _COLLECTOR = DataCollector()
        for h in holdings:
            h.yf_ticker = _COLLECTOR.resolve_yf_ticker(h)
        orch = AnalysisOrchestrator(_COLLECTOR)
        for h in holdings:
            orch.analyze(h)
        print(" [PASS] analysis engines executed")

        # 6) portfolio math
        ps = compute_portfolio(holdings)
        # unrealised = MV - invested, and return consistent
        expect = ps.total_current_value - ps.total_invested
        assert abs(ps.unrealized_pnl - expect) < 1e-6
        print(f" [PASS] portfolio totals: invested={fmt_inr(ps.total_invested)}, "
              f"mv={fmt_inr(ps.total_current_value)}")

        # 7) allocation percentages sum ~100 (asset classes)
        tot = sum(ps.asset_allocation.values())
        assert 99.0 <= tot <= 101.0, tot
        print(f" [PASS] allocation sums to {tot:.1f}%")

        # 8) ONE comprehensive PDF + exports produced
        tmp = tempfile.mkdtemp(prefix="pa_test_")
        charts = generate_charts(holdings, ps, tmp)
        _ALL_CHARTS.clear(); _ALL_CHARTS.extend(charts)
        pdfs = generate_master_pdf(holdings, ps, charts, tmp)
        assert len(pdfs) == 1, f"expected exactly 1 PDF, got {len(pdfs)}"
        assert os.path.basename(pdfs[0]) == "Portfolio_Analysis_Report.pdf"
        assert os.path.exists(pdfs[0]) and os.path.getsize(pdfs[0]) > 5000
        assert not any(f.lower().endswith(".zip")
                       for f in os.listdir(os.path.dirname(pdfs[0])))
        files = export_tabular(holdings, ps, tmp)
        assert any(f.endswith("Portfolio_Summary.csv") for f in files)
        assert any(f.endswith("Analysis_Results.xlsx") for f in files)
        print(f" [PASS] produced 1 comprehensive PDF "
              f"({os.path.getsize(pdfs[0]):,} bytes) + {len(files)} tabular file(s), no ZIP")
        shutil.rmtree(tmp, ignore_errors=True)

        # 8b) rule engine on synthetic annual statements (offline, deterministic)
        #     Rupee-scale figures so market cap / FCF ratios are consistent.
        cols = ["FY26", "FY25", "FY24", "FY23", "FY22"]      # most recent first
        fin = pd.DataFrame({
            "Total Revenue": [1.30e11, 1.17e11, 1.05e11, 0.95e11, 0.86e11],
            "Operating Income": [2.60e10, 2.50e10, 2.35e10, 2.20e10, 2.10e10],
            "Reconciled Depreciation": [6.0e9, 5.5e9, 5.0e9, 4.8e9, 4.5e9],
            "Net Income": [1.66e10, 1.73e10, 1.55e10, 1.40e10, 1.30e10],
            "Basic EPS": [61.3, 63.9, 57.3, 51.7, 48.0],
            "Interest Expense": [-1.0e9, -9e8, -8e8, -7e8, -6e8],
        }, index=cols).T
        bsheet = pd.DataFrame({
            "Stockholders Equity": [8.0e10, 7.5e10, 7.0e10, 6.5e10, 6.0e10],
            "Total Debt": [6.0e9, 6.2e9, 6.4e9, 6.6e9, 6.8e9],
        }, index=cols).T
        cflow = pd.DataFrame({
            "Operating Cash Flow": [2.00e10, 2.22e10, 1.90e10, 1.75e10, 1.60e10],
            "Capital Expenditure": [-1.4e9, -1.5e9, -1.2e9, -1.1e9, -1.0e9],
        }, index=cols).T
        fund = compute_fundamentals({"fin": fin, "bs": bsheet, "cf": cflow})
        assert fund["years"] == 5
        assert abs(fund["roe"][0] - 1.66e10 / 8.0e10 * 100) < 1e-6
        assert 20 < fund["roe_avg"] < 26
        assert 28 < fund["roce_avg"] < 33
        assert abs(fund["fcf_last"] - 1.86e10) < 1e-3
        assert abs(fund["de_last"] - 6.0e9 / 8.0e10) < 1e-12
        assert abs(fund["interest_cov"][0] - 26.0) < 1e-6
        assert abs(fund["ebitda_margin"][0] - (2.60e10 + 6.0e9) / 1.30e11 * 100) < 1e-6
        assert 10 < fund["rev_cagr"][0] < 12 and 6 < fund["eps_cagr"][0] < 8
        info = {"marketCap": 3.29e11, "trailingPE": 18.2, "priceToBook": 4.4}
        rules = build_stock_rules(fund, info,
                                  {"last_price": 1217.0, "wk52_high": 1780.0},
                                  is_financial=False)
        assert len(rules["steps"]) == 4
        total = sum(len(rows) for _, rows in rules["steps"])
        assert total == 20, f"the demo card has 20 checks, got {total}"
        assert rules["pass"] + rules["fail"] + rules["watch"] + rules["na"] == total
        # Promoter holding / pledge / promoter selling / red flags stay N/A (no free source)
        assert rules["na"] == 4, rules["na"]
        assert rules["signal"] in Config.RULE_SIGNALS
        # Quality/valuation pass but growth fails (EBITDA & EPS CAGR, profit triangle)
        assert (rules["pass"], rules["fail"], rules["watch"]) == (11, 3, 2), \
            (rules["pass"], rules["fail"], rules["watch"])
        assert rules["signal"] == "HOLD", rules["signal"]
        # Banks/NBFCs: D/E and interest coverage are NOT applicable -> N/A
        bank_rules = build_stock_rules(fund, info, {"last_price": 1217.0},
                                       is_financial=True)
        assert bank_rules["steps"][0][1][3]["signal"] == SIG_NA   # debt to equity
        assert bank_rules["steps"][0][1][4]["signal"] == SIG_NA   # interest coverage
        assert bank_rules["na"] == rules["na"] + 2
        # Missing statements must NOT be guessed into a pass/fail
        thin = build_stock_rules({}, {"marketCap": 5e10}, {"last_price": 10.0}, False)
        assert thin["signal"] == "INSUFFICIENT DATA"
        print(f" [PASS] rule engine: {rules['pass']}/{rules['applicable']} applicable "
              f"checks pass, {rules['fail']} fail, {rules['na']} N/A -> {rules['signal']}")
        shutil.rmtree(tmp, ignore_errors=True)

        # 9) multi-sheet broker workbook + ISIN-based de-duplication
        xls_bytes = make_broker_workbook_bytes()
        wb_up = UploadedFile("broker.xlsx", xls_bytes, len(xls_bytes))
        wb_holdings = _process_holdings(wb_up)
        # Equity(3) + MF(1) are kept; Combined(4) duplicates are removed by ISIN.
        assert len(wb_holdings) == 4, f"expected 4 after dedup, got {len(wb_holdings)}"
        isins = [h.ISIN for h in wb_holdings]
        assert len(isins) == len(set(isins)), "duplicate ISINs should be de-duplicated"
        mf = [h for h in wb_holdings if h.asset_class == CLASS_MF]
        etfs = [h for h in wb_holdings if h.asset_class == CLASS_ETF]
        assert len(mf) == 1 and len(etfs) == 1
        print(" [PASS] multi-sheet read + ISIN de-duplication (4 unique holdings)")

        # 9b) REAL broker layout: header far down each sheet (row 14/12) + dedup
        deep_bytes = make_deep_header_workbook_bytes()
        deep_up = UploadedFile("holdings-OOX650 (5).xlsx", deep_bytes, len(deep_bytes))
        deep_holdings = _process_holdings(deep_up)
        # Equity(3) + MF(2) kept; Combined(5) repeats all of them by ISIN.
        assert len(deep_holdings) == 5, f"expected 5 after dedup, got {len(deep_holdings)}"
        syms = {h.symbol for h in deep_holdings}
        assert syms == {"HDFCBANK", "HCLTECH", "AUTOIETF",
                        "HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN",
                        "TATA DIGITAL INDIA FUND - DIRECT PLAN"}, syms
        hdfc = next(h for h in deep_holdings if h.symbol == "HDFCBANK")
        assert hdfc.quantity == 40 and abs(hdfc.avg_price - 798.635) < 1e-6
        assert hdfc.invested is not None and hdfc.current_value is not None
        classes = {h.asset_class for h in deep_holdings}
        assert CLASS_ETF in classes and CLASS_MF in classes
        print(" [PASS] deep-header broker sheet parsed (header found below summary)")

    except AssertionError as e:
        ok = False
        print(f" [FAIL] assertion: {e}")
        traceback.print_exc()
    except Exception as e:
        ok = False
        print(f" [FAIL] unexpected: {e}")
        traceback.print_exc()
    print("\n Self-test result:", "ALL PASSED" if ok else "FAILURES PRESENT")
    print(" NOTE: these tests are executed offline. Live yfinance/API integration")
    print("       and full acceptance criteria are NOT asserted as tested here.")
    print("=" * 70)
    return ok


# -----------------------------------------------------------------------------
# ENTRY POINT
# -----------------------------------------------------------------------------
def _in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return "ipykernel" in str(get_ipython()) or "ZMQShellCommunication" in str(get_ipython())
    except Exception:
        return False

if __name__ == "__main__":
    # When executed as a script/notebook cell, run the full interactive workflow.
    main()
elif _in_notebook():
    # In Colab/Notebook, `import`-style execution still wants to run immediately.
    main()


[setup] Installing 3 missing package(s): pypdf, pdfplumber, pytesseract
05:19:01 | INFO    | Portfolio Analyzer starting. Colab=True


INFO:PortfolioAnalyzer:Portfolio Analyzer starting. Colab=True


05:19:01 | INFO    | OCR available for scanned PDFs: False


INFO:PortfolioAnalyzer:OCR available for scanned PDFs: False



 STEP 1 / 8 : UPLOAD YOUR HOLDINGS FILE
Supported formats: .xlsx, .xls, .csv, .pdf
(The file type is checked by CONTENT, so a broker export whose name is
 mislabeled - e.g. an .xlsx that is really a table - is still accepted.)


...

INFO:PortfolioAnalyzer:Generated 1 comprehensive PDF with 79 holding section(s).



[8/8] Exporting CSV/Excel companions (optional downloads)...

 EXECUTION SUMMARY
 Holdings uploaded           : 79
 Successfully identified     : 79
 Analyzed                    : 79
 Failed data lookups         : 2
 Unidentified / ambiguous    : 0
 Total invested              : ₹5,99,971.91
 Current market value        : ₹5,66,927.54
 Unrealised P/L              : -₹33,044.37 (-5.51%)
 Comprehensive PDF report    : 1 file (Portfolio_Analysis_Report.pdf)
 Supporting files (CSV/XLSX) : 5
 Charts                      : 24
 Output directory            : /content/Portfolio_Output
 Signal distribution (from the rule engines):
   - HOLD: 40
   - SELL: 26
   - REVIEW: 5
   - WATCHLIST: 4
   - INSUFFICIENT DATA: 2
   - ACCUMULATE: 2
 Holdings scored on the 6-step rule card: 58 (the rest are funds/unidentified - they use their own checks)

 DOWNLOAD LINKS
 Single comprehensive PDF report: /content/Portfolio_Output/Portfolio_Analysis_Report.pdf (2,113,865 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 [Colab] PDF download triggered. Supporting CSV/Excel files are in: /content/Portfolio_Output


# New Section

In [ ]:
# =============================================================================
#  MASTER PROMPT IMPLEMENTATION
#  Automated Portfolio Analyzer  (Indian stock market focused)
#  Target environment : Google Colab (with graceful fallback for local Jupyter)
#
#  HOW TO USE
#  ----------
#  1. Copy this ENTIRE file into a single Google Colab code cell (or a .py file).
#  2. Run it. It will:
#       a) install required dependencies,
#       b) prompt you to manually upload your holdings file (.xlsx/.xls/.csv/.pdf),
#       c) extract + validate + classify holdings,
#       d) collect available market data,
#       e) run per-asset-class analysis engines,
#       f) compute portfolio-level metrics,
#       g) generate ONE comprehensive PDF report covering every holding in full
#          detail (plus optional CSV/Excel companions) and show the download link.
#
#  DESIGN PRINCIPLES (per the master prompt)
#  -----------------------------------------
#  * Financial accuracy and transparent calculations over bold recommendations.
#  * Never fabricate quantities, prices, ratios, estimates or targets. Metrics
#    that cannot be sourced from reliable data are marked "N/A".
#  * Robust execution: a single failed lookup must never crash the run.
#  * Every uploaded row is accounted for; unidentified rows are flagged, not
#    silently dropped.
#  * No hard-coded file paths, filenames, tickers or portfolios.
# =============================================================================

import os
import sys
import io
import re
import csv
import gc
import json
import time
import math
import shutil
import hashlib
import logging
import warnings
import tempfile
import traceback
from datetime import datetime, date
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1. AUTOMATIC DEPENDENCY INSTALLATION
# -----------------------------------------------------------------------------
# Only packages we actually need. We avoid force-upgrading core Colab packages
# (numpy/pandas already present) unless import fails.
def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IS_COLAB = _running_in_colab()

def install_dependencies() -> None:
    """Install / verify required Python packages, handling failures gracefully."""
    # (import_name, pip_name, required_for_colab_only)
    packages = [
        ("pandas", "pandas", False),
        ("numpy", "numpy", False),
        ("openpyxl", "openpyxl", False),
        ("xlrd", "xlrd", False),
        ("lxml", "lxml", False),          # pandas.read_html (HTML-as-xlsx exports)
        ("yfinance", "yfinance", False),
        ("requests", "requests", False),
        ("reportlab", "reportlab", False),
        ("matplotlib", "matplotlib", False),
        ("pypdf", "pypdf", False),
        ("pdfplumber", "pdfplumber", False),
        ("PIL", "pillow", False),
        ("tabulate", "tabulate", False),
        ("tqdm", "tqdm", False),
        ("IPython", "ipython", False),
    ]
    if IS_COLAB:
        packages.append(("pytesseract", "pytesseract", True))

    missing = []
    for import_name, pip_name, _ in packages:
        try:
            __import__(import_name)
        except Exception:
            missing.append((import_name, pip_name))

    if not missing:
        print("[setup] All Python dependencies already available.")
        return

    print(f"[setup] Installing {len(missing)} missing package(s): "
          f"{', '.join(p for _, p in missing)}")
    install_cmd = "pip install --quiet"
    if IS_COLAB:
        install_cmd += " --user"
    for import_name, pip_name in missing:
        try:
            os.system(f"{install_cmd} '{pip_name}'")
        except Exception as e:
            print(f"[setup][WARN] pip install failed for {pip_name}: {e}")
    # Refresh import cache after install.
    import importlib
    for import_name, _ in missing:
        try:
            importlib.import_module(import_name)
        except Exception:
            pass

def ensure_system_ocr() -> bool:
    """
    Attempt to make OCR available for scanned PDFs.
    Returns True if pytesseract + the tesseract binary are usable.
    Never crashes the run when OCR is unavailable.
    """
    try:
        import pytesseract  # noqa: F401
    except Exception:
        return False
    # Is the tesseract binary present? Install it on Colab if apt is available.
    from shutil import which
    if which("tesseract") is None and IS_COLAB:
        os.system("apt-get install -y -q tesseract-ocr > /dev/null 2>&1")
    if which("tesseract") is None:
        return False
    try:
        import pytesseract
        from PIL import Image
        pytesseract.get_tesseract_version()
        return True
    except Exception:
        return False

install_dependencies()

# -----------------------------------------------------------------------------
# Imports that must exist after installation
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np

# Third-party market-data libraries write noisy WARNING/ERROR lines to the
# console even though we already handle the failure (metric -> N/A). Silence
# them so the run log stays readable; our own LOG still reports everything.
for _noisy in ("yfinance", "yfinance.scrapers", "peewee", "urllib3",
               "requests", "PIL", "matplotlib"):
    _lg = logging.getLogger(_noisy)
    _lg.setLevel(logging.CRITICAL)
    _lg.addHandler(logging.NullHandler())
    _lg.propagate = False

# -----------------------------------------------------------------------------
# 2/3. CONFIGURATION AND LOGGING
# -----------------------------------------------------------------------------
class Config:
    OUTPUT_DIR = "Portfolio_Output"
    CACHE_DIR = ".pa_cache"
    CURRENCY = "INR"
    CURRENCY_SYMBOL = "\u20b9"          # Rupee sign
    GENERATED_ON = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    HTTP_TIMEOUT = 20
    MAX_RETRIES = 3
    # Assessment vocabulary. Signals are derived from the USER'S OWN rules on
    # their own data; they are checklists, not investment recommendations.
    RULE_SIGNALS = ["BUY MORE", "HOLD", "SELL", "INSUFFICIENT DATA"]
    STOCK_ASSESSMENTS = ["BUY CANDIDATE", "HOLD", "WATCHLIST", "REDUCE", "AVOID"]
    FUND_ASSESSMENTS = ["ACCUMULATE", "HOLD", "REVIEW"]
    NA = "N/A"

def setup_logging() -> logging.Logger:
    logger = logging.getLogger("PortfolioAnalyzer")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    sh = logging.StreamHandler(sys.stdout)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

LOG = setup_logging()

def fmt_inr(value: Any, decimals: int = 2) -> str:
    """Indian currency formatting with the ₹ symbol and lakh/crore-friendly commas."""
    if value is None or (isinstance(value, float) and (math.isnan(value))):
        return Config.NA
    try:
        value = float(value)
    except Exception:
        return str(value)
    sign = "-" if value < 0 else ""
    value = abs(value)
    whole = int(value)
    frac = round(value - whole, decimals)
    # Indian grouping: last 3 digits, then groups of 2.
    s = str(whole)
    if len(s) > 3:
        head, tail = s[:-3], s[-3:]
        parts = []
        while len(head) > 2:
            parts.insert(0, head[-2:])
            head = head[:-2]
        if head:
            parts.insert(0, head)
        grouped = ",".join(parts) + "," + tail
    else:
        grouped = s
    return f"{sign}{Config.CURRENCY_SYMBOL}{grouped}.{int(round(frac * (10 ** decimals))):0{decimals}d}"

def safe_div(numer: Any, denom: Any) -> Optional[float]:
    """Division that returns None instead of raising on invalid/zero denominators."""
    try:
        n = float(numer)
        d = float(denom)
        if d == 0 or math.isnan(n) or math.isnan(d):
            return None
        return n / d
    except Exception:
        return None

def to_float(value: Any) -> Optional[float]:
    """Parse a numeric value from strings that may contain ₹, commas, %, () etc."""
    if value is None:
        return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if isinstance(value, float) and math.isnan(value):
            return None
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"nan", "none", "-", "\u2013", "n/a", "na"}:
        return None
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    s = s.replace(Config.CURRENCY_SYMBOL, "").replace("₹", "")
    s = s.replace(",", "").replace("%", "").replace(" ", "")
    s = s.replace("cr", "").replace("Cr", "").replace("lakh", "").replace("Lakh", "")
    try:
        val = float(s)
        return -val if neg else val
    except Exception:
        return None

# -----------------------------------------------------------------------------
# 4. MANUAL HOLDINGS FILE UPLOAD (Colab native + local fallback)
# -----------------------------------------------------------------------------
@dataclass
class UploadedFile:
    name: str
    data: bytes
    size: int

def _looks_like_notebook(name: str) -> bool:
    return name.lower().endswith((".ipynb", ".py"))

# Only these are definitely NOT holdings data. Everything else is accepted and
# judged by CONTENT (magic bytes), because broker exports often mislabel files
# (an HTML table saved as .xlsx, a CSV renamed to .xls, etc.).
_BLOCKED_EXTS = (".ipynb", ".py", ".png", ".jpg", ".jpeg", ".gif")

def upload_holdings_file() -> List[UploadedFile]:
    """
    Request the holdings file from the user.
    Accepts anything that CONTENT-wise is Excel/CSV/PDF (extension is a hint,
    not a veto). Never interprets notebook args or images as data.
    Returns a list (multiple files are handled by the caller).
    """
    allowed_exts = (".xlsx", ".xls", ".csv", ".pdf")
    print("\n" + "=" * 70)
    print(" STEP 1 / 9 : UPLOAD YOUR HOLDINGS FILE")
    print("=" * 70)
    print("Supported formats: .xlsx, .xls, .csv, .pdf")
    print("(The file type is checked by CONTENT, so a broker export whose name is")
    print(" mislabeled - e.g. an .xlsx that is really a table - is still accepted.)")

    if IS_COLAB:
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        if not uploaded:
            raise ValueError("No file was uploaded in Colab. Aborting.")
        results = []
        for name, data in uploaded.items():
            low = name.lower()
            if low.endswith(".json") or _looks_like_notebook(name):
                LOG.warning("Ignoring notebook/code file '%s' (not a holdings file).", name)
                continue
            if low.endswith(_BLOCKED_EXTS):
                LOG.warning("Ignoring file '%s' - that type can not be holdings data.", name)
                continue
            fmt = detect_format(UploadedFile(name=name, data=data, size=len(data)))
            if fmt == "unknown":
                LOG.warning("File '%s' is not recognisable as Excel/CSV/PDF content "
                            "even though its name says so. Its first bytes were: %r",
                            name, bytes(data[:24]))
                continue
            if not low.endswith(allowed_exts):
                LOG.info("File '%s' has an unusual name but its content is a valid "
                         "%s - accepting it.", name, fmt.upper())
            results.append(UploadedFile(name=name, data=data, size=len(data)))
        if not results:
            raise ValueError("Uploaded files were not in a supported holdings format.")
        return results
    else:
        # Local Jupyter fallback: try the widgets file uploader, else a path prompt.
        try:
            import ipywidgets as widgets
            from IPython.display import display
            up = widgets.FileUpload(multiple=True)   # no accept filter: judge by content
            display(up)
            LOG.info("Select your holdings file using the widget above, then re-run "
                     "only if nothing appears. Falling back to a path prompt if the "
                     "widget yields nothing after selection.")
            # Give the user a moment, then read whatever is uploaded.
            import time as _t
            deadline = _t.time() + 60
            while _t.time() < deadline and len(up.value) == 0:
                _t.sleep(1)
            if len(up.value) > 0:
                results = []
                for item in up.value:
                    name = item["name"]
                    data = item["content"]
                    results.append(UploadedFile(name=name, data=data, size=len(data)))
                return results
        except Exception as e:
            LOG.info("File-upload widget unavailable (%s). Using path prompt.", e)
        # Plain prompt fallback.
        path = input("Enter the full path to your holdings file "
                     "(.xlsx/.xls/.csv/.pdf): ").strip().strip('"')
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")
        with open(path, "rb") as f:
            data = f.read()
        return [UploadedFile(name=os.path.basename(path), data=data,
                             size=len(data))]

def resolve_multiple_uploads(files: List[UploadedFile]) -> UploadedFile:
    """
    If several files were uploaded, ask which to analyze (or list them) rather
    than silently merging duplicate holdings.
    """
    if len(files) == 1:
        return files[0]
    print("\nMultiple supported files were uploaded:")
    for i, f in enumerate(files, start=1):
        print(f"  [{i}] {f.name}  ({f.size/1024:.1f} KB)")
    choice = None
    while choice is None:
        raw = input("Enter the number of the file to analyze "
                    "(they are NOT merged automatically): ").strip()
        try:
            idx = int(raw)
            if 1 <= idx <= len(files):
                choice = files[idx - 1]
            else:
                print("Invalid selection, try again.")
        except ValueError:
            print("Please enter a valid number.")
    return choice


# ---- Optional 'extras' upload: the data yfinance/Screener cannot give us ----
# Screener/Tickertape/NSE/BSE all block data-centre IPs for bulk reads, so the
# one-time extras file (one row per holding, metrics you verified there) is how
# bank NPA/CAR, MF factsheet numbers, sector P/E and red-flag counts enter the
# score. Column names are matched loosely; metric names may be ANY metric the
# report shows (NPA %, CAR, Expense Ratio, ...), not just the examples below.
EXTRAS_SYNONYMS: Dict[str, List[str]] = {
    "symbol": ["symbol", "ticker", "scrip", "nse code", "bse code", "code"],
    "name": ["name", "security", "company"],
    "promoter_pct": ["promoter holding", "promoter %"],  # NOT bare 'promoter':
    #                                                       it would eat 'Promoter Selling'
    "pledged_pct": ["pledged", "pledging", "pledge"],
    "promoter_selling": ["promoter selling", "promoter sold"],
    "sector_pe": ["sector p/e", "sector pe", "industry p/e"],
    "red_flags": ["red flag"],
    "nim_pct": ["nim"],
    "gnpa_pct": ["gross npa"],
    "nnpa_pct": ["net npa"],
    "pcr_pct": ["provision coverage", "pcc"],
    "car_pct": ["capital adequacy", "car"],
    "cet1_pct": ["cet1", "cet 1"],
    "roa_pct": ["roa"],
    "casa_pct": ["casa"],
    "cost_to_income_pct": ["cost to income", "cost:income"],
    "solvency_ratio": ["solvency"],
    "persistency_pct": ["persistency", "persistency ratio"],
    "expense_ratio_pct": ["expense ratio", "tod"],
    "tracking_error": ["tracking error"],
    "premium_nav_pct": ["premium/discount to nav", "premium to nav",
                        "discount to nav"],
    "underlying_index": ["underlying index", "benchmark index", "benchmark"],
    "amc": ["amc", "fund house"],
    "fund_manager": ["fund manager"],
    "aum": ["aum", "total assets", "fund size"],
    "ret_3y": ["3y return", "3 year return", "rolling 3y"],
    "ret_5y": ["5y return", "5 year return", "rolling 5y"],
    "sharpe": ["sharpe"],
    "sortino": ["sortino"],
    "std_dev": ["standard deviation", "std dev", "sd"],
    "max_drawdown_pct": ["max drawdown", "drawdown"],
    "turnover_ratio": ["turnover"],
}

# extras key -> the report metric it fills (only when that metric is N/A).
EXTRAS_METRIC_KEYS: Dict[str, str] = {
    "promoter_pct": "Promoter Holding %",
    "pledged_pct": "Pledged %",
    "nim_pct": "NIM %",
    "gnpa_pct": "Gross NPA %",
    "nnpa_pct": "Net NPA %",
    "pcr_pct": "Provision Coverage %",
    "car_pct": "Capital Adequacy (CAR) %",
    "cet1_pct": "CET1 %",
    "roa_pct": "ROA %",
    "casa_pct": "CASA Ratio %",
    "cost_to_income_pct": "Cost-to-Income %",
    "solvency_ratio": "Solvency Ratio",
    "persistency_pct": "Persistency %",
    "expense_ratio_pct": "Expense Ratio %",
    "tracking_error": "Tracking Error",
    "premium_nav_pct": "Premium/Discount to NAV %",
    "underlying_index": "Underlying Index",
    "amc": "AMC",
    "fund_manager": "Fund Manager",
    "aum": "AUM",
    "ret_3y": "3Y Return %",
    "ret_5y": "5Y Return %",
    "sharpe": "Sharpe Ratio",
    "sortino": "Sortino Ratio",
    "std_dev": "Standard Deviation",
    "max_drawdown_pct": "Max Drawdown %",
    "turnover_ratio": "Turnover Ratio",
}

# extras key -> the underscore key build_stock_rules() reads from `info`.
EXTRAS_RULE_KEYS: Dict[str, str] = {
    "promoter_pct": "_promoter_pct",
    "pledged_pct": "_pledged_pct",
    "promoter_selling": "_promoter_selling",
    "sector_pe": "_sector_pe",
    "red_flags": "_red_flags",
}

def _extras_key(s: Any) -> str:
    """Canonical form of an extras header/metric name: letters/digits only."""
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def _extras_match(col: str, synonym: str) -> bool:
    """Whole-word synonym match ('car' must not hit 'sector', 'roa' not 'roar')."""
    return re.search(r"(?<![a-z0-9])" + re.escape(_extras_key(synonym)) +
                     r"(?![a-z0-9])", _extras_key(col)) is not None

def _extras_label(canon: str) -> str:
    """Readable name for an extras key: prefer the report metric it fills."""
    mkey = EXTRAS_METRIC_KEYS.get(canon)
    if mkey:
        return mkey
    return {"promoter_selling": "Promoter selling",
            "sector_pe": "Sector P/E",
            "red_flags": "Red flags"}.get(canon, canon.replace("_", " ").title())

def upload_extras_file() -> Optional[UploadedFile]:
    """
    OPTIONAL second step: ask for one extras file (CSV/Excel) carrying the
    metrics no free API can supply. Empty name / 'skip' -> None, and the run
    simply continues with those rows as N/A.
    """
    print("\n" + "=" * 70)
    print(" STEP 2 / 9 : OPTIONAL - UPLOAD AN 'EXTRAS' FILE (CSV/EXCEL)")
    print("=" * 70)
    print("Screener.in scraping is best-effort (it blocks many data-centre IPs), and")
    print("Tickertape/NSE/BSE cannot be queried freely from here - so promoter/pledge,")
    print("bank NPA/CAR/NIM, sector P/E, red-flag counts and MF/ETF factsheet numbers")
    print("arrive as N/A. If you want them SCORED, export/paste those metrics (one row")
    print("per holding, matched by Symbol or Name) into a CSV/Excel and upload it now.")
    print("Example columns: Symbol, Promoter %, Pledged %, Sector P/E, Red Flags,")
    print("Gross NPA %, CAR %, Expense Ratio %, Sharpe, 3Y Return % ...")
    print("Any other column named like a report metric is matched too - nothing is")
    print("invented: a row is only scored when your file actually supplies a number.")
    print("(Type 'skip' or leave the name empty to continue without extras.)")
    path = None
    if IS_COLAB:
        try:
            from google.colab import files as colab_files
            uploaded = colab_files.upload()
            for name, data in uploaded.items():
                if _looks_like_notebook(name):
                    LOG.warning("Ignoring notebook/code file '%s' (not an extras file).", name)
                    continue
                return UploadedFile(name=name, data=data, size=len(data))
        except Exception as e:
            LOG.info("Colab upload unavailable (%s); falling back to a path prompt.", e)
        path = input("Path to the extras file (empty/skip = continue without): "
                     ).strip().strip('"')
    else:
        path = None
        try:
            import ipywidgets as widgets
            from IPython.display import display
            import time as _t
            w = widgets.Text(description="Extras file:",
                             placeholder="path to CSV/Excel, empty = skip")
            display(w)
            deadline = _t.time() + 60
            while _t.time() < deadline and not (w.value or "").strip():
                _t.sleep(1)
            path = (w.value or "").strip()
        except Exception:
            pass
        if path is None:
            try:
                path = input("Path to the extras file (empty/skip = continue without): "
                             ).strip().strip('"')
            except (EOFError, KeyboardInterrupt):
                path = ""
    if not path or path.lower() in ("skip", "none", "n"):
        LOG.info("No extras file supplied - promoter/pledge, bank & fund-specific "
                 "rows stay N/A (they are excluded from the score, never guessed).")
        return None
    if not os.path.exists(path):
        LOG.warning("Extras path '%s' not found - continuing without extras.", path)
        return None
    with open(path, "rb") as f:
        data = f.read()
    return UploadedFile(name=os.path.basename(path), data=data, size=len(data))

def parse_extras(file: UploadedFile) -> Dict[Tuple[str, str], Dict[str, Any]]:
    """
    Parse the extras upload into {(match_key, extra_key): {metric_key: value}}.
    match_key is the upper-cased symbol, or '~name' for name-only rows. Header
    row auto-detected; values kept as text if to_float cannot parse them (a
    text metric like 'Fund Manager' is fine; numbers are still scored).
    """
    tables = extract_tables(file)
    lookup: Dict[Tuple[str, str], Dict[str, Any]] = {}
    for table in tables:
        df = table.dataframe
        hdr = _detect_header_row(df)
        cols = [str(c).strip() for c in df.iloc[hdr].tolist()]
        body = df.iloc[hdr + 1:]
        # All access below is POSITIONAL - frames from extract_tables always come
        # back with integer column labels (header=None), never the header text.
        sym_pos = -1
        for canon in ("symbol", "ticker", "scrip code", "nse code", "bse code"):
            for p, c in enumerate(cols):
                if c and c != "nan" and any(_extras_match(c, s)
                                            for s in EXTRAS_SYNONYMS[canon]):
                    sym_pos = p
                    break
            if sym_pos >= 0:
                break
        if sym_pos < 0:  # no symbol column: fall back to the name column
            for p, c in enumerate(cols):
                if c and c != "nan" and any(_extras_match(c, s)
                                            for s in EXTRAS_SYNONYMS["name"]):
                    sym_pos = p
                    break
        metric_pos = []
        for p, c in enumerate(cols):
            if not c or c == "nan" or p == sym_pos:
                continue
            for canon, syns in EXTRAS_SYNONYMS.items():
                if canon in ("symbol", "name"):
                    continue
                if any(_extras_match(c, s) for s in syns):
                    metric_pos.append((p, canon))
                    break
        if sym_pos < 0 or not metric_pos:
            LOG.warning("Extras sheet had no recognizable Symbol/Name column plus "
                        "metric columns (columns seen: %s) - skipped.", cols)
            continue
        name_positions = [p for p, c in enumerate(cols)
                          if p != sym_pos and c and c != "nan" and
                          any(_extras_match(c, s) for s in EXTRAS_SYNONYMS["name"])]
        for _, row in body.iterrows():
            ident = str(row.iloc[sym_pos]).strip().upper()
            if not ident or ident.lower() in ("nan", "none", ""):
                continue
            vals: Dict[str, Any] = {}
            for p, canon in metric_pos:
                raw = row.iloc[p]
                num = to_float(raw)
                if num is None:
                    raw_s = str(raw).strip()
                    if not raw_s or raw_s.lower() in ("nan", "none"):
                        continue
                    vals[canon] = raw_s
                else:
                    vals[canon] = num
            if vals:
                lookup[(ident, "")] = vals
                # Also register under the name column when present, so a file
                # listing either identifier matches.
                for p in name_positions:
                    nm = str(row.iloc[p]).strip().upper()
                    if nm and nm.lower() not in ("nan", "none"):
                        lookup.setdefault(("~" + re.sub(r"\s+", "", nm), ""), vals)
    LOG.info("Extras file: %d symbol/name entry point(s) loaded.", len(lookup))
    return lookup

def apply_extras_to_holdings(holdings: List[Holding],
                             lookup: Dict[Tuple[str, str], Dict[str, Any]]) -> int:
    """Attach matching extras values to each holding. Returns match count."""
    if not lookup:
        return 0
    n = 0
    for h in holdings:
        keys = set()
        sym = re.sub(r"\s+", "", (h.symbol or "").upper())
        if sym:
            keys.add(sym)
            keys.add(sym.split("-")[0])      # drop series suffix (SILVERLINE-X)
        nm = re.sub(r"\s+", "", (h.name or "").upper())
        if nm:
            keys.add("~" + nm)
        for k in keys:
            hit = lookup.get((k, ""))
            if hit:
                h.extras = dict(hit)
                n += 1
                break
    LOG.info("Extras matched to %d of %d holding(s).", n, len(holdings))
    if n == 0:
        LOG.warning("No extras row matched any holding - check that the extras "
                    "file has a Symbol (or Name) column with exact broker codes.")
    return n


# -----------------------------------------------------------------------------
# 5. FILE VALIDATION & DATA EXTRACTION  (xlsx / xls / csv / pdf)
# -----------------------------------------------------------------------------
SUPPORTED_EXTENSIONS = (".xlsx", ".xls", ".csv", ".pdf")

@dataclass
class ExtractedTable:
    """Raw tabular data pulled from the uploaded file, before column mapping."""
    dataframe: pd.DataFrame
    source_format: str
    notes: List[str] = field(default_factory=list)
    has_header: bool = False

def detect_format(file: UploadedFile) -> str:
    """
    Detect file format from CONTENT magic bytes only - the extension is never
    trusted, because broker exports are often mislabeled (HTML tables or CSV
    text saved with an .xlsx name). Returns "unknown" for anything that is not
    readable data (empty file, binary garbage), which callers report clearly.
    """
    data = file.data
    if data is None or (isinstance(data, (bytes, bytearray)) and len(data.strip()) == 0) \
            or (isinstance(data, str) and not data.strip()):
        return "unknown"
    if data[:4] == b"%PDF":
        return "pdf"
    if data[:2] == b"PK":                       # xlsx is a zip container
        return "xlsx"
    if data[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":  # legacy OLE2 (.xls)
        return "xls"
    head = data[:512].lstrip().lower() if isinstance(data, bytes) \
        else str(data)[:512].lstrip().lower()
    if head.startswith((b"<!doctype", b"<html", b"<table", b"<?xml")):
        # HTML table masquerading as a spreadsheet (very common broker export).
        return "html"
    # Delimited text only if it is mostly printable.
    sample = data[:2048] if isinstance(data, bytes) else str(data)[:2048].encode()
    printable = sum(1 for b in sample if 32 <= b < 127 or b in (9, 10, 13))
    if sample and printable / len(sample) > 0.85:
        return "csv"
    return "unknown"

def extract_tables(file: UploadedFile) -> List[ExtractedTable]:
    """
    Return a list of ExtractedTable. Excel/CSV workbooks with several sheets
    (e.g. broker statements with Equity / Mutual Funds / Combined tabs) are
    fully read; duplicate rows across sheets are removed later by ISIN/symbol.
    """
    return _extract_table(file)

def _extract_table(file: UploadedFile) -> List[ExtractedTable]:
    fmt = detect_format(file)
    LOG.info("Detected file format: %s (%s bytes)", fmt, file.size)
    if fmt == "unknown":
        raise ValueError(
            f"The file '{file.name}' could not be recognised as Excel/CSV/PDF "
            "content. Please export the holdings again from the broker as "
            ".xlsx or .csv and upload that file.")
    if fmt == "pdf":
        return [_extract_pdf(file)]
    buf = io.BytesIO(file.data)
    if fmt == "html":
        return _extract_html_tables(file, fmt)
    if fmt == "csv":
        try:
            df = _read_csv_robust(buf)
        except Exception as e:
            raise ValueError(f"Could not read the CSV file: {e}")
        if df is None or df.empty:
            raise ValueError("The uploaded file appears to be empty.")
        return [ExtractedTable(dataframe=df, source_format=fmt)]
    # Excel: read EVERY sheet (broker statements often split Equity / MF / Combined).
    try:
        engine = "openpyxl" if fmt == "xlsx" else "xlrd"
        sheets = pd.read_excel(buf, engine=engine, sheet_name=None, header=None)
    except Exception as e:
        # Mislabelled exports: a file sniffing as zip/OLE2 but not a real
        # workbook. Try HTML tables, then delimited text, before giving up.
        tables = _extract_html_tables(file, fmt, quiet_ok=True)
        if tables:
            LOG.warning("'%s' was named Excel but contained an HTML table - "
                        "parsed it as such.", file.name)
            return tables
        try:
            df = _read_csv_robust(io.BytesIO(file.data))
            if df is not None and not df.empty:
                LOG.warning("'%s' was named Excel but contained delimited text - "
                            "parsed it as CSV.", file.name)
                return [ExtractedTable(dataframe=df, source_format="csv")]
        except Exception:
            pass
        raise ValueError(f"Could not read the {fmt} file: {e}")
    tables: List[ExtractedTable] = []
    for sheet_name, df in sheets.items():
        if df is None or df.empty:
            continue
        tables.append(ExtractedTable(dataframe=df.reset_index(drop=True),
                                     source_format=fmt,
                                     notes=[f"sheet: {sheet_name}"]))
    if not tables:
        raise ValueError("The uploaded workbook appears to be empty.")
    LOG.info("Read %d worksheet(s): %s", len(tables),
             [t.notes[0] for t in tables])
    return tables

def _extract_html_tables(file: UploadedFile, sniffed_as: str,
                         quiet_ok: bool = False) -> List[ExtractedTable]:
    """
    Parse <table> markup saved under a spreadsheet name (classic broker export).
    Uses pandas.read_html (lxml/bs4); returns [] when no tables are found and
    quiet_ok is set, so the caller can try other fallbacks.
    """
    try:
        text = file.data.decode("utf-8-sig", errors="replace") \
            if isinstance(file.data, (bytes, bytearray)) else str(file.data)
        frames = pd.read_html(io.StringIO(text), header=None)
    except Exception as e:
        if quiet_ok:
            return []
        raise ValueError(
            f"The file '{file.name}' contains HTML (a web-page table saved as "
            f"{sniffed_as.upper()}), but it could not be parsed: {e}")
    tables: List[ExtractedTable] = []
    for i, df in enumerate(frames):
        if df is None or df.empty or df.dropna(how="all").empty:
            continue
        tables.append(ExtractedTable(dataframe=df.reset_index(drop=True),
                                     source_format="html",
                                     notes=[f"html table {i + 1}"]))
    if not tables:
        if quiet_ok:
            return []
        raise ValueError(f"No readable table found inside the HTML file '{file.name}'.")
    LOG.info("Parsed %d HTML table(s) from '%s'.", len(tables), file.name)
    return tables

def _read_csv_robust(buf: io.BytesIO) -> pd.DataFrame:
    """Try common delimiters + encodings for CSV holdings exports."""
    raw = buf.getvalue()
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            text = raw.decode(enc)
            break
        except Exception:
            continue
    else:
        text = raw.decode("utf-8", errors="ignore")
    sample = text[:4096]
    delim = ","
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        delim = dialect.delimiter
    except Exception:
        counts = {d: sample.count(d) for d in [",", ";", "\t", "|"]}
        if counts:
            delim = max(counts, key=counts.get)
    from io import StringIO
    return pd.read_csv(StringIO(text), sep=delim, header=None,
                       engine="python", skip_blank_lines=True)

def _extract_pdf(file: UploadedFile) -> ExtractedTable:
    """Extract tables/text from PDF; fall back to OCR for scanned pages."""
    notes: List[str] = []
    pdfplumber = None
    try:
        import pdfplumber
    except Exception as e:
        raise ValueError(f"PDF support unavailable ({e}).")
    buf = io.BytesIO(file.data)
    best_table = None
    text_lines: List[str] = []
    page_count = 0
    try:
        with pdfplumber.open(buf) as pdf:
            page_count = len(pdf.pages)
            for page in pdf.pages:
                tables = page.extract_tables() or []
                for t in tables:
                    df = pd.DataFrame(t[1:], columns=t[0]) if t and t[0] else None
                    if df is not None and len(df) > 0:
                        if best_table is None or len(df) > len(best_table):
                            best_table = df
                txt = page.extract_text() or ""
                if txt.strip():
                    text_lines.append(txt)
    except Exception as e:
        notes.append(f"pdfplumber table/text extraction error: {e}")

    if best_table is not None and len(best_table) >= 1:
        notes.append(f"Extracted a table with {len(best_table)} rows from PDF.")
        return ExtractedTable(dataframe=best_table, source_format="pdf", notes=notes,
                              has_header=True)

    # No embedded table -> try text parse, then OCR if needed.
    joined_text = "\n".join(text_lines)
    if len(joined_text.strip()) < 30 and page_count > 0:
        notes.append("PDF looks scanned (little embedded text). Attempting OCR...")
        ocr_text = _ocr_pdf(file)
        if ocr_text:
            joined_text = ocr_text
            notes.append("OCR text obtained; parsed with best-effort column inference.")
        else:
            notes.append("OCR unavailable/failed. Cannot extract reliable holdings "
                         "from this scanned PDF. Provide a text-based PDF or CSV/Excel.")
            raise ValueError("Scanned PDF without usable OCR. No reliable data extracted.")

    df = _parse_holding_text(joined_text)
    if df is None or df.empty:
        raise ValueError("Could not identify any holdings table in the PDF text.")
    return ExtractedTable(dataframe=df, source_format="pdf", notes=notes)

def _ocr_pdf(file: UploadedFile) -> Optional[str]:
    try:
        import pytesseract
        from PIL import Image
    except Exception:
        return None
    if shutil.which("tesseract") is None:
        return None
    try:
        import pdfplumber
        buf = io.BytesIO(file.data)
        chunks = []
        with pdfplumber.open(buf) as pdf:
            for page in pdf.pages[:6]:                # cap OCR pages for speed
                img = page.to_image(resolution=200).original
                chunks.append(pytesseract.image_to_string(img))
        return "\n".join(chunks)
    except Exception as e:
        LOG.warning("OCR failed: %s", e)
        return None

def _parse_holding_text(text: str) -> Optional[pd.DataFrame]:
    """Best-effort: turn whitespace-aligned text into a header+rows DataFrame."""
    lines = [ln for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2:
        return None
    rows = [re.split(r"\s{2,}|\t", ln.strip()) for ln in lines]
    width = max(len(r) for r in rows)
    if width < 3:
        return None
    rows = [r + [""] * (width - len(r)) for r in rows]
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 6. COLUMN MAPPING, NORMALIZATION & VALIDATION
# -----------------------------------------------------------------------------
# Map canonical field -> list of name fragments seen across Indian brokers.
COLUMN_SYNONYMS: Dict[str, List[str]] = {
    "name": ["name", "security", "scrip name", "stock name", "company",
             "description", "fund name", "scheme name"],
    "symbol": ["symbol", "ticker", "scrip code", "nse code", "bse code",
               "scheme code", "scrip"],
    "isin": ["isin"],
    "quantity": ["quantity", "qty", "units", "no of shares", "shares",
                 "vol", "volume", "holding", "nos"],
    "avg_price": ["avg", "average", "buy price", "purchase price", "cost",
                  "avg price", "average price", "avg cost", "bought"],
    "invested": ["invested", "buy value", "purchase value", "cost value",
                 "amount invested", "total buy", "investment", "bought value",
                 "buy amount"],
    "cmp": ["cmp", "current market price", "last price", "ltp", "market price",
            "current price", "previous closing price", "closing price",
            "price", "nav", "current nav"],
    "current_value": ["current value", "market value", "mv", "valuation",
                      "present value", "current market value", "latest value",
                      "total value", "scheme value"],
    "pnl": ["p/l", "pl", "p l", "profit", "loss", "unrealised",
            "unrealized", "gain", "mtm", "overall gain", "profit/loss"],
    "pnl_pct": ["p/l %", "p l pct", "return", "gain %", "profit %",
                "unrealised %", "unrealized %", "pct", "%"],
    "sector": ["sector"],
    "asset_type": ["instrument type", "asset type", "security type",
                   "type", "category", "kind"],
    "exchange": ["exchange", "exch", "bse/nse"],
    "currency": ["currency", "ccy"],
}

@dataclass
class Holding:
    """Normalized single holding row with full provenance."""
    source_row: int
    raw_name: str
    name: str
    symbol: str
    ISIN: str
    quantity: Optional[float]
    avg_price: Optional[float]
    invested: Optional[float]
    cmp: Optional[float]
    current_value: Optional[float]
    pnl: Optional[float]
    pnl_pct: Optional[float]
    asset_type_raw: str
    exchange: str
    currency: str
    # Filled during processing:
    asset_class: str = "Unidentified"
    sector: str = Config.NA
    broker_sector: str = ""        # sector as supplied by the broker, if any
    sub_category: str = Config.NA
    classification_source: str = Config.NA
    confidence: str = "Low"
    issues: List[str] = field(default_factory=list)
    yf_ticker: str = ""            # resolved yfinance symbol
    metrics: Dict[str, Any] = field(default_factory=dict)
    assessment: str = Config.NA
    assessment_reasons: List[str] = field(default_factory=list)
    analyzed: bool = False
    analysis: Dict[str, Any] = field(default_factory=dict)  # rule tables for the PDF
    extras: Dict[str, Any] = field(default_factory=dict)    # user-verified metrics from the optional extras file

def _norm_header(s: Any) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9/%.-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s

def _detect_header_row(df: pd.DataFrame) -> int:
    """
    Find the row that looks most like a header. Broker statements put the real
    header far down the sheet (Angel/SmartIQ equity sheets have ~7 metadata +
    6 summary lines before it), so we scan well beyond the top rows and score
    mainly by KNOWN column names - summary labels like "Invested Value" match
    nothing, while "Symbol / ISIN / Quantity Available..." matches ~14.
    """
    best_idx, best_score = 0, -1.0
    for i in range(min(200, len(df))):
        row = df.iloc[i].astype(str).tolist()
        cells = [c for c in row if c and c.lower() not in ("nan", "none", "")]
        if len(cells) < 3:
            continue
        known = 0
        non_numeric = 0
        for c in cells:
            is_num = to_float(c) is not None
            if not is_num:
                non_numeric += 1
            # A header cell must MATCH a synonym as a whole word, so data rows
            # containing words like "finance" or "capital" are not mistaken
            # for headers (\b fails after '%' so '/(\b|$)' is used too).
            nc = _norm_header(c)
            if any(re.search(r"(?:\b|/|%)" + re.escape(frag) + r"(?:\b|/|$)", nc)
                   for frags in COLUMN_SYNONYMS.values() for frag in frags):
                known += 1
        if non_numeric < len(cells) * 0.7:
            continue  # majority must be text, not numbers
        score = known * 3.0 + non_numeric
        if score > best_score:
            best_score, best_idx = score, i
            if known >= 3 and non_numeric / len(cells) >= 0.9:
                break  # confidently a header; stop early on big sheets
    return best_idx

def map_columns(df: pd.DataFrame, has_header: bool = False) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """
    Auto-detect a header row and map synonyms to canonical columns.
    If has_header is True the DataFrame's existing columns are treated as the
    header (used for tables already parsed with a header row, e.g. from PDF).
    Returns (data-with-canonical-cols, mapping dict).
    """
    if has_header:
        header = df.columns.astype(str).tolist()
        data = df.reset_index(drop=True)
        data.columns = [_norm_header(h) for h in header]
    else:
        hdr_idx = _detect_header_row(df)
        header = df.iloc[hdr_idx].astype(str).tolist()
        data = df.iloc[hdr_idx + 1:].reset_index(drop=True)
        data.columns = [ _norm_header(h) for h in header ]

    mapping: Dict[str, int] = {}
    assigned_src: set = set()
    for canon, synonyms in COLUMN_SYNONYMS.items():
        chosen = None
        for src_i, col in enumerate(data.columns):
            if src_i in assigned_src:
                continue
            for syn in synonyms:
                if syn in col:
                    chosen = src_i
                    break
            if chosen is not None:
                break
        if chosen is not None:
            mapping[canon] = chosen
            assigned_src.add(chosen)

    # Build a clean frame keyed by canonical names.
    clean = pd.DataFrame()
    for canon, src_i in mapping.items():
        clean[canon] = data.iloc[:, src_i].values
    for i, col in enumerate(data.columns):
        if i not in assigned_src:
            clean[f"_extra_{col}"] = data.iloc[:, i].values
    return clean, mapping

def build_holdings(clean: pd.DataFrame, mapping: Dict[str, int],
                   source_format: str) -> List[Holding]:
    """Convert the mapped, clean frame into validated Holding objects."""
    holdings: List[Holding] = []
    def g(row: pd.Series, key: str) -> Any:
        return row[key] if key in row.index else None

    for idx, row in clean.iterrows():
        name = _clean_str(g(row, "name"))
        symbol = _clean_str(g(row, "symbol"))
        provided_isin = _clean_str(g(row, "isin")).upper()
        provided_sector = _clean_str(g(row, "sector"))
        # Broker statements often have no separate "name" column: fall back to
        # the symbol (which for MF rows is the full fund name) then the ISIN.
        if not name:
            name = symbol or provided_isin
        if not name and not symbol and not provided_isin:
            continue  # truly empty row

        h = Holding(
            source_row=int(idx) + 1,
            raw_name=name or symbol or provided_isin,
            name=name,
            symbol=symbol,
            ISIN=provided_isin,
            quantity=to_float(g(row, "quantity")),
            avg_price=to_float(g(row, "avg_price")),
            invested=to_float(g(row, "invested")),
            cmp=to_float(g(row, "cmp")),
            current_value=to_float(g(row, "current_value")),
            pnl=to_float(g(row, "pnl")),
            pnl_pct=to_float(g(row, "pnl_pct")),
            asset_type_raw=_clean_str(g(row, "asset_type")),
            exchange=_clean_str(g(row, "exchange")).upper(),
            currency=(_clean_str(g(row, "currency")) or "INR").upper(),
        )
        if provided_sector:
            h.broker_sector = provided_sector
        # If ISIN was only available in the symbol column, capture it.
        if not h.ISIN and re.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}[0-9]", symbol or ""):
            h.ISIN = symbol
        _derive_and_validate(h)
        holdings.append(h)
    return holdings

def _clean_str(v: Any) -> str:
    if v is None:
        return ""
    s = str(v).strip()
    if s.lower() in ("nan", "none", ""):
        return ""
    # Broker "Combined" sheets use a lone dash as a not-applicable placeholder.
    if s in ("-", "\u2013", "\u2014", "--", "n/a", "na", "nil", "none"):
        return ""
    return s

def _derive_and_validate(h: Holding) -> None:
    """
    Fill derived values (invested / current_value / pnl) and validate consistency.
    Missing values are NOT invented; only derived where at least the required
    inputs exist. Distinguishes missing vs genuine zero via issues list.
    """
    # invested = quantity * avg_price  (when not given directly)
    if h.invested is None and h.quantity is not None and h.avg_price is not None:
        h.invested = h.quantity * h.avg_price
        h.classification_source = h.classification_source  # unchanged
    # current_value = quantity * cmp   (when not given directly)
    if h.current_value is None and h.quantity is not None and h.cmp is not None:
        h.current_value = h.quantity * h.cmp
    # pnl = current_value - invested
    if h.pnl is None and h.current_value is not None and h.invested is not None:
        h.pnl = h.current_value - h.invested
    # pnl_pct = pnl / invested * 100
    if h.pnl_pct is None:
        r = safe_div(h.pnl, h.invested)
        h.pnl_pct = (r * 100) if r is not None and h.invested and h.invested > 0 else None

    # Validate quantity/price presence
    if h.quantity is None and h.invested is None:
        h.issues.append("Missing both quantity and invested amount.")
    if h.invested is not None and h.invested < 0:
        h.issues.append("Negative invested amount (short position or data error).")
    if h.current_value is not None and h.invested is not None and h.current_value > 0:
        # Consistency: recomputed pnl vs provided pnl
        if h.pnl is not None:
            recomputed = h.current_value - h.invested
            if abs(recomputed - h.pnl) > max(1.0, abs(recomputed) * 0.02):
                h.issues.append(
                    f"P/L mismatch: file says {fmt_inr(h.pnl)}, "
                    f"recomputed {fmt_inr(recomputed)}.")


# -----------------------------------------------------------------------------
# 7. AUTOMATIC ASSET CLASSIFICATION
# -----------------------------------------------------------------------------
# Primary asset classes (A-E in the master prompt).
CLASS_BANKS_FI = "Banks & Financial Institutions"
CLASS_STOCKS   = "Individual Stocks"
CLASS_ETF      = "ETFs"
CLASS_MF       = "Mutual Funds"
CLASS_UNKNOWN  = "Unidentified Assets"

# Curated keyword/identifier tables. These are transparent, editable heuristics;
# every classification records its source + confidence so users can verify.
ETF_NAME_HINTS = ["etf", "exchange traded fund", "niftybees", "bankbees",
                  "goldbees", "gold etf", "silver etf", "nifty 50 etf",
                  "juniorbees", "itbees", "midcap beees", "shadowlender"]
MF_NAME_HINTS = ["mutual fund", "direct plan", "regular plan", "growth plan",
                 "idcw", "dividend plan", "elss", "flexi cap", "flexicap",
                 "multi cap", "small cap", "mid cap", "large cap", "index fund",
                 "liquid fund", "gilt fund", "fof", "fund of funds", "hybrid",
                 "savings plan", "systematic"]
BANK_FI_HINTS = ["bank", "bajaj finance", "hudco", "nbcc", "housing develop",
                 "nbf", "finance", "financial", "l&t finance", "sundaram",
                 "chothala", "shriram", "muthoot", "manappuram", "credit",
                 "capital", "muthoot finance", "iasl", "repos", "millicap",
                 "iifl", "angel one", "bse", "mcx", "csl", "cent", "small finance"]
INSURANCE_HINTS = ["insurance", "lic", "hdfc life", "icici luna", "birla sun life",
                   "sbi life", "max life", "star health", "care health",
                   "national insurance", "new india insurance", " oriental",
                   "unix", "reinsurance", "general insurance"]

# Sector detection keyword table for individual stocks (name based).
SECTOR_KEYWORDS: Dict[str, List[str]] = {
    "Information Technology": ["infosys", "tcs", "tatko", "wipro", "hcl",
        "tech mahindra", "lTIMINDTREE", "mindtree", "persistent", "cognizant",
        "mpc", " Zensar", "lnt", "page industries", "kfintech", "oracle",
        "simplify", "cummins", "it", "software", "systems", "technologies", "digital"],
    "Banking & NBFC": ["bank", "finance", "housing"],
    "Pharmaceuticals & Healthcare": ["sun pharma", "dr reddy", "cipl"
        "lupin", "aurobindo", "aurolibharam", "divis", "biocon", "glyphin",
        "pharma", "laboratories", "healthcare", "hospital", "diagnostics", "medic"],
    "Automobiles": ["maruti", "mahindra", "m&m", "tata motors", " Bajaj Auto",
        "hero", "tvS", "eicher", "ashok", "auto", "motors", "automotive", "tyres"],
    "FMCG": ["hdfc cbi", "hindustile", "hul", "itc", "nestle", "britania",
        "dabur", "godrej consumer", "colgate", "marico", "tata consumer",
        "emami", "fmcg", "foods", "beverage"],
    "Metals & Mining": ["tata steel", "jsw", "jsw steel", "vedanta", "hindalco",
        "national alum", "sAIL", "rashtriy", "jindal", "zinc", "copper",
        "metal", "steel", "mineral", "mining"],
    "Energy & Oil": ["reliance", "ongc", "gail", "indian oil", "bpcl", "hpcl",
        "coal india", "nTPC", "power", "energy", "petroleum", "renewable", "solar"],
    "Chemicals": ["srf", "pidilite", "deepak nitrite", "tata chemical", "uflex",
        "chemical", "fertilizer", "chromite", "aromatic"],
    "Infrastructure & Construction": ["l&t", "larisen", "ncc", "shapoorji",
        "irb", "dhandos", "roads", "infra", "construction", "cement", "ultratech",
        "grasim", "ambuja", "dalmia"],
    "Telecom": ["reliance jio", "bharti airtel", "vodafone idea", "vi", "telecom"],
    "Real Estate": ["dlf", "godrej prop", "sobha", "prestige", "brigade",
        "embassy", "realty", "properties", "realestate"],
    "Consumer Discretionary": ["titan", "nykaa", "avenue", "trendra", "page",
        "fashion", "retail", "jewel", "luxury"],
    "Defence & Aerospace": ["hindustan aeronautics", "hal", "beML", "cochin ship",
        "mazagon", "bharat dynamics", "defence", "aerospace"],
    "IT Services / Consulting": ["accureyt", "caltas", "tata consultancy"],
    "Agriculture & Rural": ["escorts", "icra", "rural", "agri", "seed"],
    "Transportation & Logistics": ["concor", "ship", "logistics", "container",
        "transport", "airline", "indigo", "interglobe"],
    "Media & Entertainment": ["saregama", "pvr", "inoxia", "media", "entertain",
        "cinema", "network"],
    "Electronics & EMS": ["kaynes", " Dixon", " Amber", "Syrma", "electronic",
        "semiconductor", "delta autowave"],
    "Services": ["jbm", "tata Elxsi", "unitech", "mspl", "moschip"],
    "Diversified": ["tata", "godrej", "mahindra"],
}

# A small, transparent symbol->sector map for common Indian large caps so we do
# not rely purely on fuzzy name matching. Extend freely.
KNOWN_SYMBOL_SECTOR: Dict[str, str] = {
    "RELIANCE": "Energy & Oil", "TCS": "Information Technology",
    "HDFCBANK": "Banking & NBFC", "ICICIBANK": "Banking & NBFC",
    "INFY": "Information Technology", "SBIN": "Banking & NBFC",
    "BHARTIARTL": "Telecom", "ITC": "FMCG", "LT": "Infrastructure & Construction",
    "KOTAKBANK": "Banking & NBFC", "AXISBANK": "Banking & NBFC",
    "BAJFINANCE": "Banking & NBFC", "ASIANPAINT": "Consumer Discretionary",
    "MARUTI": "Automobiles", "HCLTECH": "Information Technology",
    "SUNPHARMA": "Pharmaceuticals & Healthcare", "TITAN": "Consumer Discretionary",
    "UMIHDFC": "Banking & NBFC", "WIPRO": "Information Technology",
    "NESTLEIND": "FMCG", "TATAMOTORS": "Automobiles",
}

def _contains(hay: str, needle: str) -> bool:
    return needle.strip().lower() in (hay or "").lower()

def _prettify_sector(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().title()

def classify_holding(h: Holding) -> None:
    """
    Assign primary asset class + sector/sub-category with source + confidence.

    For Indian markets these signals are authoritative and checked first:
      * ISIN prefix:  INF -> AMFI scheme (ETF or mutual fund),
                      INE -> exchange-listed security (equity / bond / InvIT).
      * broker 'Sector' column: 'ETF' -> exchange-traded fund; financial-
                      services sectors -> Banks & NBFC; otherwise it is used
                      directly as the sector label (more reliable than fuzzy
                      name matching).
    General files without these columns still fall back to keyword heuristics.
    """
    hay = f"{h.name} {h.symbol} {h.asset_type_raw}".lower()
    type_hint = (h.asset_type_raw or "").lower()
    bsector = (h.broker_sector or "").strip().lower()
    isin = (h.ISIN or "").upper()
    is_fund_isin = isin.startswith("INF")
    is_security_isin = isin.startswith("INE")

    # --- ETF detection (broker tag is authoritative for this broker format) ---
    looks_etf = (
        bsector == "etf" or type_hint == "etf"
        or "bees" in (h.symbol or "").lower()
        or (h.symbol or "").lower().endswith("ietf")
        or re.search(r"(^|[^a-z])etf([^a-z]|$)", hay) is not None
        or any(_contains(hay, k) for k in
               ["goldbees", "niftybees", "juniorbees", "itbees", "bankbees"])
    )

    if looks_etf:
        _set_class(h, CLASS_ETF, _etf_flavour(hay),
                   "broker sector = ETF" if bsector == "etf" else "ETF name/symbol token",
                   "High" if (bsector == "etf" or is_fund_isin) else "Medium")
        h.sector = "ETF"
    elif is_fund_isin:
        # INF scheme that is not an ETF -> mutual fund.
        _set_class(h, CLASS_MF, _mf_subcategory(hay + " " + type_hint),
                   "ISIN prefix INF (AMFI scheme)", "High")
    elif _contains(hay, "mutual fund") or any(
            k in type_hint for k in ["mutual", "hybrid", "elss", "idcw"]):
        _set_class(h, CLASS_MF, _mf_subcategory(hay + " " + type_hint),
                   "asset_type/name heuristic", "Medium")
    elif bsector and bsector != "etf":
        # Security with a broker sector label -> trust it.
        if bsector in ("financial services", "banking", "bank", "nbfc",
                        "insurance", "housing finance"):
            sub = ("Insurance"
                   if any(_contains(hay, k) for k in INSURANCE_HINTS)
                   else _bank_sub(hay))
            _set_class(h, CLASS_BANKS_FI, sub, "broker sector (financials)", "High")
        else:
            _set_class(h, CLASS_STOCKS, Config.NA, "broker sector (equity)", "High")
        h.sector = _prettify_sector(bsector)
    else:
        _classify_by_keywords(h, hay, type_hint)
        return

    _flag_unknown(h)

def _classify_by_keywords(h: Holding, hay: str, type_hint: str) -> None:
    """Fallback classification for files lacking ISIN/broker-sector columns."""
    if "etf" in type_hint:
        _set_class(h, CLASS_ETF, _etf_flavour(hay), "asset_type column", "High")
    elif any(k in type_hint for k in ["mutual", "mf", "scheme", "fund"]):
        _set_class(h, CLASS_MF, _mf_subcategory(hay), "asset_type column", "High")
    elif any(k in type_hint for k in ["equity", "stock", "share", "common"]):
        _set_class(h, CLASS_STOCKS, Config.NA, "asset_type column", "High")
    elif any(_contains(hay, k) for k in MF_NAME_HINTS):
        _set_class(h, CLASS_MF, _mf_subcategory(hay), "name heuristic", "Medium")
    elif any(_contains(hay, k) for k in INSURANCE_HINTS):
        _set_class(h, CLASS_BANKS_FI, "Insurance", "name heuristic", "Medium")
    elif any(_contains(hay, k) for k in BANK_FI_HINTS):
        _set_class(h, CLASS_BANKS_FI, _bank_sub(hay), "name heuristic", "Medium")
    else:
        _set_class(h, CLASS_STOCKS, Config.NA, "default (equity-like)", "Low")

    if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI):
        sym_key = (h.symbol or "").upper().replace(" ", "")
        if sym_key in KNOWN_SYMBOL_SECTOR:
            h.sector = KNOWN_SYMBOL_SECTOR[sym_key]
            if h.classification_source == "default (equity-like)":
                h.classification_source = "known-symbol sector map"
                h.confidence = "Medium"
        else:
            h.sector = _detect_sector(hay)
    _flag_unknown(h)

def _flag_unknown(h: Holding) -> None:
    if h.asset_class == CLASS_STOCKS and h.sector in (Config.NA, "") and h.confidence == "Low":
        h.asset_class = CLASS_UNKNOWN
        h.issues.append("Could not confidently identify asset type/sector.")

def _set_class(h: Holding, cls: str, sub: str, source: str, conf: str):
    h.asset_class = cls
    h.sub_category = sub
    h.classification_source = source
    h.confidence = conf

def _detect_sector(hay: str) -> str:
    for sector, kws in SECTOR_KEYWORDS.items():
        for kw in kws:
            if _contains(hay, kw):
                return sector
    return Config.NA

def _bank_sub(hay: str) -> str:
    if "insurance" in hay:
        return "Insurance"
    if any(k in hay for k in ["small finance", "sfb"]):
        return "Small Finance Bank"
    if any(k in hay for k in ["housing", "hudco", "home fin"]):
        return "Housing Finance"
    if any(k in hay for k in ["nbfc", "finance", "credit", "capital", "loan"]):
        return "NBFC"
    return "Bank"

def _etf_flavour(hay: str) -> str:
    if "gold" in hay:
        return "Gold ETF"
    if "silver" in hay:
        return "Silver ETF"
    if any(k in hay for k in ["us", "nasdaq", "s&p", "sp500", "global"]):
        return "International / US ETF"
    if "bank" in hay:
        return "Sectoral ETF (Bank)"
    if "nifty 50" in hay or "niftybees" in hay:
        return "Broad-market ETF (Nifty 50)"
    if any(k in hay for k in ["midcap", "mid cap", "juniorbees", "small"]):
        return "Mid/Small-cap ETF"
    if "it" in hay or "tech" in hay:
        return "Thematic ETF (IT/Tech)"
    return "ETF"

def _mf_subcategory(hay: str) -> str:
    for key, label in [
        ("elss", "ELSS"), ("flexi", "Flexi-cap"), ("multi", "Multi-cap"),
        ("small cap", "Small-cap"), ("mid cap", "Mid-cap"),
        ("large cap", "Large-cap"), ("index", "Index Fund"),
        ("liquid", "Liquid Fund"), ("gilt", "Gilt / Duration Fund"),
        ("debt", "Debt Fund"), ("hybrid", "Hybrid Fund"),
        ("fof", "Fund of Funds"), ("fund of fund", "Fund of Funds"),
        ("nifty 50", "Index Fund"), ("sensex", "Index Fund"),
    ]:
        if key in hay:
            plan = "Direct" if "direct" in hay else ("Regular" if "regular" in hay else Config.NA)
            opt = "Growth" if "growth" in hay else ("IDCW" if any(k in hay for k in
                  ["idcw", "dividend"]) else Config.NA)
            return f"{label} ({plan}, {opt})".replace("(, )", "").replace("N/A", "Unknown")
    return "Mutual Fund"


# -----------------------------------------------------------------------------
# 8. FINANCIAL DATA COLLECTION  (yfinance where suitable + caching + retries)
# -----------------------------------------------------------------------------
# IMPORTANT DISCLOSURE:
#   yfinance provides *delayed / historical* data and does NOT reliably cover
#   Indian mutual-fund NAVs, NSE/BSE filings, shareholding, or bank-specific
#   fundamentals (NPA, CASA, NIM, etc.). Anything not available is marked N/A.
#   We never invent numbers.
class DataCollector:
    # Broker NSE symbols that differ from the Yahoo Finance symbol. The broker
    # export uses one scrip code, Yahoo lists the company under another; without
    # this the lookup 404s and the whole holding degrades to N/A.
    YF_SYMBOL_ALIASES = {
        "SILVERLINE": "SILTECH",   # Silverline Technologies Ltd (NSE: SILTECH)
        "NSDL": "NSEL",            # NSE Strategic Depository Ltd (NSE: NSEL, BSE-listed)
    }

    def __init__(self):
        self.cache: Dict[str, Any] = {}
        self.source_timestamps: Dict[str, str] = {}
        self.failures: List[str] = []
        self.uses_network = False
        try:
            import yfinance as yf
            self.yf = yf
            self.has_yf = True
        except Exception as e:
            self.yf = None
            self.has_yf = False
            LOG.warning("yfinance unavailable (%s); market data will be N/A.", e)
        os.makedirs(Config.CACHE_DIR, exist_ok=True)

    # ---- ticker resolution ------------------------------------------------
    def resolve_yf_ticker(self, h: Holding) -> str:
        """
        Map a holding to a yfinance symbol. Indian equities trade on NSE and use
        '.NS' suffix. We only append it to plausible alphabetic tickers; we never
        guess on mutual funds (yf lacks most Indian MF schemes).
        """
        sym = (h.symbol or h.name or "").strip().upper()
        sym = re.sub(r"\s+", "", sym)
        # Already suffixed?
        if sym.endswith(".NS") or sym.endswith(".BO"):
            return sym
        # Drop fund/plan noise for MF: no reliable yf ticker.
        if h.asset_class == CLASS_MF:
            return ""
        # NSE broker symbols sometimes carry an exchange/series suffix
        # (TATAGOLD-E, SILVERLINE-X, IRBINVIT-IV, TATSILV-E). Strip it.
        base = sym.split("-")[0]
        base = self.YF_SYMBOL_ALIASES.get(base, base)
        # A usable NSE ticker: starts with a letter, then alphanumerics
        # (e.g. NIFTYBEES, NEXT50IETF, ICICIB22, IRCTC).
        if re.fullmatch(r"[A-Z][A-Z0-9]{1,13}", base):
            return f"{base}.NS"
        return ""

    def _ticker_candidates(self, ticker: str) -> List[str]:
        """Primary ticker plus the other Indian exchange suffix as a fallback
        (a symbol missing on NSE may exist on BSE and vice-versa)."""
        if ticker.endswith(".NS"):
            return [ticker, ticker[:-3] + ".BO"]
        if ticker.endswith(".BO"):
            return [ticker, ticker[:-3] + ".NS"]
        return [ticker]

    def _cache_key(self, kind: str, ident: str) -> str:
        raw = f"{kind}:{ident}"
        return hashlib.md5(raw.encode()).hexdigest()

    def _cached(self, key: str, ttl_sec: int = 6 * 3600) -> Optional[Any]:
        path = os.path.join(Config.CACHE_DIR, key + ".json")
        if not os.path.exists(path):
            return None
        if time.time() - os.path.getmtime(path) > ttl_sec:
            return None
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return None

    def _store_cache(self, key: str, obj: Any) -> None:
        try:
            path = os.path.join(Config.CACHE_DIR, key + ".json")
            with open(path, "w", encoding="utf-8") as f:
                json.dump(obj, f, default=str)
        except Exception:
            pass

    def fetch_price_history(self, ticker: str, period: str = "1y") -> Optional[pd.DataFrame]:
        if not self.has_yf or not ticker:
            return None
        key = self._cache_key("hist", f"{ticker}:{period}")
        cached = self._cached(key)
        if cached is not None:
            return pd.DataFrame(cached)
        for cand in self._ticker_candidates(ticker):
            for attempt in range(1, Config.MAX_RETRIES + 1):
                try:
                    self.uses_network = True
                    tk = self.yf.Ticker(cand)
                    hist = tk.history(period=period)
                    if hist is None or hist.empty:
                        break           # no data here -> try the other exchange
                    out = hist.reset_index()[["Date", "Close", "Volume"]].copy()
                    out["Date"] = out["Date"].astype(str)
                    self.cache[ticker] = out
                    self.source_timestamps[ticker] = str(out["Date"].iloc[-1])
                    self._store_cache(key, out.to_dict(orient="records"))
                    if cand != ticker:
                        LOG.info("Price history for %s found via %s.", ticker, cand)
                    return out
                except Exception as e:
                    LOG.debug("history attempt %d failed for %s: %s", attempt, cand, e)
                    time.sleep(min(2 ** attempt, 6))
        self.failures.append(f"price-history:{ticker}")
        return None

    def fetch_info(self, ticker: str) -> Dict[str, Any]:
        """Fetch yfinance .info / .fast_info; returns {} on failure. N/A-friendly."""
        if not self.has_yf or not ticker:
            return {}
        key = self._cache_key("info", ticker)
        cached = self._cached(key, ttl_sec=6 * 3600)
        if cached is not None:
            return cached
        for cand in self._ticker_candidates(ticker):
            for attempt in range(1, Config.MAX_RETRIES + 1):
                try:
                    self.uses_network = True
                    tk = self.yf.Ticker(cand)
                    info = {}
                    try:
                        info.update(tk.info or {})
                    except Exception:
                        pass
                    # Merge fast_info primitives.
                    try:
                        fi = dict(tk.fast_info) if tk.fast_info else {}
                        info.setdefault("lastPrice", fi.get("lastPrice"))
                        info.setdefault("currency", fi.get("currency"))
                    except Exception:
                        pass
                    # An empty / quote-missing payload -> try the other exchange.
                    has_price = any(info.get(k) is not None for k in
                                    ("regularMarketPrice", "previousClose", "lastPrice"))
                    if not info or not has_price:
                        break
                    # Only JSON-serializable subset cached.
                    slim = {k: v for k, v in info.items()
                            if isinstance(v, (int, float, str, bool)) or v is None}
                    self._store_cache(key, slim)
                    if cand != ticker:
                        LOG.info("Quote info for %s found via %s.", ticker, cand)
                    return info
                except Exception as e:
                    LOG.debug("info attempt %d failed for %s: %s", attempt, cand, e)
                    time.sleep(min(2 ** attempt, 6))
        self.failures.append(f"info:{ticker}")
        return {}

    def fetch_financials(self, ticker: str) -> Dict[str, pd.DataFrame]:
        """
        Fetch multi-year annual statements (balance sheet, income statement,
        cash flow) from yfinance. In-memory cache per run; empty dict on
        failure. These let us COMPUTE ratios like a Screener-style report
        instead of trusting (often missing) pre-computed .info fields.
        """
        if not self.has_yf or not ticker:
            return {}
        if not hasattr(self, "fin_cache"):
            self.fin_cache: Dict[str, Dict[str, pd.DataFrame]] = {}
        if ticker in self.fin_cache:
            return self.fin_cache[ticker]
        out: Dict[str, pd.DataFrame] = {}
        try:
            self.uses_network = True
            tk = self.yf.Ticker(ticker)
            for name, attr in (("bs", "balance_sheet"), ("fin", "income_stmt"),
                               ("cf", "cashflow")):
                try:
                    df = getattr(tk, attr)
                    if df is not None and not df.empty:
                        out[name] = df
                except Exception:
                    pass
        except Exception as e:
            LOG.debug("financials fetch failed for %s: %s", ticker, e)
        self.fin_cache[ticker] = out
        if not out:
            self.failures.append(f"financials:{ticker}")
        return out

    # ---- Best-effort Screener.in fetch (promoter / pledge / ratio cross-check)
    def _http_get(self, url: str) -> Optional[str]:
        """GET a page with a browser-like UA; returns text or None. Never raises."""
        try:
            import requests
        except Exception:
            return None
        headers = {
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/124.0 Safari/537.36"),
            "Accept-Language": "en-US,en;q=0.9",
        }
        try:
            self.uses_network = True
            r = requests.get(url, headers=headers, timeout=Config.HTTP_TIMEOUT)
            if r.status_code == 200 and r.text:
                return r.text
            LOG.debug("HTTP %s for %s", getattr(r, 'status_code', '?'), url)
        except Exception as e:
            LOG.debug("HTTP GET failed for %s: %s", url, e)
        return None

    def fetch_screener(self, h: "Holding") -> Dict[str, Any]:
        """
        Best-effort scrape of Screener.in (the only one of Screener/Tickertape/
        NSE/BSE that is reliably fetchable from a Colab data-centre IP) to fill
        the rows yfinance cannot: promoter holding %, pledged %, and a couple of
        ratio cross-checks. Returns {} on ANY failure so those rows simply stay
        N/A - we never guess. Cached per run.
        """
        if not hasattr(self, "screener_cache"):
            self.screener_cache: Dict[str, Dict[str, Any]] = {}
        sym = (h.symbol or h.name or "").strip().upper()
        base = re.sub(r"\s+", "", sym.split("-")[0])
        if not base:
            return {}
        if base in self.screener_cache:
            return self.screener_cache[base]
        slugs = []
        primary = DataCollector.YF_SYMBOL_ALIASES.get(base, base).lower()
        for s in (primary, base.lower()):
            if s and s not in slugs:
                slugs.append(s)
        out: Dict[str, Any] = {}
        html = None
        for slug in slugs:
            html = self._http_get(f"https://www.screener.in/company/{slug}/consolidated/")
            if html and ("Promoters" in html or "company-info" in html):
                out["url"] = f"https://www.screener.in/company/{slug}/consolidated/"
                break
            html = None
        if not html:
            self.screener_cache[base] = {}
            return {}
        try:
            # Promoter holding %: the 'Promoters' shareholding row, latest column.
            m = re.search(r"Promoters\s*</td>(.*?)</tr>", html, re.IGNORECASE | re.DOTALL)
            if m:
                cells = re.findall(r">\s*([0-9]+(?:\.[0-9]+)?)\s*<", m.group(1))
                if cells:
                    out["promoter_pct"] = float(cells[-1])
            # Pledged %: Screener prints an explicit sentence when it is non-zero.
            pm = re.search(r"pledged[^.]*?([0-9]+(?:\.[0-9]+)?)\s*%", html,
                           re.IGNORECASE)
            if pm:
                out["pledged_pct"] = float(pm.group(1))
            elif re.search(r"no pledged|0%.*pledged|pledged.*0%", html, re.IGNORECASE):
                out["pledged_pct"] = 0.0
            # Cross-check ratios from the top 'company-info' block (optional).
            for label, key in (("Stock P/E", "pe"), ("ROE", "roe"), ("ROCE", "roce"),
                               ("Book Value", "book_value"), ("Dividend Yield", "div_yield")):
                lm = re.search(re.escape(label) + r"\s*</li>\s*<[^>]*class=\"data-item\">\s*([^<]+)<",
                               html, re.IGNORECASE)
                if not lm:
                    lm = re.search(re.escape(label) + r"[^0-9₹%-]{0,40}([0-9][0-9,]*\.?[0-9]*)", html,
                                   re.IGNORECASE)
                if lm:
                    v = to_float(lm.group(1))
                    if v is not None:
                        out[key] = v
        except Exception as e:
            LOG.debug("Screener parse failed for %s: %s", base, e)
        self.screener_cache[base] = out
        if out:
            LOG.info("Screener.in data for %s: %s", base,
                     {k: v for k, v in out.items() if k != "url"})
        return out


# -----------------------------------------------------------------------------
# 9. SEPARATE ANALYSIS ENGINES  (do NOT share one scoring formula)
# -----------------------------------------------------------------------------
# ---- statement-mining helpers (Screener-style ratios from raw statements) ----
def _clean_num(x: Any) -> Optional[float]:
    try:
        v = float(x)
        return None if math.isnan(v) else v
    except Exception:
        return None

def _series_row(df: Optional[pd.DataFrame], names: List[str]) -> List[Optional[float]]:
    """Values of the first matching line item, most-recent period first."""
    if df is None or getattr(df, "empty", True):
        return []
    for nm in names:
        try:
            if nm in df.index:
                row = df.loc[nm]
                if isinstance(row, pd.DataFrame):   # duplicated line-item labels
                    row = row.iloc[0]
                return [_clean_num(x) for x in row.tolist()]
        except Exception:
            continue
    return []

def _paired_pct(a: List[Optional[float]], b: List[Optional[float]]) -> List[Optional[float]]:
    out = []
    for i in range(min(len(a), len(b))):
        if a[i] is None or not b[i]:
            out.append(None)
        else:
            out.append(a[i] / b[i] * 100.0)
    return out

def _avg(vals: List[Optional[float]]) -> Optional[float]:
    vs = [v for v in vals if v is not None]
    return (sum(vs) / len(vs)) if vs else None

def _last(vals: List[Optional[float]]) -> Optional[float]:
    for v in vals:
        if v is not None:
            return v
    return None

def _oldest(vals: List[Optional[float]]) -> Optional[float]:
    for v in reversed(vals):
        if v is not None:
            return v
    return None

def _cagr(vals: List[Optional[float]]) -> Tuple[Optional[float], int]:
    """(growth % per year, periods) between latest and oldest non-null value."""
    v0, vn = _last(vals), _oldest(vals)
    n = len(vals) - 1
    if v0 is None or vn is None or n < 1 or v0 <= 0 or vn <= 0:
        return (None, n)
    return (((v0 / vn) ** (1.0 / n) - 1.0) * 100.0, n)

def _yoy(vals: List[Optional[float]]) -> Optional[float]:
    """Latest vs previous period growth, percent."""
    if len(vals) < 2 or vals[0] is None or not vals[1]:
        return None
    return (vals[0] / vals[1] - 1.0) * 100.0

def compute_fundamentals(fs: Dict[str, pd.DataFrame]) -> Dict[str, Any]:
    """
    Compute multi-year ratios and growth rates from raw annual statements
    (balance sheet 'bs', income statement 'fin', cash flow 'cf'). Anything the
    statements cannot support stays None -> shown as N/A, never guessed.
    """
    fin, bs, cf = fs.get("fin"), fs.get("bs"), fs.get("cf")
    revenue = _series_row(fin, ["Total Revenue", "Operating Revenue"])
    ebit    = _series_row(fin, ["Operating Income"])
    dep     = _series_row(fin, ["Reconciled Depreciation",
                               "Depreciation And Amortization",
                               "Depreciation & Amortization"])
    ni      = _series_row(fin, ["Net Income", "Net Income Common Stockholders"])
    eps     = _series_row(fin, ["Basic EPS", "Diluted Normalized EPS"])
    intexp  = _series_row(fin, ["Interest Expense"])
    equity  = _series_row(bs, ["Stockholders Equity",
                               "Total Equity Gross Minority Interest",
                               "Common Stock Equity"])
    debt    = _series_row(bs, ["Total Debt", "Long Term Debt", "LongTerm Debt"])
    cfo     = _series_row(cf, ["Operating Cash Flow",
                               "Total Cash From Operating Activities"])
    capex   = _series_row(cf, ["Capital Expenditure", "Capital Expenditures"])

    ebitda: List[Optional[float]] = []
    for i in range(min(len(ebit), len(dep)) if dep else len(ebit)):
        if ebit[i] is None:
            ebitda.append(None)
        else:
            ebitda.append(ebit[i] + abs(dep[i] if i < len(dep) and dep[i] else 0.0))

    roce: List[Optional[float]] = []
    for i in range(min(len(ebit), len(equity), len(debt)) if debt else min(len(ebit), len(equity))):
        d = debt[i] if i < len(debt) and debt[i] else 0.0
        ce = (equity[i] or 0.0) + d if i < len(equity) else None
        roce.append(None if (ebit[i] is None or not ce) else ebit[i] / ce * 100.0)

    de_ratio: List[Optional[float]] = []
    for i in range(min(len(debt), len(equity))):
        e = equity[i]
        de_ratio.append(None if (debt[i] is None or not e) else debt[i] / e)

    icov: List[Optional[float]] = []
    for i in range(min(len(ebit), len(intexp))):
        ie = abs(intexp[i]) if intexp[i] is not None else None
        if ebit[i] is None:
            icov.append(None)
        elif not ie:
            icov.append(None)   # no interest expense -> coverage not meaningful
        else:
            icov.append(ebit[i] / ie)

    fcf: List[Optional[float]] = []
    for i in range(min(len(cfo), len(capex))):
        if cfo[i] is None:
            fcf.append(None)
        else:
            fcf.append(cfo[i] - abs(capex[i] or 0.0))

    roe = _paired_pct(ni, equity)
    ebitda_m = _paired_pct(ebitda, revenue)
    net_m = _paired_pct(ni, revenue)
    cf_m = _paired_pct(cfo, revenue)
    return {
        "revenue": revenue, "ebitda": ebitda, "ni": ni, "eps": eps,
        "cfo": cfo, "capex": capex, "equity": equity, "debt": debt,
        "fcf": fcf, "roe": roe, "roce": roce,
        "ebitda_margin": ebitda_m, "net_margin": net_m, "cf_margin": cf_m,
        "debt_equity": de_ratio, "interest_cov": icov,
        "roe_avg": _avg(roe), "roce_avg": _avg(roce),
        "ebitda_margin_avg": _avg(ebitda_m), "net_margin_avg": _avg(net_m),
        "cf_margin_avg": _avg(cf_m),
        "fcf_last": _last(fcf), "de_last": _last(de_ratio),
        "icov_last": _last(icov),
        "rev_cagr": _cagr(revenue), "ebitda_cagr": _cagr(ebitda),
        "eps_cagr": _cagr(eps), "ni_cagr": _cagr(ni),
        "rev_yoy": _yoy(revenue), "ni_yoy": _yoy(ni),
        "cfo_yoy": _yoy(cfo), "fcf_yoy": _yoy(fcf),
        "years": max(len(revenue), len(ni), len(roe), len(roce)),
    }


# -----------------------------------------------------------------------------
# 6-STEP RULE ENGINE (modelled on the user's Screener-style checklist)
# -----------------------------------------------------------------------------
SIG_PASS, SIG_FAIL, SIG_WATCH, SIG_NA = "PASS", "FAIL", "WATCH", "N/A"
_ACTION = {SIG_PASS: "BUY MORE", SIG_FAIL: "HOLD", SIG_WATCH: "HOLD", SIG_NA: "VERIFY"}

def _rule(check: str, rule: str, value: str, signal: str) -> Dict[str, str]:
    return {"check": check, "rule": rule, "value": value,
            "signal": signal, "action": _ACTION.get(signal, "HOLD")}

def _pc(v: Optional[float], dp: int = 1) -> str:
    return Config.NA if v is None else f"{v:.{dp}f}%"

def _xx(v: Optional[float], dp: int = 2) -> str:
    return Config.NA if v is None else f"{v:.{dp}f}x"

def _cr(v: Optional[float]) -> str:
    return Config.NA if v is None else f"{Config.CURRENCY_SYMBOL}{v:,.0f} cr"

def _sig(cond: bool, na: bool = False) -> str:
    if na:
        return SIG_NA
    return SIG_PASS if cond else SIG_FAIL

def build_stock_rules(fund: Dict[str, Any], info: Dict[str, Any],
                      pr: Dict[str, Any], is_financial: bool = False) -> Dict[str, Any]:
    """
    Build the four scored steps. Every check shows the value and a PASS/FAIL/
    WATCH/N/A signal. Checks that cannot be computed from available data are
    marked N/A and EXCLUDED from the score (never guessed). For Banks/NBFCs the
    Debt/Equity and Interest-coverage rules are not applicable (N/A).
    """
    def num(v):
        return v if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v)) else None

    mc_cr = (info.get("marketCap") or 0) / 1e7 if num(info.get("marketCap")) else None
    roce_avg, roe_avg = fund.get("roce_avg"), fund.get("roe_avg")
    roce_now, roe_now = _last(fund.get("roce", [])), _last(fund.get("roe", []))
    fcf = fund.get("fcf_last")
    de = fund.get("de_last")
    icov = fund.get("icov_last")
    rev_c = fund.get("rev_cagr", (None, 0))[0]
    eb_c = fund.get("ebitda_cagr", (None, 0))[0]
    eps_c = fund.get("eps_cagr", (None, 0))[0]
    rev_y, ni_y, cfo_y = fund.get("rev_yoy"), fund.get("ni_yoy"), fund.get("cfo_yoy")
    pe, pb = num(info.get("trailingPE")), num(info.get("priceToBook"))
    pfcf = (mc_cr / (fcf / 1e7)) if (fcf and mc_cr) else None
    years = fund.get("years", 0)
    # Values merged in from Screener.in / the optional extras upload (underscore
    # keys). When present these turn previously-N/A rows into real PASS/FAIL.
    prom = num(info.get("_promoter_pct"))
    pledge = num(info.get("_pledged_pct"))
    sector_pe = num(info.get("_sector_pe"))
    red_flags = num(info.get("_red_flags"))
    _ps = info.get("_promoter_selling")
    prom_selling = None if _ps is None else (str(_ps).strip().lower() in
                                             ("1", "true", "yes", "y", "selling"))

    # STEP 1 - quality & risk
    s1 = [
        _rule("Market cap", ">= " + _cr(1000), _cr(mc_cr),
              _sig(bool(mc_cr and mc_cr >= 1000), mc_cr is None)),
        _rule("Promoter holding", "> 50%",
              (f"{prom:.1f}% (Screener/extras)" if prom is not None
               else "N/A (needs BSE/NSE shareholding filing)"),
              _sig(bool(prom and prom > 50), prom is None)),
        _rule("Promoter pledge", "< 15% / ideally 0%",
              (f"{pledge:.1f}% pledged (Screener/extras)" if pledge is not None
               else "N/A (needs latest filing)"),
              _sig(bool(pledge is not None and pledge < 15), pledge is None)),
        _rule("Debt to equity", "< 1.5" if not is_financial else "N/A for banks/NBFCs",
              (_pc(None) if is_financial else _xx(de)),
              SIG_NA if is_financial else _sig(bool(de is not None and de < 1.5), de is None)),
        _rule("Interest coverage", "> 3" if not is_financial else "N/A for banks/NBFCs",
              _xx(icov), SIG_NA if is_financial else _sig(bool(icov and icov > 3), icov is None)),
    ]
    # STEP 2 - profitability
    s2 = [
        _rule("5y avg ROCE", "> 15%", f"{_pc(roce_avg)} (now {_pc(roce_now)})",
              _sig(bool(roce_avg and roce_avg > 15), roce_avg is None)),
        _rule("5y avg ROE", "> 12%", f"{_pc(roe_avg)} (now {_pc(roe_now)})",
              _sig(bool(roe_avg and roe_avg > 12), roe_avg is None)),
        _rule("EBITDA margin", "> 18%", _pc(_last(fund.get("ebitda_margin", []))),
              _sig(bool(_last(fund.get("ebitda_margin", [])) and _last(fund.get("ebitda_margin", [])) > 18),
                   _last(fund.get("ebitda_margin", [])) is None)),
        _rule("Net profit margin", "> 10%", _pc(_last(fund.get("net_margin", []))),
              _sig(bool(_last(fund.get("net_margin", [])) and _last(fund.get("net_margin", [])) > 10),
                   _last(fund.get("net_margin", [])) is None)),
        _rule("Cash-flow margin", "> 10%", _pc(_last(fund.get("cf_margin", []))),
              _sig(bool(_last(fund.get("cf_margin", [])) and _last(fund.get("cf_margin", [])) > 10),
                   _last(fund.get("cf_margin", [])) is None)),
        _rule("Free cash flow", "Positive", _cr(fcf / 1e7 if fcf else None),
              _sig(bool(fcf and fcf > 0), fcf is None)),
    ]
    # STEP 3 - growth
    triangle_ok = None
    if rev_y is not None and ni_y is not None and cfo_y is not None:
        triangle_ok = (rev_y > 0 and ni_y > 0 and cfo_y > 0)
    s3 = [
        _rule(f"Revenue growth ({years}y CAGR)", "> 10%", _pc(rev_c),
              _sig(bool(rev_c and rev_c > 10), rev_c is None)),
        _rule(f"EBITDA growth ({years}y CAGR)", "> 12%", _pc(eb_c),
              _sig(bool(eb_c and eb_c > 12), eb_c is None)),
        _rule(f"EPS growth ({years}y CAGR)", "> 10%", _pc(eps_c),
              _sig(bool(eps_c and eps_c > 10), eps_c is None)),
        _rule("Profit triangle", "Revenue, Profit & Cash flow all grow",
              (f"Rev {_pc(rev_y)}, Profit {_pc(ni_y)}, CFO {_pc(cfo_y)} (YoY)"),
              _sig(bool(triangle_ok), triangle_ok is None)),
    ]
    # STEP 4 - price / valuation
    if pe and sector_pe:
        _pe_val = f"P/E {_xx(pe)} vs sector {_xx(sector_pe)} (extras)"
        _pe_sig = _sig(pe <= sector_pe * 1.25)
    else:
        _pe_val = f"P/E {_xx(pe)} (sector P/E not supplied)" if pe else \
                  "P/E N/A (sector P/E not supplied)"
        _pe_sig = SIG_WATCH if pe else SIG_NA
    s4 = [
        _rule("P/E vs sector", "No big premium to sector", _pe_val, _pe_sig),
        _rule("P/B vs sector", "No big premium to sector",
              f"P/B {_xx(pb)} supported by ROE {_pc(roe_now)}",
              SIG_WATCH if pb else SIG_NA),
        _rule("Price to free cash flow", "< 25", _xx(pfcf, 1),
              _sig(bool(pfcf and pfcf < 25), pfcf is None)),
        _rule("Promoter selling", "No continuous selling",
              ("Promoter selling noted (extras)" if prom_selling
               else "No promoter selling (extras)" if prom_selling is not None
               else "N/A (needs shareholding history)"),
              _sig(not prom_selling, prom_selling is None) if prom_selling is not None
              else SIG_NA),
        _rule("Red flags", "<= 2 red flags",
              (f"{int(red_flags)} red flag(s) (extras)" if red_flags is not None
               else "N/A (needs Tickertape/Screener screen)"),
              _sig(bool(red_flags is not None and red_flags <= 2), red_flags is None)),
    ]
    steps = [("Step 1: Eliminating low quality and risk", s1),
             ("Step 2: Is it profitable?", s2),
             ("Step 3: Is it growing?", s3),
             ("Step 4: Is the price right?", s4)]

    allrows = [r for _, rows in steps for r in rows]
    n_pass = sum(1 for r in allrows if r["signal"] == SIG_PASS)
    n_fail = sum(1 for r in allrows if r["signal"] == SIG_FAIL)
    n_watch = sum(1 for r in allrows if r["signal"] == SIG_WATCH)
    n_na = sum(1 for r in allrows if r["signal"] == SIG_NA)
    applicable = n_pass + n_fail + n_watch
    step1_applicable = [r for r in s1 if r["signal"] != SIG_NA]
    step1_all_pass = bool(step1_applicable) and all(
        r["signal"] == SIG_PASS for r in step1_applicable)
    ratio = safe_div(n_pass, applicable)
    if applicable < 8 or ratio is None:
        signal = "INSUFFICIENT DATA"
    elif ratio >= 0.85 and step1_all_pass:
        signal = "BUY MORE"
    elif ratio >= 0.55:
        signal = "HOLD"
    else:
        signal = "SELL"
    return {"steps": steps, "pass": n_pass, "fail": n_fail, "watch": n_watch,
            "na": n_na, "applicable": applicable, "ratio_pct": (ratio * 100 if ratio else None),
            "signal": signal, "step1_all_pass": step1_all_pass}


# Shared helper: transparent scoring from available metrics only.
class BaseEngine:
    def __init__(self, collector: DataCollector):
        self.collector = collector

    def _na(self, h: Holding, key: str, val: Any) -> Any:
        h.metrics[key] = val if val not in (None, "") else Config.NA
        return h.metrics[key]

    def price_and_returns(self, h: Holding) -> Dict[str, Any]:
        """Common market-data block: latest price, ranges, moving averages."""
        out: Dict[str, Any] = {}
        hist = self.collector.fetch_price_history(h.yf_ticker) if h.yf_ticker else None
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        last = info.get("regularMarketPrice") or info.get("previousClose") or \
            info.get("lastPrice")
        if last is None and hist is not None and len(hist):
            last = float(hist["Close"].iloc[-1])
        out["last_price"] = last
        out["price_as_of"] = self.collector.source_timestamps.get(h.yf_ticker, Config.NA)
        if hist is not None and len(hist):
            closes = hist["Close"].astype(float)
            out["wk52_high"] = float(closes.max())
            out["wk52_low"] = float(closes.min())
            out["sma_50"] = float(closes.tail(50).mean()) if len(closes) >= 50 else None
            out["sma_200"] = float(closes.tail(200).mean()) if len(closes) >= 200 else None
            ret = safe_div(last, closes.iloc[0]) if closes.iloc[0] else None
            out["period_return_pct"] = ((ret - 1) * 100) if ret else None
            # Simple annualised volatility from daily returns.
            dr = closes.pct_change().dropna()
            out["volatility_pct"] = float(dr.std() * math.sqrt(252) * 100) if len(dr) > 5 else None
            # Rolling horizon returns computed from the same price series.
            dts = pd.to_datetime(hist["Date"].astype(str))
            for label, yrs in (("ret_1y", 1), ("ret_3y", 3), ("ret_5y", 5)):
                out[label] = None
                if last:
                    target = dts.iloc[-1] - pd.Timedelta(days=int(365.25 * yrs))
                    sel = dts[dts <= target]
                    if len(sel):
                        base = float(closes.loc[sel.index[-1]])
                        if base:
                            out[label] = (safe_div(last, base) - 1) * 100 if safe_div(last, base) else None
        else:
            out["wk52_high"] = out["wk52_low"] = out["sma_50"] = \
                out["sma_200"] = out["period_return_pct"] = out["volatility_pct"] = None
        for label in ("ret_1y", "ret_3y", "ret_5y"):
            out.setdefault(label, None)
        return out

    # ---- extras / Screener merging (rows the free API alone leaves N/A) ----
    def _apply_extras(self, h: Holding, m: Dict[str, Any],
                      info: Dict[str, Any]) -> None:
        """
        Merge the optional user upload: values fill ONLY metrics that are still
        N/A (never overwrite a computed number), and the scoring underscore keys
        are set so previously-N/A rule rows become real PASS/FAIL.
        """
        if not h.extras:
            return
        by_mkey = {}
        for ekey, canon in EXTRAS_METRIC_KEYS.items():
            if canon in h.extras:
                by_mkey[_extras_key(canon)] = h.extras[canon]
        for k in list(m.keys()):
            v = by_mkey.get(_extras_key(k))
            if v is not None and str(m.get(k)) == Config.NA:
                m[k] = v
        for extra_col, mkey in EXTRAS_METRIC_KEYS.items():
            if extra_col in h.extras and _extras_key(mkey) not in by_mkey:
                m[mkey] = h.extras[extra_col]      # new metric supplied by the user
        for extra_col, ikey in EXTRAS_RULE_KEYS.items():
            if extra_col in h.extras and info.get(ikey) is None:
                info[ikey] = h.extras[extra_col]

    def _apply_screener(self, h: Holding, info: Dict[str, Any]) -> Dict[str, Any]:
        """
        Best-effort Screener.in merge (promoter %, pledged %, ratio cross-checks).
        Extras win over Screener: Screener only fills underscore keys still absent.
        """
        sc = self.collector.fetch_screener(h)
        if not sc:
            return {}
        for src, dst in (("promoter_pct", "_promoter_pct"),
                         ("pledged_pct", "_pledged_pct"),
                         ("pe", "_screener_pe"), ("roe", "_screener_roe"),
                         ("roce", "_screener_roce")):
            if sc.get(src) is not None and info.get(dst) is None:
                info[dst] = sc[src]
        return sc

    def _source_note(self, h: Holding, info: Dict[str, Any]) -> str:
        """Honest, per-holding provenance line shown in the PDF."""
        parts = []
        if h.yf_ticker:
            parts.append("yfinance: quotes, price history & ratios computed from "
                         "annual statements (delayed/historical)")
        if info.get("_promoter_pct") is not None or info.get("_pledged_pct") is not None:
            parts.append("Screener.in: promoter/pledge (best-effort scrape)")
        if h.extras:
            parts.append("user-supplied extras file: " +
                         ", ".join(_extras_label(k) for k in sorted(h.extras.keys())))
        if not parts:
            return "no market data"
        return ("; ".join(parts) + ". Anything not supplied stays N/A - never "
                "guessed.")


class IndividualStockEngine(BaseEngine):
    """Fundamental + technical scoring for non-financial companies."""
    def analyze(self, h: Holding) -> None:
        m = {}
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        fs = self.collector.fetch_financials(h.yf_ticker) if h.yf_ticker else {}
        fund = compute_fundamentals(fs) if fs else {}
        m["Sector"] = h.sector
        m["Last Price"] = pr["last_price"]
        m["52W High"] = pr["wk52_high"]
        m["52W Low"] = pr["wk52_low"]
        m["SMA50"] = pr["sma_50"]
        m["SMA200"] = pr["sma_200"]
        m["Period Return %"] = pr["period_return_pct"]
        m["Volatility % (ann.)"] = pr["volatility_pct"]
        m["1Y Return %"] = pr["ret_1y"]
        m["3Y Return %"] = pr["ret_3y"]
        m["5Y Return %"] = pr["ret_5y"]
        # Valuation / quality from yfinance info (all optional, N/A if absent).
        m["P/E"] = info.get("trailingPE")
        m["P/B"] = info.get("priceToBook")
        m["Dividend Yield %"] = (info.get("dividendYield", 0) * 100) if info.get("dividendYield") else None
        m["Market Cap"] = info.get("marketCap")
        # --- Computed fundamentals from raw annual statements (preferred) ------
        m["ROCE % (5y avg)"] = fund.get("roce_avg")
        m["ROCE % (latest)"] = _last(fund.get("roce", []))
        m["ROE % (5y avg)"] = fund.get("roe_avg")
        m["ROE % (latest)"] = _last(fund.get("roe", []))
        m["EBITDA Margin %"] = _last(fund.get("ebitda_margin", []))
        m["Net Profit Margin %"] = _last(fund.get("net_margin", []))
        m["Cash-Flow Margin %"] = _last(fund.get("cf_margin", []))
        m["Debt/Equity"] = fund.get("de_last")
        m["Interest Coverage"] = fund.get("icov_last")
        m["Free Cash Flow"] = fund.get("fcf_last")
        m["Revenue CAGR %"] = fund.get("rev_cagr", (None, 0))[0]
        m["EBITDA CAGR %"] = fund.get("ebitda_cagr", (None, 0))[0]
        m["EPS CAGR %"] = fund.get("eps_cagr", (None, 0))[0]
        # Fall back to yfinance .info if a computed value is missing.
        if m["ROE % (latest)"] is None:
            m["ROE % (latest)"] = (info.get("returnOnEquity", 0) * 100) if info.get("returnOnEquity") else None
        if m["Free Cash Flow"] is None:
            m["Free Cash Flow"] = info.get("freeCashflow")
        # Metrics not obtainable from free data for India (honest N/A unless the
        # Screener scrape or the extras upload supplied them):
        sc = self._apply_screener(h, info)
        if sc.get("promoter_pct") is not None:
            m["Promoter Holding %"] = sc["promoter_pct"]
        if sc.get("pledged_pct") is not None:
            m["Pledged %"] = sc["pledged_pct"]
        for k in ["Promoter Holding %", "Pledged %", "Receivables Days",
                  "Inventory Turnover", "Auditor Remarks"]:
            m.setdefault(k, Config.NA)
        self._apply_extras(h, m, info)
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = self._source_note(h, info)
        # Build the 6-step rule engine result when we have fundamentals/statements.
        if fund and (info or pr.get("last_price")):
            rules = build_stock_rules(fund, info, pr, is_financial=False)
            h.analysis["rules"] = rules
            h.assessment = rules["signal"]
            h.assessment_reasons = self._rule_reasons(rules)
        else:
            h.assessment, h.assessment_reasons = self._score(h, m)
        h.analyzed = True

    def _rule_reasons(self, rules: Dict[str, Any]) -> List[str]:
        out = [f"Rule-based score: {rules['pass']} pass / {rules['fail']} fail / "
               f"{rules['watch']} watch / {rules['na']} N/A "
               f"(signal: {rules['signal']})."]
        for title, rows in rules["steps"]:
            fails = [r["check"] for r in rows if r["signal"] == SIG_FAIL]
            if fails:
                out.append(f"{title.split(':')[0]} fails: {', '.join(fails)}")
        if rules["na"] >= 6:
            out.append("Several checks are N/A (promoter/pledge, sector P/E, red flags) "
                       "because that data is not in the free source; verify on "
                       "BSE/NSE filings or Screener before acting.")
        return out


    def _score(self, h: Holding, m: Dict[str, Any]) -> Tuple[str, List[str]]:
        reasons: List[str] = []
        score = 0
        # Each rule only fires when the underlying metric is available.
        def num(v):
            return v if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v)) else None
        pe = num(m.get("P/E")); roe = num(m.get("ROE % (latest)"))
        growth = num(m.get("EPS CAGR %"))
        dte = num(m.get("Debt/Equity")); prc = num(m.get("Last Price"))
        sma200 = num(m.get("SMA200")); margin = num(m.get("Net Profit Margin %"))
        vol = num(m.get("Volatility % (ann.)"))

        if roe is not None:
            if roe >= 18: score += 2; reasons.append(f"Strong ROE {roe:.1f}%")
            elif roe >= 12: score += 1; reasons.append(f"Decent ROE {roe:.1f}%")
            elif roe < 8: score -= 1; reasons.append(f"Weak ROE {roe:.1f}%")
        if growth is not None:
            if growth >= 15: score += 2; reasons.append(f"Earnings growing {growth:.1f}%")
            elif growth < 0: score -= 2; reasons.append(f"Earnings declining {growth:.1f}%")
        if margin is not None and margin >= 15:
            score += 1; reasons.append(f"Healthy net margin {margin:.1f}%")
        if dte is not None:
            if dte <= 0.6: score += 1; reasons.append(f"Low leverage (D/E {dte:.2f})")
            elif dte > 1.5: score -= 2; reasons.append(f"High leverage (D/E {dte:.2f})")
        if pe is not None:
            if 0 < pe < 20: score += 1; reasons.append(f"Reasonable P/E {pe:.1f}")
            elif pe > 60: score -= 1; reasons.append(f"Rich P/E {pe:.1f}")
        if prc is not None and sma200 is not None:
            if prc >= sma200: score += 1; reasons.append("Price above 200-DMA (trend intact)")
            else: score -= 1; reasons.append("Price below 200-DMA (weak trend)")
        if vol is not None and vol > 40:
            reasons.append(f"Elevated volatility {vol:.0f}%")
        if h.pnl_pct is not None and h.pnl_pct < -25:
            reasons.append("Deep unrealised loss - check if temporary vs fundamental")

        # Data-sufficiency guard.
        available = sum(1 for k in ["P/E","ROE % (latest)","EPS CAGR %","Debt/Equity","Last Price"]
                        if num(m.get(k)) is not None)
        if available < 2:
            return "WATCHLIST", ["Insufficient reliable data for a firm assessment. "
                                 "Metrics marked N/A; not forced into BUY/SELL.", *reasons]
        if score >= 5: return "BUY CANDIDATE", reasons
        if score >= 2: return "HOLD", reasons
        if score >= 0: return "WATCHLIST", reasons
        if score >= -2: return "REDUCE", reasons
        return "AVOID", reasons


class BankNBFCEngine(BaseEngine):
    """Banking/NBFC-specific metrics. Most are NOT in yfinance -> honest N/A."""
    def analyze(self, h: Holding) -> None:
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        fs = self.collector.fetch_financials(h.yf_ticker) if h.yf_ticker else {}
        fund = compute_fundamentals(fs) if fs else {}
        m = {
            "Sector": h.sector or "Banking & NBFC",
            "Last Price": pr["last_price"],
            "52W High": pr["wk52_high"], "52W Low": pr["wk52_low"],
            "SMA50": pr["sma_50"], "SMA200": pr["sma_200"],
            "Volatility % (ann.)": pr["volatility_pct"],
            "1Y Return %": pr["ret_1y"], "3Y Return %": pr["ret_3y"],
            "5Y Return %": pr["ret_5y"],
            "Period Return %": pr["period_return_pct"],
            "Market Cap": info.get("marketCap"),
            "P/E": info.get("trailingPE"),
            "P/B": info.get("priceToBook"),
            "Dividend Yield %": (info.get("dividendYield", 0) * 100) if info.get("dividendYield") else None,
            "ROE % (5y avg)": fund.get("roe_avg"),
            "ROE % (latest)": _last(fund.get("roe", [])),
            "Revenue CAGR %": fund.get("rev_cagr", (None, 0))[0],
            "EPS CAGR %": fund.get("eps_cagr", (None, 0))[0],
            # Bank-specific metrics that require authoritative Indian filings:
            "NIM %": Config.NA, "Gross NPA %": Config.NA, "Net NPA %": Config.NA,
            "Provision Coverage %": Config.NA, "Capital Adequacy (CAR) %": Config.NA,
            "CET1 %": Config.NA, "ROA %": Config.NA, "CASA Ratio %": Config.NA,
            "Cost-to-Income %": Config.NA, "Credit Growth %": Config.NA,
            "Deposit Growth %": Config.NA, "Slippages %": Config.NA,
            "Promoter Holding %": Config.NA, "Pledged %": Config.NA,
        }
        if m["ROE % (latest)"] is None:
            m["ROE % (latest)"] = (info.get("returnOnEquity", 0) * 100) if info.get("returnOnEquity") else None
        sc = self._apply_screener(h, info)
        if sc.get("promoter_pct") is not None:
            m["Promoter Holding %"] = sc["promoter_pct"]
        if sc.get("pledged_pct") is not None:
            m["Pledged %"] = sc["pledged_pct"]
        self._apply_extras(h, m, info)
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = (("Bank metrics (NPA, NIM, CAR, CASA) need RBI/company "
                                      "filings; yfinance has none. " if h.yf_ticker else "")
                                     + self._source_note(h, info))
        reasons = ["Bank/NBFC balance-sheet metrics (NPA, NIM, CAR, CASA) are not "
                   "available from the current data source and are shown as N/A.",
                   "Debt/Equity and interest-coverage rules are NOT applicable to "
                   "banks/NBFCs, so they are excluded from the score."]
        pb = m["P/B"] if isinstance(m["P/B"], (int, float)) else None
        roe = m["ROE % (latest)"] if isinstance(m["ROE % (latest)"], (int, float)) else None
        score = 0
        if roe is not None:
            score += 1 if roe >= 14 else (-1 if roe < 9 else 0)
            reasons.append(f"ROE {roe:.1f}%")
        if pb is not None:
            if 0 < pb < 1.5: score += 1; reasons.append(f"P/B {pb:.2f} (not expensive)")
            elif pb > 4: score -= 1; reasons.append(f"P/B {pb:.2f} (premium)")
        if pr["last_price"] and pr["sma_200"] and pr["last_price"] < pr["sma_200"]:
            reasons.append("Trading below 200-DMA")
        assess = "HOLD" if score >= 1 else ("WATCHLIST" if score >= 0 else "REDUCE")
        if fund and (info or pr["last_price"]):
            rules = build_stock_rules(fund, info, pr, is_financial=True)
            h.analysis["rules"] = rules
            reasons.insert(0, f"Rule-based score: {rules['pass']} pass / {rules['fail']} fail / "
                              f"{rules['watch']} watch / {rules['na']} N/A (signal: {rules['signal']}).")
            assess = rules["signal"]
        h.assessment = assess
        h.assessment_reasons = reasons
        h.analyzed = True


class InsuranceEngine(BaseEngine):
    def analyze(self, h: Holding) -> None:
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        fs = self.collector.fetch_financials(h.yf_ticker) if h.yf_ticker else {}
        fund = compute_fundamentals(fs) if fs else {}
        m = {
            "Sector": h.sector or "Insurance",
            "Last Price": pr["last_price"], "52W High": pr["wk52_high"],
            "52W Low": pr["wk52_low"], "SMA50": pr["sma_50"], "SMA200": pr["sma_200"],
            "Volatility % (ann.)": pr["volatility_pct"],
            "1Y Return %": pr["ret_1y"], "3Y Return %": pr["ret_3y"],
            "5Y Return %": pr["ret_5y"],
            "Period Return %": pr["period_return_pct"],
            "Market Cap": info.get("marketCap"),
            "P/E": info.get("trailingPE"), "P/B": info.get("priceToBook"),
            "Dividend Yield %": (info.get("dividendYield", 0) * 100) if info.get("dividendYield") else None,
            "ROE % (5y avg)": fund.get("roe_avg"),
            "ROE % (latest)": _last(fund.get("roe", [])),
            "Revenue CAGR %": fund.get("rev_cagr", (None, 0))[0],
            "EPS CAGR %": fund.get("eps_cagr", (None, 0))[0],
            # Insurance-specific (need company filings / IRDAI data):
            "Solvency Ratio": Config.NA, "Persistency %": Config.NA,
            "Embedded Value": Config.NA, "Value of New Business (VNB)": Config.NA,
            "Product Mix": Config.NA,
            "Promoter Holding %": Config.NA, "Pledged %": Config.NA,
        }
        sc = self._apply_screener(h, info)
        if sc.get("promoter_pct") is not None:
            m["Promoter Holding %"] = sc["promoter_pct"]
        if sc.get("pledged_pct") is not None:
            m["Pledged %"] = sc["pledged_pct"]
        self._apply_extras(h, m, info)
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = (("Insurance metrics (solvency, persistency, EV, VNB) "
                                      "need company/IRDAI filings; yfinance has none. "
                                      if h.yf_ticker else "") + self._source_note(h, info))
        h.assessment = "WATCHLIST"
        h.assessment_reasons = [
            "Insurance-specific metrics (solvency, persistency, EV, VNB margin) are "
            "unavailable from the current source and are shown as N/A.",
            "Firm assessment deferred until authoritative insurance data is added.",
        ]
        if fund and (info or pr["last_price"]):
            rules = build_stock_rules(fund, info, pr, is_financial=True)
            h.analysis["rules"] = rules
            h.assessment_reasons.insert(
                0, f"Rule-based score on generally-available data: {rules['pass']} pass / "
                   f"{rules['fail']} fail / {rules['watch']} watch / {rules['na']} N/A "
                   f"(signal: {rules['signal']}). Insurance checks are still N/A.")
            h.assessment = rules["signal"]
        h.analyzed = True


class ETFEngine(BaseEngine):
    def analyze(self, h: Holding) -> None:
        pr = self.price_and_returns(h)
        info = self.collector.fetch_info(h.yf_ticker) if h.yf_ticker else {}
        m = {
            "ETF Flavour": h.sub_category,
            "Last Price": pr["last_price"], "52W High": pr["wk52_high"],
            "52W Low": pr["wk52_low"], "SMA50": pr["sma_50"], "SMA200": pr["sma_200"],
            "Period Return %": pr["period_return_pct"],
            "Volatility % (ann.)": pr["volatility_pct"],
            "1Y Return %": pr["ret_1y"], "3Y Return %": pr["ret_3y"],
            "5Y Return %": pr["ret_5y"],
            "Expense Ratio %": info.get("annualReportExpenseRatio"),
            "Total Assets (AUM)": info.get("totalAssets"),
            "NAV (yfinance)": info.get("navPrice") or info.get("previousClose"),
            # Need fund factsheet / AMFI data:
            "Tracking Error": Config.NA, "Tracking Difference": Config.NA,
            "Bid-Ask Spread": Config.NA,
            "Premium/Discount to NAV %": Config.NA, "Underlying Index": Config.NA,
            "Top Holdings": Config.NA, "Geographic Exposure": Config.NA,
        }
        # ETF factsheet numbers (tracking error, NAV premium, index) are not in
        # yfinance for Indian schemes - an extras upload fills them when supplied.
        self._apply_extras(h, m, info)
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = (("Tracking error, premium to NAV and holdings need "
                                      "the fund factsheet; yfinance has none for Indian "
                                      "ETFs. " if h.yf_ticker else "")
                                     + self._source_note(h, info))
        reasons = ["ETF is a diversified instrument; no arbitrary price target applied."]
        if isinstance(m["Expense Ratio %"], (int, float)):
            er = m["Expense Ratio %"]
            reasons.append(f"Expense ratio {er*100:.2f}%" if er < 1 else
                           f"Expense ratio {er:.2f}%")
        if pr["last_price"] and pr["sma_200"] and pr["last_price"] >= pr["sma_200"]:
            reasons.append("Trading above long-term average - accumulation can continue")
        else:
            reasons.append("Below long-term average - consider staggered accumulation")
        h.assessment = "ACCUMULATE" if "above long-term average" in " ".join(reasons) else "HOLD"
        h.assessment_reasons = reasons
        h.analyzed = True


class MutualFundEngine(BaseEngine):
    def analyze(self, h: Holding) -> None:
        # yfinance does not carry Indian MF scheme performance; be honest.
        m = {
            "Scheme Category": h.sub_category,
            "AMC": Config.NA, "Plan (Direct/Regular)": Config.NA,
            "Option (Growth/IDCW)": Config.NA, "Benchmark": Config.NA,
            "Fund Manager": Config.NA, "Expense Ratio %": Config.NA,
            "AUM": Config.NA,
            "1Y Return %": h.pnl_pct if h.invested and h.invested > 0 and h.pnl_pct is not None else Config.NA,
            "3Y Return %": Config.NA, "5Y Return %": Config.NA,
            "Rolling Returns": Config.NA, "Standard Deviation": Config.NA,
            "Sharpe Ratio": Config.NA, "Sortino Ratio": Config.NA,
            "Max Drawdown %": Config.NA, "Turnover Ratio": Config.NA,
            "Top Holdings": Config.NA, "Concentration": Config.NA,
            "Overlap": Config.NA, "Style Drift": Config.NA,
            "Exit Load": Config.NA,
        }
        if isinstance(h.sub_category, str) and ("Debt" in h.sub_category or "Gilt" in h.sub_category
                                                or "Liquid" in h.sub_category):
            m["Credit Quality"] = Config.NA
            m["Modified Duration"] = Config.NA
            m["Yield to Maturity %"] = Config.NA
            m["Interest-rate Risk"] = Config.NA
        # No free NAV/factsheet feed here: the extras upload is the only way fund
        # metrics (Sharpe, expense ratio, holdings...) enter the report.
        self._apply_extras(h, m, {})
        h.metrics = {k: (v if v not in (None, "") else Config.NA) for k, v in m.items()}
        h.metrics["_data_source"] = (("No reliable free source in this environment for "
                                      "Indian MF factsheet data (AMFI/AMC); only the "
                                      "user-supplied P/L is shown. ")
                                     + self._source_note(h, {}))
        h.assessment = "REVIEW"
        h.assessment_reasons = [
            "Fund-level metrics (Sharpe, rolling returns, benchmark relative, expense "
            "ratio, holdings) are unavailable from the current data source (marked N/A).",
            "No accumulation/HOLD verdict is forced without benchmark & risk data.",
            "Recommended action: review once AMFI/AMC factsheet data is connected.",
        ]
        h.analyzed = True


class AnalysisOrchestrator:
    def __init__(self, collector: DataCollector):
        self.collector = collector
        self.stock = IndividualStockEngine(collector)
        self.bank = BankNBFCEngine(collector)
        self.ins = InsuranceEngine(collector)
        self.etf = ETFEngine(collector)
        self.mf = MutualFundEngine(collector)

    def analyze(self, h: Holding) -> None:
        try:
            if h.asset_class == CLASS_STOCKS:
                self.stock.analyze(h)
            elif h.asset_class == CLASS_BANKS_FI:
                # Insurance is a sub-type of financials with its own engine.
                if str(h.sub_category).lower().startswith("insurance"):
                    self.ins.analyze(h)
                else:
                    self.bank.analyze(h)
            elif h.asset_class == CLASS_ETF:
                self.etf.analyze(h)
            elif h.asset_class == CLASS_MF:
                self.mf.analyze(h)
            else:
                h.assessment = Config.NA
                h.assessment_reasons = ["Unidentified asset - not analyzed. "
                                        "Please verify symbol/ISIN."]
                h.analyzed = False
        except Exception as e:
            LOG.warning("Analysis failed for %s: %s", h.name, e)
            h.issues.append(f"Analysis error: {e}")
            h.assessment = Config.NA
            h.analyzed = False


# -----------------------------------------------------------------------------
# 10. PORTFOLIO-LEVEL CALCULATIONS & RISK ANALYSIS
# -----------------------------------------------------------------------------
@dataclass
class PortfolioSummary:
    total_invested: float = 0.0
    total_current_value: float = 0.0
    unrealized_pnl: float = 0.0
    unrealized_return_pct: Optional[float] = None
    n_holdings: int = 0
    n_identified: int = 0
    n_analyzed: int = 0
    n_failed_lookups: int = 0
    asset_allocation: Dict[str, float] = field(default_factory=dict)
    sector_allocation: Dict[str, float] = field(default_factory=dict)
    financial_sector_exposure_pct: float = 0.0
    domestic_pct: float = 0.0
    international_pct: float = 0.0
    top_holdings: List[Tuple[str, float, float]] = field(default_factory=list)
    concentration_top5_pct: float = 0.0
    duplicate_exposures: List[str] = field(default_factory=list)
    dividend_income: Optional[float] = None
    valuation_notes: List[str] = field(default_factory=list)

def effective_value(h: Holding) -> Tuple[float, bool]:
    """
    Return (current_value, used_invested_fallback).
    Never invents a market price: if no current value / cmp is known we fall back
    to the invested amount (flat assumption) and flag it.
    """
    if h.current_value is not None:
        return h.current_value, False
    if h.cmp is not None and h.quantity is not None:
        return h.cmp * h.quantity, False
    if h.invested is not None:
        return h.invested, True
    return 0.0, True

def compute_portfolio(holdings: List[Holding]) -> PortfolioSummary:
    ps = PortfolioSummary()
    ps.n_holdings = len(holdings)
    total_inv = 0.0
    total_mv = 0.0
    alloc_by_class: Dict[str, float] = {}
    alloc_by_sector: Dict[str, float] = {}
    per_holding_mv: List[Tuple[str, float]] = []
    seen_symbols: Dict[str, int] = {}
    fallback_count = 0
    div_income = 0.0
    div_known = False

    for h in holdings:
        mv, used_fallback = effective_value(h)
        if used_fallback and h.current_value is None:
            fallback_count += 1
            h.issues.append("Current market value unavailable; used invested amount "
                            "as a flat (0% return) proxy in portfolio totals.")
        inv = h.invested if h.invested is not None else 0.0
        total_inv += inv
        total_mv += mv
        alloc_by_class[h.asset_class] = alloc_by_class.get(h.asset_class, 0.0) + mv
        if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI) and h.sector != Config.NA:
            alloc_by_sector[h.sector] = alloc_by_sector.get(h.sector, 0.0) + mv
        per_holding_mv.append((h.name or h.symbol, mv))
        key = (h.symbol or h.name or "").upper().replace(" ", "")
        if key:
            seen_symbols[key] = seen_symbols.get(key, 0) + 1
        # Dividend income (only when we actually have a yield + value).
        dy = h.metrics.get("Dividend Yield %") if h.metrics else None
        if isinstance(dy, (int, float)) and mv:
            div_income += mv * dy / 100.0
            div_known = True
        if h.asset_class != CLASS_UNKNOWN:
            ps.n_identified += 1
        if h.analyzed:
            ps.n_analyzed += 1

    ps.total_invested = total_inv
    ps.total_current_value = total_mv
    ps.unrealized_pnl = total_mv - total_inv
    r = safe_div(ps.unrealized_pnl, total_inv)
    ps.unrealized_return_pct = (r * 100) if r is not None else None

    # Allocation percentages (guard denominator).
    for k, v in alloc_by_class.items():
        pct = safe_div(v, total_mv)
        ps.asset_allocation[k] = round(pct * 100, 2) if pct is not None else 0.0
    for k, v in alloc_by_sector.items():
        pct = safe_div(v, total_mv)
        ps.sector_allocation[k] = round(pct * 100, 2) if pct is not None else 0.0

    # Financial-sector exposure.
    fin_mv = sum(v for k, v in alloc_by_class.items() if k == CLASS_BANKS_FI)
    fin_mv += sum(v for k, v in alloc_by_sector.items() if k == "Banking & NBFC")
    p = safe_div(fin_mv, total_mv)
    ps.financial_sector_exposure_pct = round(p * 100, 2) if p is not None else 0.0

    # Domestic vs international (based on classification sub-flavours).
    intl_mv = 0.0
    for h in holdings:
        mv, _ = effective_value(h)
        sub = str(h.sub_category).lower()
        if any(t in sub for t in ["international", "us", "global"]) or \
           (h.currency and h.currency != "INR"):
            intl_mv += mv
    p_intl = safe_div(intl_mv, total_mv)
    ps.international_pct = round(p_intl * 100, 2) if p_intl is not None else 0.0
    ps.domestic_pct = round(100 - ps.international_pct, 2)

    # Top holdings & concentration.
    per_holding_mv.sort(key=lambda x: x[1], reverse=True)
    top5_mv = 0.0
    for i, (nm, mv) in enumerate(per_holding_mv):
        pct = safe_div(mv, total_mv)
        ps.top_holdings.append((nm, mv, round(pct * 100, 2) if pct else 0.0))
        if i < 5:
            top5_mv += mv
    p_top5 = safe_div(top5_mv, total_mv)
    ps.concentration_top5_pct = round(p_top5 * 100, 2) if p_top5 is not None else 0.0

    # Duplicate exposure (same symbol appearing multiple times).
    ps.duplicate_exposures = [k for k, c in seen_symbols.items() if c > 1]

    ps.dividend_income = round(div_income, 2) if div_known else None

    if fallback_count:
        ps.valuation_notes.append(
            f"{fallback_count} holding(s) had no current market value; invested "
            f"amount used as proxy (totals are conservative, not live valuations).")
    ps.n_failed_lookups = len(set(_COLLECTOR.failures)) if _COLLECTOR else 0
    return ps


# -----------------------------------------------------------------------------
# 11. CHART GENERATION
# -----------------------------------------------------------------------------
def generate_charts(holdings: List[Holding], ps: PortfolioSummary,
                    out_dir: str) -> List[str]:
    """Create PNG charts, returning file paths. Never crashes report generation."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    charts: List[str] = []

    def pie(data: Dict[str, float], title: str, fname: str):
        data = {k: v for k, v in data.items() if v and v > 0}
        if not data:
            return
        try:
            fig, ax = plt.subplots(figsize=(7, 5))
            labels = list(data.keys())
            sizes = list(data.values())
            ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90,
                   textprops={"fontsize": 8})
            ax.set_title(title)
            path = os.path.join(out_dir, fname)
            fig.tight_layout()
            fig.savefig(path, dpi=130)
            plt.close(fig)
            charts.append(path)
        except Exception as e:
            LOG.warning("Chart '%s' failed: %s", title, e)

    def bar(items: List[Tuple[str, float]], title: str, fname: str, color="#2a6f97"):
        items = [it for it in items if it[1] and it[1] > 0][:12]
        if not items:
            return
        try:
            fig, ax = plt.subplots(figsize=(8, 5))
            names = [i[0] for i in items]
            vals = [i[1] for i in items]
            ax.barh(names[::-1], vals[::-1], color=color)
            ax.set_title(title)
            ax.set_xlabel("Value (" + Config.CURRENCY_SYMBOL + ")")
            ax.tick_params(axis="y", labelsize=8)
            path = os.path.join(out_dir, fname)
            fig.tight_layout()
            fig.savefig(path, dpi=130)
            plt.close(fig)
            charts.append(path)
        except Exception as e:
            LOG.warning("Chart '%s' failed: %s", title, e)

    pie(ps.asset_allocation, "Asset-Class Allocation (%)", "chart_asset_allocation.png")
    pie(ps.sector_allocation, "Sector Allocation (Stocks & Financials)",
        "chart_sector_allocation.png")
    bar(ps.top_holdings, "Top Holdings by Market Value", "chart_top_holdings.png")

    # Per-holding historical price charts (limited to those with data).
    hist_dir = os.path.join(out_dir, "price_charts")
    os.makedirs(hist_dir, exist_ok=True)
    made = 0
    for h in holdings:
        if made >= 20:
            break
        hist = _COLLECTOR.cache.get(h.yf_ticker) if _COLLECTOR else None
        if hist is None or len(hist) < 20:
            continue
        try:
            fig, ax = plt.subplots(figsize=(8, 4))
            closes = hist["Close"].astype(float).values
            ax.plot(range(len(closes)), closes, linewidth=1.2, color="#1b4965")
            ax.set_title(f"{h.name or h.symbol} - price history (delayed/historical)")
            ax.set_ylabel("Price (" + Config.CURRENCY_SYMBOL + ")")
            ax.set_xlabel("trading days ->")
            path = os.path.join(hist_dir, f"price_{re.sub(r'[^A-Za-z0-9]','_', (h.symbol or h.name or 'h'))}.png")
            fig.tight_layout()
            fig.savefig(path, dpi=120)
            plt.close(fig)
            charts.append(path)
            _CHARTS_INDEX.setdefault(h.symbol or h.name, []).append(path)
            made += 1
        except Exception as e:
            LOG.debug("Price chart failed for %s: %s", h.name, e)

    # Risk / return scatter (volatility vs period return).
    try:
        pts = []
        for h in holdings:
            vol = h.metrics.get("Volatility % (ann.)")
            ret = h.metrics.get("Period Return %")
            if isinstance(vol, (int, float)) and isinstance(ret, (int, float)):
                pts.append((h.name or h.symbol, vol, ret))
        if pts:
            fig, ax = plt.subplots(figsize=(7.5, 5.5))
            for nm, v, r in pts:
                ax.scatter(v, r, s=45, alpha=0.8)
                ax.annotate(nm[:14], (v, r), fontsize=7,
                            xytext=(3, 3), textcoords="offset points")
            ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
            ax.set_xlabel("Annualised Volatility % (risk)")
            ax.set_ylabel("Historical Period Return %")
            ax.set_title("Risk vs Return (where data available)")
            path = os.path.join(out_dir, "chart_risk_return.png")
            fig.tight_layout()
            fig.savefig(path, dpi=130)
            plt.close(fig)
            charts.append(path)
    except Exception as e:
        LOG.warning("Risk/return chart failed: %s", e)

    LOG.info("Generated %d chart image(s).", len(charts))
    return charts


# Module-level handle used by chart/portfolio helpers (set in main()).
_COLLECTOR: Optional[DataCollector] = None


# -----------------------------------------------------------------------------
# 12. PROFESSIONAL PDF REPORT GENERATION  (ReportLab)
# -----------------------------------------------------------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph,
                                Spacer, Table, TableStyle, Image, PageBreak)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

PDF_FONT = "Helvetica"
PDF_FONT_BOLD = "Helvetica-Bold"
RUPEE_OK = False          # base Helvetica has no ₹ glyph
_FONT_RESOLVED = False

def _resolve_fonts() -> None:
    """
    Use a Unicode TrueType font (DejaVu Sans, shipped with matplotlib) so the ₹
    symbol and other glyphs render. Falls back to Helvetica + 'Rs.' text.
    """
    global PDF_FONT, PDF_FONT_BOLD, RUPEE_OK, _FONT_RESOLVED
    if _FONT_RESOLVED:
        return
    _FONT_RESOLVED = True
    try:
        from matplotlib import font_manager as fm
        regular = bold = None
        try:
            for p in fm.findSystemFonts():
                n = os.path.basename(p).lower()
                if not n.endswith(".ttf"):
                    continue
                if n in ("dejavusans.ttf", "notosansregular.ttf", "latha.ttf"):
                    regular = regular or p
                if n in ("dejavusans-bold.ttf", "notosansbold.ttf"):
                    bold = p
        except Exception:
            pass
        if not regular:
            cand = fm.findfont("DejaVu Sans")
            regular = cand if os.path.exists(cand) else None
        if not regular:
            return
        pdfmetrics.registerFont(TTFont("UnicodeSans", regular))
        PDF_FONT = "UnicodeSans"
        if bold and os.path.exists(bold):
            pdfmetrics.registerFont(TTFont("UnicodeSans-Bold", bold))
            PDF_FONT_BOLD = "UnicodeSans-Bold"
            pdfmetrics.registerFontFamily(
                "UnicodeSans", normal="UnicodeSans", bold="UnicodeSans-Bold",
                italic="UnicodeSans", boldItalic="UnicodeSans-Bold")
        RUPEE_OK = True
    except Exception as e:
        LOG.debug("Unicode PDF font unavailable, using Rs. fallback: %s", e)

import re as _re_amp
def _fix_amp(text: Any) -> str:
    """Escape lone '&' and stray '<' so ReportLab's mini-XML parser does not choke
    on real company names / rule texts (e.g. 'ARE&M', '< 1.5'), while preserving
    intended entities (&#8226;, &#9744;) and tags (<b>, <br/>)."""
    s = str(text)
    s = _re_amp.sub(r"&(?!(?:#\d+;|\w+;))", "&amp;", s)
    s = _re_amp.sub(r"<(?![a-zA-Z/!])", "&lt;", s)
    if not RUPEE_OK and "\u20b9" in s:
        s = s.replace("\u20b9", "Rs.")   # no Unicode font -> avoid blank boxes
    return s

def _plain(text: Any) -> str:
    """Canvas (non-XML) text: only fix the rupee glyph, never escape markup."""
    s = str(text)
    if not RUPEE_OK and "\u20b9" in s:
        s = s.replace("\u20b9", "Rs.")
    return s

class NumberedCanvas:
    """Mixin-free page numbering via a custom DocTemplate."""

class ReportDoc(BaseDocTemplate):
    def __init__(self, filename, title, **kw):
        super().__init__(filename, pagesize=A4,
                         leftMargin=16*mm, rightMargin=16*mm,
                         topMargin=18*mm, bottomMargin=18*mm, title=title, **kw)
        self.doc_title = title
        frame = Frame(self.leftMargin, self.bottomMargin,
                      self.width, self.height, id="main")
        self.addPageTemplates([PageTemplate(id="t", frames=[frame],
                             onPage=self._decorate)])

    def _decorate(self, canvas, doc):
        canvas.saveState()
        # Header
        canvas.setFont(PDF_FONT_BOLD, 9)
        canvas.setFillColor(colors.HexColor("#1b4965"))
        canvas.drawString(doc.leftMargin, A4[1] - 12*mm, "Portfolio Analyzer")
        canvas.setFont(PDF_FONT, 8)
        canvas.setFillColor(colors.grey)
        canvas.drawRightString(A4[0] - doc.rightMargin, A4[1] - 12*mm,
                               _plain(self.doc_title))
        canvas.setStrokeColor(colors.HexColor("#cfd8dc"))
        canvas.line(doc.leftMargin, A4[1] - 14*mm, A4[0] - doc.rightMargin, A4[1] - 14*mm)
        # Footer with page number
        canvas.setFont(PDF_FONT, 8)
        canvas.setFillColor(colors.grey)
        canvas.drawString(doc.leftMargin, 10*mm,
                          _plain(f"Generated {Config.GENERATED_ON} | Not investment advice"))
        canvas.drawRightString(A4[0] - doc.rightMargin, 10*mm, f"Page {doc.page}")
        canvas.restoreState()

class PDFBuilder:
    BRAND = colors.HexColor("#1b4965")
    LIGHT = colors.HexColor("#e8f1f5")

    def __init__(self, title: str, fname: str, out_dir: str):
        _resolve_fonts()
        self.font, self.font_bold = PDF_FONT, PDF_FONT_BOLD
        self.styles = getSampleStyleSheet()
        self.h1 = ParagraphStyle("h1", parent=self.styles["Heading1"],
                                 fontName=self.font_bold, textColor=self.BRAND,
                                 fontSize=16, spaceAfter=6)
        self.h2 = ParagraphStyle("h2", parent=self.styles["Heading2"],
                                 fontName=self.font_bold, textColor=self.BRAND,
                                 fontSize=12, spaceBefore=8, spaceAfter=4)
        self.body = ParagraphStyle("body", parent=self.styles["BodyText"],
                                   fontName=self.font, fontSize=9, leading=12,
                                   alignment=TA_LEFT)
        self.small = ParagraphStyle("small", parent=self.styles["BodyText"],
                                    fontName=self.font, fontSize=7.5, leading=9.5,
                                    textColor=colors.grey)
        self.title_txt = title
        self.path = os.path.join(out_dir, fname)
        self.story: List[Any] = []

    def title_block(self, subtitle=""):
        self.story.append(Paragraph(_fix_amp(self.title_txt), self.h1))
        if subtitle:
            self.story.append(Paragraph(_fix_amp(subtitle), self.body))
        self.story.append(Paragraph(f"Generated: {Config.GENERATED_ON}", self.small))
        self.story.append(Spacer(1, 6))

    def para(self, text): self.story.append(Paragraph(_fix_amp(text), self.body))
    def heading(self, text): self.story.append(Paragraph(_fix_amp(text), self.h2))
    def spacer(self, h=6): self.story.append(Spacer(1, h))
    def pagebreak(self): self.story.append(PageBreak())

    def table(self, rows: List[List[Any]], col_widths: Optional[List[float]] = None,
              font_size: float = 7.5):
        if not rows:
            return
        # Wrap cells as Paragraphs so long text wraps within page width.
        def cell(v, style):
            return Paragraph(_fix_amp(v), style)
        hdr_style = ParagraphStyle("th", parent=self.body, fontSize=font_size,
                                   leading=font_size+2, textColor=colors.white,
                                   fontName=self.font_bold)
        td_style = ParagraphStyle("td", parent=self.body, fontSize=font_size,
                                  leading=font_size+2, fontName=self.font)
        wrapped = []
        for ri, row in enumerate(rows):
            st = hdr_style if ri == 0 else td_style
            wrapped.append([cell(c, st) for c in row])
        avail = A4[0] - 32*mm
        if col_widths is None:
            ncol = len(rows[0])
            col_widths = [avail / ncol] * ncol
        else:
            total = sum(col_widths)
            col_widths = [w / total * avail for w in col_widths]
        t = Table(wrapped, colWidths=col_widths, repeatRows=1, hAlign="LEFT")
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), self.BRAND),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, self.LIGHT]),
            ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#b0bec5")),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3),
            ("TOPPADDING", (0, 0), (-1, -1), 2),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ]))
        self.story.append(t)
        self.spacer(8)

    def image(self, path: str, max_w_mm=170, max_h_mm=95):
        if not path or not os.path.exists(path):
            return
        try:
            from PIL import Image as PILImage
            with PILImage.open(path) as im:
                w, h = im.size
            scale = min((max_w_mm*mm)/w, (max_h_mm*mm)/h)
            self.story.append(Image(path, width=w*scale, height=h*scale))
            self.spacer(8)
        except Exception as e:
            LOG.debug("Image embed failed %s: %s", path, e)

    def disclaimer(self):
        self.spacer(6)
        self.story.append(Paragraph(
            "DISCLAIMER: This report is an automated, data-driven analysis for "
            "research and educational purposes only. It is NOT investment advice, "
            "and NOT a recommendation to buy/sell. Data may be delayed, historical "
            "or incomplete; unavailable metrics are shown as N/A. Verify against "
            "authoritative sources before acting.", self.small))

    def build(self):
        doc = ReportDoc(self.path, self.title_txt)
        doc.build(self.story)
        return self.path


def _fmt_metric(v: Any) -> str:
    if v is None:
        return Config.NA
    if isinstance(v, float):
        if abs(v) >= 1e7:
            return fmt_inr(v)
        return f"{v:,.2f}"
    if isinstance(v, int):
        return f"{v:,}"
    return str(v)

def generate_master_pdf(holdings: List[Holding], ps: PortfolioSummary,
                        charts: List[str], out_dir: str) -> List[str]:
    """
    Produce ONE comprehensive PDF: portfolio summary + risk + a full A-to-Z
    analysis section for every single holding (rule tables, all metrics, chart,
    decision table and monitoring). Returns [path], or [] on failure.
    """
    os.makedirs(out_dir, exist_ok=True)
    try:
        b = PDFBuilder("Consolidated Portfolio Analysis Report",
                       "Portfolio_Analysis_Report.pdf", out_dir)
        b.title_block(
            f"Single-file detailed report | {ps.n_holdings} holding(s) | "
            f"{ps.n_identified} identified | {ps.n_analyzed} analysed | Indian market")
        _exec_section(b, holdings, ps, charts)
        _risk_section(b, holdings, ps)
        _all_holdings_register(b, holdings, ps)

        order = [CLASS_STOCKS, CLASS_BANKS_FI, CLASS_ETF, CLASS_MF, CLASS_UNKNOWN]
        grouped: List[Holding] = []
        for cls in order:
            grouped += sorted([h for h in holdings if h.asset_class == cls],
                              key=lambda x: (x.current_value or x.invested or 0),
                              reverse=True)
        grouped += [h for h in holdings if h.asset_class not in order]
        b.pagebreak()
        b.heading("Part II - Detailed analysis, holding by holding")
        b.para("Each holding gets its own section: position data, the rule-based "
               "scorecard (Steps 1-4) where fundamentals are available, every metric "
               "collected (including N/A items), the decision table, what would change "
               "the signal, and a monitoring checklist. Sections are ordered by asset "
               "class, then by market value.")
        for idx, h in enumerate(grouped, 1):
            try:
                _holding_section(b, h, ps, idx)
            except Exception as e:
                LOG.warning("Detail section failed for %s: %s", h.name, e)
                b.pagebreak()
                b.heading(f"Holding {idx}: {_short(h.name or h.symbol)}")
                b.para(f"This section could not be rendered: {e}")
        path = b.build()
        LOG.info("Generated 1 comprehensive PDF with %d holding section(s).", len(grouped))
        return [path]
    except Exception as e:
        LOG.error("Master PDF generation failed: %s", e)
        return []

def _slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")[:40] or "holding"

def _short(s: str) -> str:
    s = str(s)
    return s if len(s) <= 22 else s[:20] + ".."

def _all_holdings_register(b: PDFBuilder, holdings: List[Holding], ps: PortfolioSummary):
    """One register listing EVERY uploaded row so nothing is unaccounted for."""
    b.pagebreak()
    b.heading("Part I (cont.) - Register of all uploaded holdings")
    rows = [["#", "Symbol", "Name", "Class", "Sector / Category", "Qty",
             "Invested", "Value", "P/L %", "Weight %", "Assessment", "Conf."]]
    for i, h in enumerate(holdings, 1):
        w = safe_div(h.current_value, ps.total_current_value)
        rows.append([i, h.symbol or Config.NA, _short(h.name), _short(h.asset_class),
                     _short(h.sector if h.sector != Config.NA else h.sub_category),
                     _fmt_metric(h.quantity), fmt_inr(h.invested),
                     fmt_inr(h.current_value),
                     (f"{h.pnl_pct:.1f}" if h.pnl_pct is not None else Config.NA),
                     (f"{w*100:.1f}" if w is not None else Config.NA),
                     h.assessment, h.confidence])
    b.table(rows, col_widths=[3, 11, 16, 11, 14, 8, 11, 11, 6, 7, 9, 6], font_size=6.5)
    b.para("Every row from the uploaded file appears here. Holdings that could not be "
           "identified are listed under 'Unidentified' with their data issues instead "
           "of being guessed.")

_STEP_NOTE = {
    "Step 1: Eliminating low quality and risk":
        "Filters weed out risky companies before spending time on analysis. "
        "Debt/equity and interest coverage are NOT applicable to banks and NBFCs "
        "(deposits and bonds are their raw material). Promoter holding and pledging "
        "come from BSE/NSE shareholding filings, which the free data source does not "
        "provide - those rows stay N/A.",
    "Step 2: Is it profitable?":
        "ROCE/ROE averages and margins are COMPUTED from the last available annual "
        "statements (yfinance). Check ROE together with ROCE: high ROE on low ROCE "
        "means the return is running on debt. Cash-flow margin and free cash flow show "
        "whether profits are real cash.",
    "Step 3: Is it growing?":
        "CAGRs use every annual period the data source supplies (usually 4-5 years). "
        "The profit triangle needs revenue, profit AND cash flow to rise together; if "
        "profit rises without cash flow it is only paper profit.",
    "Step 4: Is the price right?":
        "Valuation last, and only last. Sector P/E and P/B comparisons need a sector "
        "multiple source (Screener/Sharpely) that is not in the free feed, so those "
        "rows are marked WATCH with the stock's own multiple shown - compare it "
        "against peers before acting.",
}

def _change_conditions(rules: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    """Concrete, data-derived conditions that would flip the signal either way."""
    upgrade: List[str] = []
    for _, rows in rules["steps"]:
        for r in rows:
            if r["signal"] == SIG_FAIL:
                upgrade.append(f"{r['check']}: {r['rule']} (now {r['value']})")
    upgrade.append("Quarterly results confirm margins/profit recovering while "
                   "quality checks stay intact")
    downgrade = [
        "Any Step 1 check turning FAIL (leverage up, coverage down, pledge appearing)",
        "Promoter pledging above 15% or continuous promoter selling in filings",
        "ROCE below 15% or interest coverage below 3",
        "Revenue declining two consecutive quarters",
        "More than 2 red flags on a Screener/Tickertape screen",
    ]
    return upgrade[:8], downgrade

def _instrument_rule_rows(h: Holding, m: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Rule rows for instruments where the 6-step equity rules do NOT apply
    (ETFs, mutual funds, unclassified rows). Honest N/A for anything the free
    data cannot supply.
    """
    def num(v):
        return v if isinstance(v, (int, float)) else None
    rows: List[Dict[str, str]] = []
    if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI):
        # Equity/bank holding whose statements could not be fetched (offline run,
        # unresolved ticker or failed lookup): report it without inventing a score.
        rows.append(_rule("Row accounted for", "Every uploaded row is reported",
                          f"Row {h.source_row}: {h.raw_name or h.symbol or '(blank)'}",
                          SIG_PASS))
        rows.append(_rule("Market data & annual statements",
                          "Needed to score Steps 1-4 of the rule card",
                          "N/A - no data returned for this symbol (lookup failed, "
                          "unresolved ticker or offline run). Re-run with internet "
                          "access; nothing was guessed.", SIG_NA))
    elif h.asset_class == CLASS_ETF:
        px, sma = num(m.get("Last Price")), num(m.get("SMA200"))
        rows.append(_rule("Trend vs 200-DMA", "At/above long-term average",
                          (f"{_fmt_metric(px)} vs SMA200 {_fmt_metric(sma)}"),
                          _sig(bool(px and sma and px >= sma), not (px and sma))))
        r1, r3, r5 = num(m.get("1Y Return %")), num(m.get("3Y Return %")), num(m.get("5Y Return %"))
        have = [v for v in (r1, r3, r5) if v is not None]
        rows.append(_rule("1Y / 3Y / 5Y returns", "Positive over the holding horizon",
                          f"1Y {_pc(r1)}, 3Y {_pc(r3)}, 5Y {_pc(r5)}",
                          _sig(bool(have) and all(v > 0 for v in have), not have)))
        vol = num(m.get("Volatility % (ann.)"))
        rows.append(_rule("Volatility", "Roughly matches an index instrument",
                          _pc(vol), SIG_WATCH if vol else SIG_NA))
        er = num(m.get("Expense Ratio %"))
        rows.append(_rule("Expense ratio", "Lower is better for an ETF",
                          _pc(er, 2) if er is not None else Config.NA,
                          SIG_NA if er is None else SIG_PASS))
        aum = num(m.get("Total Assets (AUM)"))
        rows.append(_rule("AUM / liquidity", "Large enough to trade tight",
                          _cr(aum / 1e7 if aum else None),
                          SIG_NA if aum is None else SIG_PASS))
        for k, lbl in (("Tracking Error", "vs the index it copies"),
                       ("Premium/Discount to NAV %", "Trade price vs NAV"),
                       ("Underlying Index", "Know exactly what you own"),
                       ("Top Holdings", "Overlap with direct stock holdings")):
            rows.append(_rule(k, lbl, "N/A (fund factsheet not in the free data)", SIG_NA))
    elif h.asset_class == CLASS_MF:
        rows.append(_rule("Position recorded", "Units & invested value present",
                          f"{Config.CURRENCY_SYMBOL}{_fmt_metric(h.quantity)} units at avg "
                          f"{_fmt_metric(h.avg_price)}",
                          SIG_PASS if h.quantity else SIG_FAIL))
        rows.append(_rule("Broker valuation", "Current value / P&L from the file",
                          f"{fmt_inr(h.current_value)} ({_pc(h.pnl_pct)})",
                          SIG_PASS if h.current_value else SIG_NA))
        rows.append(_rule("Live NAV", "Independent NAV check",
                          "N/A (no free NAV feed for Indian schemes here)", SIG_NA))
        for k, lbl in (("1Y/3Y/5Y & rolling returns", "Scheme performance vs benchmark"),
                       ("Expense Ratio %", "Direct cheaper than regular"),
                       ("AUM / fund manager / AMC", "Size, stability, manager tenure"),
                       ("Sharpe / Std dev / Max drawdown", "Risk-adjusted behaviour"),
                       ("Top holdings / overlap", "Concentration and duplication"),
                       ("Exit load", "Cost of exiting early")):
            rows.append(_rule(k, lbl, "N/A (AMFI/AMC factsheet required)", SIG_NA))
    else:
        rows.append(_rule("Row accounted for", "Every uploaded row is reported",
                          f"Row {h.source_row}: {h.raw_name or h.symbol or '(blank)'}",
                          SIG_PASS))
        rows.append(_rule("Identification", "Symbol/ISIN resolvable to a real instrument",
                          ("; ".join(h.issues) or "Not matched by name, ISIN or symbol"),
                          SIG_FAIL))
        rows.append(_rule("Valuation", "Price and returns",
                          "N/A (instrument not identified - nothing was guessed)", SIG_NA))
    rows.append(_rule("Position P/L", "Review any deep unrealised loss",
                      _pc(h.pnl_pct),
                      SIG_PASS if (h.pnl_pct is None or h.pnl_pct >= -25) else SIG_WATCH))
    # Fund instruments are not 'bought more' on a rule pass - use neutral actions.
    fund_action = {SIG_PASS: "OK / CONTINUE", SIG_FAIL: "REVIEW", SIG_WATCH: "MONITOR",
                   SIG_NA: "VERIFY"}
    for r in rows:
        r["action"] = fund_action.get(r["signal"], "MONITOR")
    return rows

def _holding_section(b: PDFBuilder, h: Holding, ps: PortfolioSummary, idx: int):
    """
    One complete A-to-Z section per holding, laid out like the user's 6-step
    demo: snapshot -> rule score -> Step 1..4 tables -> decision table -> what
    would change the signal -> every collected metric -> data source and
    limitations -> chart -> monitoring checklist.
    """
    m = h.metrics
    rules = h.analysis.get("rules")
    weight = safe_div(h.current_value, ps.total_current_value)
    cat = h.sector if h.sector != Config.NA else h.sub_category
    b.pagebreak()
    b.heading(f"Holding {idx}: {h.name or h.symbol} ({h.symbol or Config.NA})")
    b.para(f"{h.asset_class} | {cat} | ISIN {h.ISIN or Config.NA} | classified via "
           f"{h.classification_source} (confidence {h.confidence}) | uploaded row "
           f"{h.source_row} | currency {h.currency}")

    def mv(key):
        return _fmt_metric(m.get(key, Config.NA))
    mc = m.get("Market Cap")
    mc_cr = (mc / 1e7) if isinstance(mc, (int, float)) else None
    b.heading("Snapshot")
    b.table([["Metric", "Value", "Metric", "Value"],
             ["Latest price", mv("Last Price"), "Market cap", _cr(mc_cr)],
             ["52-week high / low", f"{mv('52W High')} / {mv('52W Low')}",
              "P/E | P/B", f"{mv('P/E')} | {mv('P/B')}"],
             ["Dividend yield", mv("Dividend Yield %"), "SMA50 / SMA200",
              f"{mv('SMA50')} / {mv('SMA200')}"],
             ["1Y / 3Y / 5Y return",
              f"{mv('1Y Return %')} / {mv('3Y Return %')} / {mv('5Y Return %')}",
              "Volatility (ann.)", mv("Volatility % (ann.)")],
             ["Quantity / units", _fmt_metric(h.quantity),
              "Avg cost / CMP", f"{_fmt_metric(h.avg_price)} / {_fmt_metric(h.cmp)}"],
             ["Invested", fmt_inr(h.invested), "Current value", fmt_inr(h.current_value)],
             ["Unrealised P/L",
              (f"{fmt_inr(h.pnl)} ({h.pnl_pct:.1f}%)" if h.pnl_pct is not None
               else fmt_inr(h.pnl)),
              "Portfolio weight", f"{weight*100:.2f}%" if weight is not None else Config.NA],
             ], col_widths=[21, 29, 21, 29], font_size=7)

    if rules:
        b.heading("Rule-based score and overall signal")
        pct_txt = f" ({rules['ratio_pct']:.0f}%)" if rules["ratio_pct"] is not None else ""
        b.table([["Score from your rules (N/A checks are excluded)", "Overall signal"],
                 [f"{rules['pass']} pass, {rules['fail']} fail, {rules['watch']} watch, "
                  f"{rules['na']} N/A  ->  {rules['pass']} of {rules['applicable']} "
                  f"applicable checks passed{pct_txt}", rules["signal"]]],
                col_widths=[74, 26], font_size=7.5)
        n = 0
        for title, rrows in rules["steps"]:
            b.heading(title)
            b.para(_STEP_NOTE.get(title, ""))
            rows = [["#", "Check", "Your rule", "Data & note (computed here)",
                     "Signal", "Action"]]
            for r in rrows:
                n += 1
                rows.append([n, r["check"], r["rule"], r["value"], r["signal"], r["action"]])
            b.table(rows, col_widths=[4, 16, 18, 40, 8, 9], font_size=6.8)
    else:
        rows = _instrument_rule_rows(h, m)
        if rows:
            b.heading("Rule checks for this instrument")
            if h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI):
                b.para("The 6-step equity rule card could not be scored for this "
                       "holding because no market data / annual statements were "
                       "returned. The row is still fully reported - it is NOT dropped "
                       "and no number is invented for it.")
            else:
                b.para("The equity 6-step rules (ROCE, promoter holding, debt/equity, "
                       "margins) do NOT apply to this asset class - it is analysed with "
                       "its own checks, and anything the free data cannot supply stays "
                       "N/A instead of being guessed.")
            trows = [["#", "Check", "Rule / what to look for", "Data & note",
                      "Status", "Action"]]
            for i, r in enumerate(rows, 1):
                trows.append([i, r["check"], r["rule"], r["value"], r["signal"], r["action"]])
            b.table(trows, col_widths=[4, 16, 20, 38, 8, 9], font_size=6.8)

    b.heading("Final decision table")
    label = _short(h.name or h.symbol)
    if rules:
        pct = rules["ratio_pct"]
        pct_s = f"{pct:.0f}%" if pct is not None else Config.NA
        s1_note = ("all applicable Step 1 checks pass" if rules["step1_all_pass"]
                   else "Step 1 is not fully clean")
        b.table([["Rule (edit to your own style)", f"Result for {label}"],
                 ["BUY MORE if >= 85% of applicable checks pass and Step 1 is clean",
                  ("Met: " if rules["signal"] == "BUY MORE" else "Not met: ") +
                  f"{rules['pass']} of {rules['applicable']} pass ({pct_s}), {s1_note}"],
                 ["HOLD if >= 55% of applicable checks pass",
                  ("Met: " if rules["signal"] == "HOLD" else "Not met: ") +
                  f"{rules['pass']} of {rules['applicable']} pass ({pct_s})"],
                 ["SELL if < 55% pass or any Step 1 check fails",
                  ("Met: " if rules["signal"] == "SELL" else "Not met: ") +
                  f"{rules['fail']} failed check(s), {s1_note}"],
                 ["FINAL SIGNAL", rules["signal"]],
                 ["Why", "; ".join(h.assessment_reasons) or Config.NA]],
                col_widths=[48, 52], font_size=7)
        up, down = _change_conditions(rules)
        b.heading("What would change the signal")
        b.table([["Move to BUY MORE if", "Move to SELL if"],
                 ["<br/>".join(f"&#9642; {u}" for u in up),
                  "<br/>".join(f"&#9642; {d}" for d in down)]],
                col_widths=[50, 50], font_size=7)
    else:
        b.table([["Outcome", f"Result for {label}"],
                 ["Assessment from the " + str(h.asset_class) + " engine", h.assessment],
                 ["Why", "; ".join(h.assessment_reasons) or
                  "No assessment possible - the row could not be identified, so no "
                  "data was invented for it."]],
                col_widths=[40, 60], font_size=7)

    b.heading("Every metric collected (A to Z)")
    metric_rows = [["Metric", "Value", "Metric", "Value"]]
    items = [(k, v) for k, v in m.items() if not str(k).startswith("_")]
    half = (len(items) + 1) // 2
    for i in range(half):
        left = items[i]
        right = items[i + half] if i + half < len(items) else ("", "")
        metric_rows.append([left[0], _fmt_metric(left[1]), right[0], _fmt_metric(right[1])])
    if len(items):
        b.table(metric_rows, col_widths=[28, 22, 28, 22], font_size=6.8)
    else:
        b.para("No metrics available for this holding.")
    if h.extras:
        # Show every extras value, including ones that only feed the rule score
        # (sector P/E, red flags...) and are not a report metric themselves.
        erows = [["Extras you supplied (matched from your file)", "Value"]]
        for k in sorted(h.extras.keys()):
            erows.append([_extras_label(k), _fmt_metric(h.extras[k])])
        b.table(erows, col_widths=[50, 50], font_size=6.8)
    b.para(f"Data source: {m.get('_data_source', Config.NA)} | prices are "
           f"delayed/historical, not real-time.")
    if h.issues:
        b.heading("Data quality notes")
        for i in h.issues:
            b.para(f"&#8226; {i}")
    price_img = None
    for c in _CHARTS_INDEX.get(h.symbol or h.name, []):
        price_img = c
        break
    b.heading("Historical price chart")
    if price_img and os.path.exists(price_img):
        b.image(price_img)
    else:
        b.para("No price chart for this holding (no market data source matched it).")
    b.heading("Monitoring checklist")
    b.para(_monitoring_checklist(h))
    b.spacer(4)
    b.disclaimer()

def _monitoring_checklist(h: Holding) -> str:
    if h.asset_class == CLASS_MF:
        items = ["Compare vs benchmark & category on rolling 3/5Y returns",
                 "Track expense ratio, fund-manager changes, style drift",
                 "Review AUM changes and liquidity for debt schemes"]
    elif h.asset_class == CLASS_ETF:
        items = ["Monitor tracking error vs index", "Watch premium/discount to NAV & spreads",
                 "Check AUM/liquidity and overlap with direct holdings"]
    elif h.asset_class == CLASS_BANKS_FI:
        items = ["Track GNPA/NNPA, NIM, CASA, provisions & CAR in quarterly filings",
                 "Watch credit/deposit growth and slippages",
                 "Review management/ownership changes and regulatory actions"]
    else:
        items = ["Track revenue, profit, margin & ROE trends quarterly",
                 "Watch debt, cash flows, promoter pledging & governance",
                 "Reassess valuation vs earnings growth and peer multiples"]
    return "<br/>".join(f"&#9744; {i}" for i in items)

# Populated in main(): maps holding symbol/name -> its price chart path(s).
_CHARTS_INDEX: Dict[str, List[str]] = {}


def _exec_section(b: PDFBuilder, holdings, ps: PortfolioSummary, charts):
    """Part I of the single report: portfolio-level numbers and allocation."""
    b.heading("Part I - Portfolio summary")
    b.heading("Headline Numbers")
    b.table([["Metric", "Value"],
             ["Total holdings", ps.n_holdings],
             ["Identified holdings", ps.n_identified],
             ["Analyzed holdings", ps.n_analyzed],
             ["Failed data lookups", ps.n_failed_lookups],
             ["Total invested", fmt_inr(ps.total_invested)],
             ["Current market value", fmt_inr(ps.total_current_value)],
             ["Unrealised P/L", fmt_inr(ps.unrealized_pnl)],
             ["Unrealised return %", (f"{ps.unrealized_return_pct:.2f}%"
                 if ps.unrealized_return_pct is not None else Config.NA)],
             ["Dividend income (est.)", (fmt_inr(ps.dividend_income)
                 if ps.dividend_income is not None else Config.NA)],
             ], col_widths=[45, 55])
    b.heading("Allocation")
    b.image(_find_chart(charts, "chart_asset_allocation.png"))
    b.image(_find_chart(charts, "chart_sector_allocation.png"))
    b.heading("Asset-Class Allocation")
    b.table([["Asset Class", "Value", "Weight %"]] +
            [[k, fmt_inr(sum(h.current_value or h.invested or 0 for h in holdings if h.asset_class == k)),
              ps.asset_allocation.get(k, 0)] for k in ps.asset_allocation],
            col_widths=[45, 30, 25])
    b.heading("Top Holdings & Concentration")
    b.image(_find_chart(charts, "chart_top_holdings.png"))
    rows = [["#", "Holding", "Value", "Weight %"]]
    for i, (nm, mv, pct) in enumerate(ps.top_holdings[:10], 1):
        rows.append([i, nm, fmt_inr(mv), pct])
    b.table(rows, col_widths=[8, 50, 25, 17])
    b.para(f"Top-5 concentration: {ps.concentration_top5_pct:.1f}% | "
           f"Financial-sector exposure: {ps.financial_sector_exposure_pct:.1f}% | "
           f"Domestic: {ps.domestic_pct:.1f}% / International: {ps.international_pct:.1f}%")
    if ps.duplicate_exposures:
        b.para(f"Possible duplicate exposures: {', '.join(ps.duplicate_exposures)}")
    if ps.valuation_notes:
        b.heading("Valuation Notes / Limitations")
        for n in ps.valuation_notes:
            b.para(f"&#8226; {n}")
    b.heading("How this report is built (method & data honesty)")
    for t in [
        "Each asset class is analysed with its OWN engine - individual stocks, "
        "banks/NBFCs/insurance, ETFs and mutual funds never share one scoring formula.",
        "The rule card implements Steps 1-4 (20 checks) exactly as written in the "
        "sample review. Steps 5 and 6 of the notebook method are not defined yet - "
        "they are left blank instead of being invented; add them and the score expands.",
        "Equity rule scores are computed from the last available annual statements "
        "(revenue, EBITDA, net profit, equity, debt, cash flow) pulled by yfinance, "
        "plus prices/valuation multiples. Checks the free data cannot support "
        "(promoter holding, pledging, sector P/E, red flags, bank NPA/CAR, scheme "
        "factsheets) are printed as N/A and EXCLUDED from the score - never guessed.",
        "No ETF/mutual-fund underlying holdings are expanded into the stock sections, "
        "so nothing is double counted; fund-level overlap needs factsheet holdings.",
        "Prices are delayed/historical, not real-time. Every calculation shown can be "
        "reproduced from the printed inputs.",
        "This is a self-study checklist built from your own rules and publicly "
        "available numbers. It is NOT investment advice and no forced BUY/SELL "
        "target is issued; verify with BSE/NSE filings and a SEBI-registered advisor "
        "before acting.",
    ]:
        b.para(f"&#8226; {t}")
    if _COLLECTOR and _COLLECTOR.failures:
        b.heading("Failed / unverified lookups")
        for f in sorted(set(_COLLECTOR.failures))[:30]:
            b.para(f"&#8226; {f}")

def _risk_section(b: PDFBuilder, holdings, ps: PortfolioSummary):
    b.heading("Risk, concentration & diversification")
    b.heading("Concentration")
    b.table([["Risk Metric", "Value"],
             ["Top-5 holdings weight %", f"{ps.concentration_top5_pct:.1f}%"],
             ["Largest single holding %", (f"{ps.top_holdings[0][2]:.1f}%" if ps.top_holdings else Config.NA)],
             ["Financial-sector exposure %", f"{ps.financial_sector_exposure_pct:.1f}%"],
             ["International exposure %", f"{ps.international_pct:.1f}%"],
             ["Number of distinct asset classes", len([k for k,v in ps.asset_allocation.items() if v>0])],
             ], col_widths=[55, 45])
    b.para("Concentration guidance (transparent, not a rule): a single holding above "
           "~10-15% or top-5 above ~50% increases idiosyncratic risk. Review against "
           "your own risk tolerance.")
    b.heading("Duplicate / Overlapping Exposure (symbol-level)")
    if ps.duplicate_exposures:
        b.para("The following symbols appear more than once across uploaded rows. Note: "
               "underlying holdings of ETFs/MFs are NOT auto-expanded here to avoid "
               "double counting - overlap between an ETF/MF and direct stocks requires "
               "factsheet holdings data (shown as N/A when unavailable).")
        b.table([["Symbol appearing multiple times"]] +
                [[s] for s in ps.duplicate_exposures], col_widths=[100])
    else:
        b.para("No exact duplicate symbols detected among uploaded rows.")
    b.image(_find_chart(_ALL_CHARTS, "chart_risk_return.png"))
    b.heading("Data-Driven Risk Observations")
    obs = []
    vols = [h.metrics.get("Volatility % (ann.)") for h in holdings
            if isinstance(h.metrics.get("Volatility % (ann.)"), (int, float))]
    if vols:
        hi = [h for h in holdings if isinstance(h.metrics.get("Volatility % (ann.)"), (int,float))
              and h.metrics["Volatility % (ann.)"] > 35]
        if hi:
            obs.append(f"{len(hi)} holding(s) show high annualised volatility (>35%): "
                       + ", ".join(_short(h.name or h.symbol) for h in hi[:8]))
    losses = [h for h in holdings if h.pnl_pct is not None and h.pnl_pct < -25]
    if losses:
        obs.append(f"{len(losses)} holding(s) are down >25% (temporary weakness vs "
                   "fundamental change should be assessed per report).")
    below = [h for h in holdings if isinstance(h.metrics.get("SMA200"), (int,float))
             and isinstance(h.metrics.get("Last Price"), (int,float))
             and h.metrics["Last Price"] < h.metrics["SMA200"]]
    if below:
        obs.append(f"{len(below)} holding(s) trading below their 200-DMA.")
    if not obs:
        obs.append("Insufficient market data to derive volatility/trend risk flags.")
    for o in obs:
        b.para(f"&#8226; {o}")

def _find_chart(chart_list, base):
    for c in chart_list:
        if os.path.basename(c) == base:
            return c
    return None

_ALL_CHARTS: List[str] = []


# -----------------------------------------------------------------------------
# 13. CSV / EXCEL EXPORTS
# -----------------------------------------------------------------------------
def export_tabular(holdings: List[Holding], ps: PortfolioSummary,
                   out_dir: str) -> List[str]:
    files: List[str] = []
    os.makedirs(out_dir, exist_ok=True)

    # Portfolio_Summary.csv
    summary_df = pd.DataFrame([{
        "Total Holdings": ps.n_holdings, "Identified": ps.n_identified,
        "Analyzed": ps.n_analyzed, "Failed Lookups": ps.n_failed_lookups,
        "Total Invested": ps.total_invested, "Current Value": ps.total_current_value,
        "Unrealized P/L": ps.unrealized_pnl, "Return %": ps.unrealized_return_pct,
        "Top5 Concentration %": ps.concentration_top5_pct,
        "Financial Sector %": ps.financial_sector_exposure_pct,
        "Domestic %": ps.domestic_pct, "International %": ps.international_pct,
    }])
    p = os.path.join(out_dir, "Portfolio_Summary.csv"); summary_df.to_csv(p, index=False); files.append(p)

    # Holdings_Classification.csv
    cls_df = pd.DataFrame([{
        "Source Row": h.source_row, "Uploaded Name": h.raw_name, "Symbol/ISIN": h.symbol,
        "Asset Class": h.asset_class, "Sector": h.sector, "Sub-Category": h.sub_category,
        "Classification Source": h.classification_source, "Confidence": h.confidence,
        "Issues": "; ".join(h.issues),
    } for h in holdings])
    p = os.path.join(out_dir, "Holdings_Classification.csv"); cls_df.to_csv(p, index=False); files.append(p)

    # Financial_Metrics.csv
    metric_rows = []
    for h in holdings:
        row = {"Holding": h.name or h.symbol, "Asset Class": h.asset_class,
               "Assessment": h.assessment}
        for k, v in h.metrics.items():
            if k.startswith("_"):
                continue
            row[k] = v
        metric_rows.append(row)
    met_df = pd.DataFrame(metric_rows)
    p = os.path.join(out_dir, "Financial_Metrics.csv"); met_df.to_csv(p, index=False); files.append(p)

    # Portfolio_Allocation.csv
    alloc_rows = [{"Category": k, "Type": "AssetClass", "Weight %": v}
                  for k, v in ps.asset_allocation.items()]
    alloc_rows += [{"Category": k, "Type": "Sector", "Weight %": v}
                   for k, v in ps.sector_allocation.items()]
    al_df = pd.DataFrame(alloc_rows)
    p = os.path.join(out_dir, "Portfolio_Allocation.csv"); al_df.to_csv(p, index=False); files.append(p)

    # Analysis_Results.xlsx (multi-sheet)
    try:
        xlsx = os.path.join(out_dir, "Analysis_Results.xlsx")
        with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
            summary_df.to_excel(xw, sheet_name="Summary", index=False)
            cls_df.to_excel(xw, sheet_name="Classification", index=False)
            met_df.to_excel(xw, sheet_name="Metrics", index=False)
            al_df.to_excel(xw, sheet_name="Allocation", index=False)
            pd.DataFrame([{"Holding": n, "Value": v, "Weight%": w}
                          for n, v, w in ps.top_holdings]).to_excel(
                xw, sheet_name="TopHoldings", index=False)
        files.append(xlsx)
    except Exception as e:
        LOG.warning("Excel export failed (%s); CSV exports still available.", e)
    return files


# -----------------------------------------------------------------------------
# 14. MAIN ORCHESTRATION
# -----------------------------------------------------------------------------
def _process_holdings(file: UploadedFile) -> List[Holding]:
    tables = extract_tables(file)
    holdings: List[Holding] = []
    any_mapped = False
    for table in tables:
        for n in table.notes:
            LOG.info("Extraction note: %s", n)
        clean, mapping = map_columns(table.dataframe, has_header=table.has_header)
        if not mapping:
            continue  # sheet without recognizable holdings columns (e.g. notes)
        any_mapped = True
        rows = build_holdings(clean, mapping, table.source_format)
        LOG.info("Sheet %s: mapped %d column(s), extracted %d holding row(s).",
                 table.notes[0] if table.notes else "?", len(mapping), len(rows))
        holdings.extend(rows)
    if not any_mapped:
        raise ValueError("Could not map any recognizable columns in the file.")
    if not holdings:
        raise ValueError(
            "Columns were recognized, but no data rows were found under them. "
            "Check that the sheet actually lists holdings below its header row.")
    holdings = _dedupe_holdings(holdings)
    for h in holdings:
        classify_holding(h)
    return holdings

def _dedupe_holdings(holdings: List[Holding]) -> List[Holding]:
    """
    Broker workbooks often contain an 'Equity' sheet, a 'Mutual Funds' sheet and
    a 'Combined' sheet that repeats both. De-duplicate by ISIN (falling back to
    symbol/name) keeping the first occurrence so nothing is double counted.
    """
    seen: set = set()
    out: List[Holding] = []
    dupes = 0
    for h in holdings:
        key = ("ISIN", h.ISIN) if h.ISIN else ("SYM", (h.symbol or h.name or "").upper())
        if key in seen:
            dupes += 1
            continue
        seen.add(key)
        out.append(h)
    if dupes:
        LOG.info("Removed %d duplicate row(s) across worksheets (matched by ISIN/symbol).",
                 dupes)
    return out

def run_analysis(file: UploadedFile,
                 extras_file: Optional[UploadedFile] = None) -> Dict[str, Any]:
    global _COLLECTOR, _ALL_CHARTS
    print("\n" + "=" * 70)
    print(f" Processing: {file.name}")
    print("=" * 70)

    print("\n[3/9] Extracting & validating holdings...")
    holdings = _process_holdings(file)
    print(f"      Extracted {len(holdings)} holding(s).")

    print("\n[4/9] Resolving market-data tickers & collecting data...")
    _COLLECTOR = DataCollector()
    for h in holdings:
        h.yf_ticker = _COLLECTOR.resolve_yf_ticker(h)

    # Optional user file: bank NPA/CAR, MF factsheet, sector P/E, red flags...
    if extras_file is not None:
        print("\n[5/9] Merging the extras file (screener/tickertape-style metrics)...")
        try:
            lookup = parse_extras(extras_file)
            n = apply_extras_to_holdings(holdings, lookup)
            print(f"      Extras matched to {n}/{len(holdings)} holding(s).")
        except Exception as e:
            LOG.warning("Extras file could not be parsed (%s) - continuing "
                        "without it; affected rows stay N/A.", e)

    print("\n[6/9] Running per-asset-class analysis engines...")
    orch = AnalysisOrchestrator(_COLLECTOR)
    for i, h in enumerate(holdings, 1):
        orch.analyze(h)
        if i % 5 == 0 or i == len(holdings):
            print(f"      analyzed {i}/{len(holdings)} ...")

    print("\n[7/9] Computing portfolio-level metrics...")
    ps = compute_portfolio(holdings)

    print("\n[8/9] Generating charts and the single comprehensive PDF report...")
    out_dir = os.path.abspath(Config.OUTPUT_DIR)
    os.makedirs(out_dir, exist_ok=True)
    _ALL_CHARTS = generate_charts(holdings, ps, out_dir)

    print("\n[9/9] Generating the report + CSV/Excel companions...")
    pdfs = generate_master_pdf(holdings, ps, _ALL_CHARTS, out_dir)

    print("\nExporting CSV/Excel companions (optional downloads)...")
    tabular = export_tabular(holdings, ps, out_dir)

    return {"holdings": holdings, "summary": ps, "pdfs": pdfs,
            "tabular": tabular, "charts": _ALL_CHARTS,
            "report_pdf": pdfs[0] if pdfs else None,
            "out_dir": out_dir}

def print_execution_summary(result: Dict[str, Any]) -> None:
    ps: PortfolioSummary = result["summary"]
    holdings = result["holdings"]
    unidentified = [h for h in holdings if h.asset_class == CLASS_UNKNOWN]
    print("\n" + "=" * 70)
    print(" EXECUTION SUMMARY")
    print("=" * 70)
    print(f" Holdings uploaded           : {ps.n_holdings}")
    print(f" Successfully identified     : {ps.n_identified}")
    print(f" Analyzed                    : {ps.n_analyzed}")
    print(f" Failed data lookups         : {ps.n_failed_lookups}")
    print(f" Unidentified / ambiguous    : {len(unidentified)}")
    if unidentified:
        print("   -> " + ", ".join(_short(h.raw_name) for h in unidentified[:15]))
    print(f" Total invested              : {fmt_inr(ps.total_invested)}")
    print(f" Current market value        : {fmt_inr(ps.total_current_value)}")
    print(f" Unrealised P/L              : {fmt_inr(ps.unrealized_pnl)} "
          f"({(f'{ps.unrealized_return_pct:.2f}%' if ps.unrealized_return_pct is not None else 'N/A')})")
    print(f" Comprehensive PDF report    : {len(result['pdfs'])} file "
          f"(Portfolio_Analysis_Report.pdf)")
    print(f" Supporting files (CSV/XLSX) : {len(result['tabular'])}")
    print(f" Charts                      : {len(result['charts'])}")
    print(f" Output directory            : {result['out_dir']}")
    counts: Dict[str, int] = {}
    for h in holdings:
        counts[h.assessment] = counts.get(h.assessment, 0) + 1
    if counts:
        print(" Signal distribution (from the rule engines):")
        for k in sorted(counts, key=lambda x: -counts[x]):
            print(f"   - {k}: {counts[k]}")
    ruled = sum(1 for h in holdings if h.analysis.get("rules"))
    print(f" Holdings scored on the 6-step rule card: {ruled} "
          f"(the rest are funds/unidentified - they use their own checks)")
    if ps.valuation_notes:
        print("\n Limitations:")
        for n in ps.valuation_notes:
            print("   - " + n)

def provide_downloads(result: Dict[str, Any]) -> None:
    print("\n" + "=" * 70)
    print(" DOWNLOAD LINKS")
    print("=" * 70)
    pdf = result.get("report_pdf")
    if pdf and os.path.exists(pdf):
        print(f" Single comprehensive PDF report: {pdf} "
              f"({os.path.getsize(pdf):,} bytes)")
    else:
        print("[WARN] The PDF report could not be produced - check the log above.")
    if IS_COLAB:
        try:
            from google.colab import files as cf
            if pdf and os.path.exists(pdf):
                cf.download(pdf)
                print(" [Colab] PDF download triggered. Supporting CSV/Excel files "
                      "are in:", result["out_dir"])
        except Exception as e:
            print(" [Colab] Automatic download unavailable:", e)
            print("         Use the Files panel / files.download manually on:", pdf)
    else:
        print(" Local run: open the folder above to view the report.")

def main() -> Optional[Dict[str, Any]]:
    LOG.info("Portfolio Analyzer starting. Colab=%s", IS_COLAB)
    try:
        ocr_ok = ensure_system_ocr()
        LOG.info("OCR available for scanned PDFs: %s", ocr_ok)
        files = upload_holdings_file()
        chosen = resolve_multiple_uploads(files)
        extras = None
        try:
            extras = upload_extras_file()
        except Exception as e:
            LOG.info("Extras upload skipped (%s) - continuing without it.", e)
        result = run_analysis(chosen, extras)
        print_execution_summary(result)
        provide_downloads(result)
        return result
    except (ValueError, FileNotFoundError) as e:
        LOG.error("Input problem: %s", e)
        print("\nPlease upload a valid holdings file (.xlsx/.xls/.csv/.pdf) and re-run.")
    except Exception as e:
        LOG.error("Unexpected error: %s", e)
        traceback.print_exc()
    return None


# -----------------------------------------------------------------------------
# 16. SELF-TESTS (safe, offline-friendly). Executed only via run_self_tests().
# -----------------------------------------------------------------------------
def make_sample_holdings_df() -> pd.DataFrame:
    """
    A realistic broker-style holdings table for testing (NOT used in live runs).
    Mirrors the columns of an Indian broker 'Combined Holdings Statement' export:
    metadata rows at the top, then a header row, and rows that mix equities,
    ETFs and mutual funds (ISIN prefix INE vs INF drives classification).
    """
    header = ["Symbol", "ISIN", "Sector", "Instrument Type", "Quantity Available",
              "Quantity Discrepant", "Quantity Long Term", "Quantity Pledged (Margin)",
              "Quantity Pledged (Loan)", "Average Price", "Previous Closing Price",
              "Unrealized P&L", "Unrealized P&L Pct.", "Exchange", "Currency"]
    rows = [
        ["HDFCBANK", "INE040A01019", "FINANCIAL SERVICES", "-", 40, 0, 0, 0, 0, 1450.5, 1700.2, 9908.0, 17.1, "NSE", "INR"],
        ["RELIANCE", "INE002A01018", "ENERGY", "-", 10, 0, 0, 0, 0, 2400.0, 2950.0, 5500.0, 22.9, "NSE", "INR"],
        ["TCS", "INE467B01029", "SOFTWARE SERVICES", "-", 5, 0, 0, 0, 0, 3300.0, 3900.0, 3000.0, 18.2, "NSE", "INR"],
        ["SUNPHARMA", "INE044A01036", "PHARMACEUTICALS", "-", 12, 0, 0, 0, 0, 1000.0, 1650.0, 7800.0, 65.0, "NSE", "INR"],
        ["M&M", "INE101A01026", "AUTOMOBILE", "-", 3, 0, 0, 0, 0, 2842.9, 2796.2, -140.1, -1.6, "NSE", "INR"],
        ["NIFTYBEES", "INF204K01E23", "ETF", "-", 100, 0, 0, 0, 0, 250.0, 290.0, 4000.0, 16.0, "NSE", "INR"],
        ["ITBEES", "INF204K01E23", "ETF", "-", 800, 0, 0, 0, 0, 34.11, 31.84, -1816.07, -6.65, "NSE", "INR"],
        ["TATAGOLD-E", "INF002K01B23", "ETF", "-", 400, 0, 0, 0, 0, 14.69, 14.42, -110.14, -1.87, "NSE", "INR"],
        ["HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN", "INF179K01WA6", "-",
         "Hybrid - Dynamic Asset Allocation/Balanced Advantage", 19.166, 0, 0, 0, 0, 553.04, 546.08, -133.31, -1.25, "AMFI", "INR"],
        ["JIOBLACKROCK FLEXI CAP FUND - DIRECT PLAN", "INF22M001093", "-",
         "Equity - Flexi Cap", 146.002, 0, 0, 0, 0, 10.27, 9.51, -110.95, -7.39, "AMFI", "INR"],
        ["TOTALLY UNKNOWN INSTRUMENT", "XYZ123", "", "-", 10, 0, 0, 0, 0, 100.0, 90.0, -100.0, -10.0, "", "INR"],
    ]
    meta = [["Client ID", "TEST123"], ["Combined Holdings Statement as on 2026-10-12"],
            ["Summary"], ["Invested Value", 500000], ["Present Value", 520000]]
    return pd.DataFrame(meta + [header] + rows, dtype=object)

def make_broker_workbook_bytes() -> bytes:
    """Build a multi-sheet broker-style .xlsx in memory (Equity/MF/Combined).
    The Combined sheet repeats both, to verify ISIN-based de-duplication."""
    eq = [
        ["Symbol", "ISIN", "Sector", "Quantity Available", "Quantity Discrepant",
         "Quantity Long Term", "Quantity Pledged (Margin)", "Quantity Pledged (Loan)",
         "Average Price", "Previous Closing Price", "Unrealized P&L", "Unrealized P&L Pct."],
        ["HDFCBANK", "INE040A01019", "FINANCIAL SERVICES", 40, 0, 0, 0, 0, 1450.5, 1700.2, 9908.0, 17.1],
        ["RELIANCE", "INE002A01018", "ENERGY", 10, 0, 0, 0, 0, 2400.0, 2950.0, 5500.0, 22.9],
        ["NIFTYBEES", "INF204K01E23", "ETF", 100, 0, 0, 0, 0, 250.0, 290.0, 4000.0, 16.0],
    ]
    mf = [
        ["Symbol", "ISIN", "Instrument Type", "Quantity Available", "Quantity Discrepant",
         "Quantity Pledged (Margin)", "Quantity Pledged (Loan)", "Average Price",
         "Previous Closing Price", "Unrealized P&L", "Unrealized P&L Pct."],
        ["JIOBLACKROCK FLEXI CAP FUND - DIRECT PLAN", "INF22M001093",
         "Equity - Flexi Cap", 146.002, 0, 0, 0, 10.27, 9.51, -110.95, -7.39],
    ]
    combined = [
        ["Symbol", "ISIN", "Sector", "Instrument Type", "Quantity Available",
         "Quantity Discrepant", "Quantity Long Term", "Quantity Pledged (Margin)",
         "Quantity Pledged (Loan)", "Average Price", "Previous Closing Price",
         "Unrealized P&L", "Unrealized P&L Pct."],
        ["HDFCBANK", "INE040A01019", "FINANCIAL SERVICES", "-", 40, 0, 0, 0, 0, 1450.5, 1700.2, 9908.0, 17.1],
        ["RELIANCE", "INE002A01018", "ENERGY", "-", 10, 0, 0, 0, 0, 2400.0, 2950.0, 5500.0, 22.9],
        ["NIFTYBEES", "INF204K01E23", "ETF", "-", 100, 0, 0, 0, 0, 250.0, 290.0, 4000.0, 16.0],
        ["JIOBLACKROCK FLEXI CAP FUND - DIRECT PLAN", "INF22M001093", "-",
         "Equity - Flexi Cap", 146.002, 0, 0, 0, 0, 10.27, 9.51, -110.95, -7.39],
    ]
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        pd.DataFrame(eq, dtype=object).to_excel(xw, sheet_name="Equity", index=False, header=False)
        pd.DataFrame(mf, dtype=object).to_excel(xw, sheet_name="Mutual Funds", index=False, header=False)
        pd.DataFrame(combined, dtype=object).to_excel(xw, sheet_name="Combined", index=False, header=False)
    return buf.getvalue()

def make_deep_header_workbook_bytes() -> bytes:
    """
    Replica of the REAL Angel/SmartIQ export layout: each sheet starts with a
    client/statement/summary block, so the header row sits at row 14 (Equity)
    and row 12 (Mutual Funds). Regression test for header detection.
    """
    equity_block = [
        ["", "Client ID", "OOX650"],
        ["", "Equity Holdings Statement as on 2026-10-12"],
        ["", "Summary"],
        ["", "Invested Value", 520775.9917],
        ["", "Present Value", 488827.03],
        ["", "Unrealized P&L", -31948.9617],
        ["", "Unrealized P&L Pct.", -6.1349],
    ]
    eq_header = ["", "Symbol", "ISIN", "Sector", "Quantity Available",
                 "Quantity Discrepant", "Quantity Long Term",
                 "Quantity Pledged (Margin)", "Quantity Pledged (Loan)",
                 "Average Price", "Previous Closing Price", "Unrealized P&L",
                 "Unrealized P&L Pct."]
    eq = equity_block + [eq_header] + [
        ["", "HDFCBANK", "INE040A01034", "FINANCIAL SERVICES", 40, 0, 0, 0, 0, 798.635, 707.25, -3655.4, -11.4426],
        ["", "HCLTECH", "INE860A01027", "SOFTWARE SERVICES", 5, 0, 0, 0, 0, 1275.75, 1216.7, -295.25, -4.6286],
        ["", "AUTOIETF", "INF109KC10V2", "ETF", 140, 0, 0, 0, 0, 26.4429, 25.89, -77.4, -2.0908],
    ]
    mf_block = [
        ["", "Client ID", "OOX650"],
        ["", "Mutual Funds Holdings Statement as on 2026-10-12"],
        ["", "Summary"],
        ["", "Invested Value", 79196.0912],
        ["", "Present Value", 78100.5143],
        ["", "Unrealized P&L", -1095.5768],
        ["", "Unrealized P&L Pct.", -1.3834],
    ]
    mf_header = ["", "Symbol", "ISIN", "Instrument Type", "Quantity Available",
                 "Quantity Discrepant", "Quantity Pledged (Margin)",
                 "Quantity Pledged (Loan)", "Average Price",
                 "Previous Closing Price", "Unrealized P&L", "Unrealized P&L Pct."]
    mf = mf_block + [mf_header] + [
        ["", "HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN", "INF179K01WA6",
         "Hybrid - Dynamic Asset Allocation/Balanced Advantage", 19.166, 0, 0, 0, 553.0437, 546.088, -133.312, -1.2577],
        ["", "TATA DIGITAL INDIA FUND - DIRECT PLAN", "INF277K01Z77",
         "Equity - Sectoral/Thematic", 206.783, 0, 0, 0, 48.3574, 46.2504, -435.6877, -4.3571],
    ]
    combined = [
        ["", "Client ID", "OOX650"],
        ["", "Combined Holdings Statement as on 2026-10-12"],
        ["", "Summary"],
        ["", "Invested Value", 599972.0829],
        ["", "Present Value", 566927.5443],
        ["", "Unrealized P&L", -33044.5385],
        ["", "Unrealized P&L Pct.", -5.5077],
    ] + [
        ["", "Symbol", "ISIN", "Sector", "Instrument Type", "Quantity Available",
         "Quantity Discrepant", "Quantity Long Term", "Quantity Pledged (Margin)",
         "Quantity Pledged (Loan)", "Average Price", "Previous Closing Price",
         "Unrealized P&L", "Unrealize P&L Pct."],
        ["", "HDFCBANK", "INE040A01034", "FINANCIAL SERVICES", "-", 40, 0, 0, 0, 0, 798.635, 707.25, -3655.4, -11.4426],
        ["", "HCLTECH", "INE860A01027", "SOFTWARE SERVICES", "-", 5, 0, 0, 0, 0, 1275.75, 1216.7, -295.25, -4.6286],
        ["", "AUTOIETF", "INF109KC10V2", "ETF", "-", 140, 0, 0, 0, 0, 26.4429, 25.89, -77.4, -2.0908],
        ["", "HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN", "INF179K01WA6", "-",
         "Hybrid - Dynamic Asset Allocation/Balanced Advantage", 19.166, 0, "-", 0, 0, 553.0437, 546.088, -133.312, -1.2577],
        ["", "TATA DIGITAL INDIA FUND - DIRECT PLAN", "INF277K01Z77", "-",
         "Equity - Sectoral/Thematic", 206.783, 0, "-", 0, 0, 48.3574, 46.2504, -435.6877, -4.3571],
    ]
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        pd.DataFrame(eq, dtype=object).to_excel(xw, sheet_name="Equity", index=False, header=False)
        pd.DataFrame(mf, dtype=object).to_excel(xw, sheet_name="Mutual Funds", index=False, header=False)
        pd.DataFrame(combined, dtype=object).to_excel(xw, sheet_name="Combined", index=False, header=False)
    return buf.getvalue()

def run_self_tests() -> bool:
    """
    Run offline unit tests over the pipeline with sample data.
    These exercise: column mapping, derivation, classification, calculations,
    PDF + export generation. No live network is required (analysis engines fall
    back to N/A when yfinance data is absent).
    """
    global _COLLECTOR
    print("\n" + "=" * 70)
    print(" RUNNING SELF-TESTS (sample data, offline-safe)")
    print("=" * 70)
    ok = True
    try:
        # 1) to_float parsing
        assert to_float("\u20b91,234.50") == 1234.5
        assert to_float("(500)") == -500
        assert to_float("abc") is None
        assert to_float("") is None
        print(" [PASS] numeric parsing / missing-vs-zero")

        # 2) fmt_inr Indian grouping
        assert fmt_inr(1234567) == "\u20b912,34,567.00", fmt_inr(1234567)
        print(" [PASS] Indian currency formatting")

        # 3) extraction + mapping from a broker-style CSV byte stream
        df = make_sample_holdings_df()
        csv_bytes = df.to_csv(index=False, header=False).encode("utf-8")
        up = UploadedFile("sample.csv", csv_bytes, len(csv_bytes))
        tables = extract_tables(up)
        table = tables[0]
        clean, mapping = map_columns(table.dataframe, has_header=table.has_header)
        assert "symbol" in mapping and "isin" in mapping and "quantity" in mapping
        assert "cmp" in mapping and "sector" in mapping
        print(" [PASS] column auto-mapping:", list(mapping.keys()))

        holdings = build_holdings(clean, mapping, "csv")
        assert len(holdings) == 11, len(holdings)
        print(f" [PASS] built {len(holdings)} holdings")

        # 3b) file type is judged by CONTENT, not by the name
        html_bytes = ("<html><body><table><tr><td>SYMBOL</td><td>ISIN</td></tr>"
                      "<tr><td>TESTCO</td><td>INE000A01000</td></tr></table></body></html>"
                      ).encode("utf-8")
        assert detect_format(UploadedFile("x.html", html_bytes, len(html_bytes))) == "html"
        # Broker-style mislabel: HTML content saved under an .xlsx name.
        mis = UploadedFile("holdings-OOX650 (5).xlsx", html_bytes, len(html_bytes))
        assert detect_format(mis) == "html"
        t_html = extract_tables(mis)
        assert t_html and t_html[0].source_format == "html"
        assert len(t_html[0].dataframe) == 2
        assert detect_format(UploadedFile("junk.csv", bytes(range(200)), 200)) == "unknown"
        assert detect_format(UploadedFile("empty.xlsx", b"", 0)) == "unknown"
        assert detect_format(UploadedFile("a.csv", csv_bytes, len(csv_bytes))) == "csv"
        print(" [PASS] content-based file-type detection (mislabelled/HTML/unknown)")

        # 4) classification coverage - every row accounted for & broker-aware
        for h in holdings:
            classify_holding(h)
            assert h.asset_class in (CLASS_STOCKS, CLASS_BANKS_FI, CLASS_ETF,
                                     CLASS_MF, CLASS_UNKNOWN)
        classes = {h.asset_class for h in holdings}
        by = {h.asset_class for h in holdings}
        assert CLASS_ETF in by and CLASS_MF in by and CLASS_BANKS_FI in by and CLASS_STOCKS in by
        assert CLASS_UNKNOWN in by, "unknown row should be flagged, not dropped"
        assert len(holdings) == 11, "every uploaded row must be accounted for"
        print(" [PASS] asset classification:", by)

        # 5) analysis engines run without network and don't crash
        _COLLECTOR = DataCollector()
        for h in holdings:
            h.yf_ticker = _COLLECTOR.resolve_yf_ticker(h)
        orch = AnalysisOrchestrator(_COLLECTOR)
        for h in holdings:
            orch.analyze(h)
        print(" [PASS] analysis engines executed")

        # 6) portfolio math
        ps = compute_portfolio(holdings)
        # unrealised = MV - invested, and return consistent
        expect = ps.total_current_value - ps.total_invested
        assert abs(ps.unrealized_pnl - expect) < 1e-6
        print(f" [PASS] portfolio totals: invested={fmt_inr(ps.total_invested)}, "
              f"mv={fmt_inr(ps.total_current_value)}")

        # 7) allocation percentages sum ~100 (asset classes)
        tot = sum(ps.asset_allocation.values())
        assert 99.0 <= tot <= 101.0, tot
        print(f" [PASS] allocation sums to {tot:.1f}%")

        # 8) ONE comprehensive PDF + exports produced
        tmp = tempfile.mkdtemp(prefix="pa_test_")
        charts = generate_charts(holdings, ps, tmp)
        _ALL_CHARTS.clear(); _ALL_CHARTS.extend(charts)
        pdfs = generate_master_pdf(holdings, ps, charts, tmp)
        assert len(pdfs) == 1, f"expected exactly 1 PDF, got {len(pdfs)}"
        assert os.path.basename(pdfs[0]) == "Portfolio_Analysis_Report.pdf"
        assert os.path.exists(pdfs[0]) and os.path.getsize(pdfs[0]) > 5000
        assert not any(f.lower().endswith(".zip")
                       for f in os.listdir(os.path.dirname(pdfs[0])))
        files = export_tabular(holdings, ps, tmp)
        assert any(f.endswith("Portfolio_Summary.csv") for f in files)
        assert any(f.endswith("Analysis_Results.xlsx") for f in files)
        print(f" [PASS] produced 1 comprehensive PDF "
              f"({os.path.getsize(pdfs[0]):,} bytes) + {len(files)} tabular file(s), no ZIP")
        shutil.rmtree(tmp, ignore_errors=True)

        # 8b) rule engine on synthetic annual statements (offline, deterministic)
        #     Rupee-scale figures so market cap / FCF ratios are consistent.
        cols = ["FY26", "FY25", "FY24", "FY23", "FY22"]      # most recent first
        fin = pd.DataFrame({
            "Total Revenue": [1.30e11, 1.17e11, 1.05e11, 0.95e11, 0.86e11],
            "Operating Income": [2.60e10, 2.50e10, 2.35e10, 2.20e10, 2.10e10],
            "Reconciled Depreciation": [6.0e9, 5.5e9, 5.0e9, 4.8e9, 4.5e9],
            "Net Income": [1.66e10, 1.73e10, 1.55e10, 1.40e10, 1.30e10],
            "Basic EPS": [61.3, 63.9, 57.3, 51.7, 48.0],
            "Interest Expense": [-1.0e9, -9e8, -8e8, -7e8, -6e8],
        }, index=cols).T
        bsheet = pd.DataFrame({
            "Stockholders Equity": [8.0e10, 7.5e10, 7.0e10, 6.5e10, 6.0e10],
            "Total Debt": [6.0e9, 6.2e9, 6.4e9, 6.6e9, 6.8e9],
        }, index=cols).T
        cflow = pd.DataFrame({
            "Operating Cash Flow": [2.00e10, 2.22e10, 1.90e10, 1.75e10, 1.60e10],
            "Capital Expenditure": [-1.4e9, -1.5e9, -1.2e9, -1.1e9, -1.0e9],
        }, index=cols).T
        fund = compute_fundamentals({"fin": fin, "bs": bsheet, "cf": cflow})
        assert fund["years"] == 5
        assert abs(fund["roe"][0] - 1.66e10 / 8.0e10 * 100) < 1e-6
        assert 20 < fund["roe_avg"] < 26
        assert 28 < fund["roce_avg"] < 33
        assert abs(fund["fcf_last"] - 1.86e10) < 1e-3
        assert abs(fund["de_last"] - 6.0e9 / 8.0e10) < 1e-12
        assert abs(fund["interest_cov"][0] - 26.0) < 1e-6
        assert abs(fund["ebitda_margin"][0] - (2.60e10 + 6.0e9) / 1.30e11 * 100) < 1e-6
        assert 10 < fund["rev_cagr"][0] < 12 and 6 < fund["eps_cagr"][0] < 8
        info = {"marketCap": 3.29e11, "trailingPE": 18.2, "priceToBook": 4.4}
        rules = build_stock_rules(fund, info,
                                  {"last_price": 1217.0, "wk52_high": 1780.0},
                                  is_financial=False)
        assert len(rules["steps"]) == 4
        total = sum(len(rows) for _, rows in rules["steps"])
        assert total == 20, f"the demo card has 20 checks, got {total}"
        assert rules["pass"] + rules["fail"] + rules["watch"] + rules["na"] == total
        # Promoter holding / pledge / promoter selling / red flags stay N/A (no free source)
        assert rules["na"] == 4, rules["na"]
        assert rules["signal"] in Config.RULE_SIGNALS
        # Quality/valuation pass but growth fails (EBITDA & EPS CAGR, profit triangle)
        assert (rules["pass"], rules["fail"], rules["watch"]) == (11, 3, 2), \
            (rules["pass"], rules["fail"], rules["watch"])
        assert rules["signal"] == "HOLD", rules["signal"]
        # Banks/NBFCs: D/E and interest coverage are NOT applicable -> N/A
        bank_rules = build_stock_rules(fund, info, {"last_price": 1217.0},
                                       is_financial=True)
        assert bank_rules["steps"][0][1][3]["signal"] == SIG_NA   # debt to equity
        assert bank_rules["steps"][0][1][4]["signal"] == SIG_NA   # interest coverage
        assert bank_rules["na"] == rules["na"] + 2
        # Missing statements must NOT be guessed into a pass/fail
        thin = build_stock_rules({}, {"marketCap": 5e10}, {"last_price": 10.0}, False)
        assert thin["signal"] == "INSUFFICIENT DATA"
        print(f" [PASS] rule engine: {rules['pass']}/{rules['applicable']} applicable "
              f"checks pass, {rules['fail']} fail, {rules['na']} N/A -> {rules['signal']}")

        # 8c) optional extras file: column matching, parsing, merging, scoring
        assert _extras_match("Gross NPA %", "gross npa")
        assert _extras_match("CAR %", "car")
        assert not _extras_match("Sector P/E", "car")      # whole-word matching
        assert not _extras_match("ROCE %", "roa")
        assert _extras_match("Pledged % (share)", "pledged")
        extras_csv = ("Symbol,Name,Promoter %,Pledged %,Sector P/E,Red Flags,"
                      "Gross NPA %,CAR %,Fund Manager\n"
                      "TESTCO,Test Co Ltd,61.2,0,28.5,1,1.4,15.2,Someone\n"
                      "NSEL,NSDL Ltd,59.9,0.3,30.1,0,,,\n").encode("utf-8")
        ex_up = UploadedFile("extras.csv", extras_csv, len(extras_csv))
        lk = parse_extras(ex_up)
        assert ("TESTCO", "") in lk, list(lk)[:4]
        assert lk[("TESTCO", "")]["gnpa_pct"] == 1.4
        assert lk[("TESTCO", "")]["promoter_pct"] == 61.2
        assert lk[("TESTCO", "")]["fund_manager"] == "Someone"   # text metric
        # Match by symbol; the series suffix (SILVERLINE-X style) is dropped.
        th = Holding(1, "", "", "TESTCO-X", "", None, None, None, None, None,
                     None, None, "", "", "")
        assert apply_extras_to_holdings([th], lk) == 1
        assert th.extras["red_flags"] == 1
        # Extras merge: fill N/A metrics only, never overwrite a computed value.
        th2 = Holding(2, "", "", "TESTCO", "", None, None, None, None, None,
                      None, None, "", "", "")
        th2.asset_class = CLASS_BANKS_FI
        apply_extras_to_holdings([th2], lk)
        eng = BankNBFCEngine(DataCollector())
        mm = {"Gross NPA %": Config.NA, "Capital Adequacy (CAR) %": Config.NA,
              "P/E": 18.2, "Promoter Holding %": Config.NA}
        ii = {}
        eng._apply_extras(th2, mm, ii)
        assert mm["Gross NPA %"] == 1.4 and mm["Capital Adequacy (CAR) %"] == 15.2
        assert mm["P/E"] == 18.2                       # computed value untouched
        assert mm["Promoter Holding %"] == 61.2
        assert ii["_sector_pe"] == 28.5 and ii["_red_flags"] == 1
        assert ii["_promoter_pct"] == 61.2
        assert "Gross NPA %" in eng._source_note(th2, ii)
        # Previously-N/A rule rows become real PASS/FAIL when Screener/extras
        # supply promoter %, pledge %, sector P/E and red-flag counts.
        info2 = dict(info)
        info2.update({"_promoter_pct": 51.0, "_pledged_pct": 0.0,
                      "_sector_pe": 20.0, "_red_flags": 0, "_promoter_selling": "no"})
        r2 = build_stock_rules(fund, info2, {"last_price": 1217.0},
                               is_financial=False)
        assert r2["na"] == rules["na"] - 4, (r2["na"], rules["na"])
        assert r2["applicable"] == rules["applicable"] + 4
        s4rows = r2["steps"][3][1]
        assert s4rows[0]["signal"] == SIG_PASS                       # P/E 18.2 vs sector 20
        assert s4rows[3]["signal"] == SIG_PASS                       # promoter selling
        assert r2["steps"][0][1][1]["signal"] == SIG_PASS            # promoter > 50%
        assert r2["steps"][0][1][2]["signal"] == SIG_PASS            # pledge < 15%
        # A bad upload must degrade gracefully, not raise.
        junk = UploadedFile("j.xlsx", b"PK\x03\x04garbage", 12)
        try:
            parse_extras(junk)
            junk_failed = False
        except Exception:
            junk_failed = True
        assert junk_failed or parse_extras(junk) == {}
        print(" [PASS] extras upload: column match, merge into metrics, rule rows scored")
        shutil.rmtree(tmp, ignore_errors=True)

        # 9) multi-sheet broker workbook + ISIN-based de-duplication
        xls_bytes = make_broker_workbook_bytes()
        wb_up = UploadedFile("broker.xlsx", xls_bytes, len(xls_bytes))
        wb_holdings = _process_holdings(wb_up)
        # Equity(3) + MF(1) are kept; Combined(4) duplicates are removed by ISIN.
        assert len(wb_holdings) == 4, f"expected 4 after dedup, got {len(wb_holdings)}"
        isins = [h.ISIN for h in wb_holdings]
        assert len(isins) == len(set(isins)), "duplicate ISINs should be de-duplicated"
        mf = [h for h in wb_holdings if h.asset_class == CLASS_MF]
        etfs = [h for h in wb_holdings if h.asset_class == CLASS_ETF]
        assert len(mf) == 1 and len(etfs) == 1
        print(" [PASS] multi-sheet read + ISIN de-duplication (4 unique holdings)")

        # 9b) REAL broker layout: header far down each sheet (row 14/12) + dedup
        deep_bytes = make_deep_header_workbook_bytes()
        deep_up = UploadedFile("holdings-OOX650 (5).xlsx", deep_bytes, len(deep_bytes))
        deep_holdings = _process_holdings(deep_up)
        # Equity(3) + MF(2) kept; Combined(5) repeats all of them by ISIN.
        assert len(deep_holdings) == 5, f"expected 5 after dedup, got {len(deep_holdings)}"
        syms = {h.symbol for h in deep_holdings}
        assert syms == {"HDFCBANK", "HCLTECH", "AUTOIETF",
                        "HDFC BALANCED ADVANTAGE FUND - DIRECT PLAN",
                        "TATA DIGITAL INDIA FUND - DIRECT PLAN"}, syms
        hdfc = next(h for h in deep_holdings if h.symbol == "HDFCBANK")
        assert hdfc.quantity == 40 and abs(hdfc.avg_price - 798.635) < 1e-6
        assert hdfc.invested is not None and hdfc.current_value is not None
        classes = {h.asset_class for h in deep_holdings}
        assert CLASS_ETF in classes and CLASS_MF in classes
        print(" [PASS] deep-header broker sheet parsed (header found below summary)")

    except AssertionError as e:
        ok = False
        print(f" [FAIL] assertion: {e}")
        traceback.print_exc()
    except Exception as e:
        ok = False
        print(f" [FAIL] unexpected: {e}")
        traceback.print_exc()
    print("\n Self-test result:", "ALL PASSED" if ok else "FAILURES PRESENT")
    print(" NOTE: these tests are executed offline. Live yfinance/API integration")
    print("       and full acceptance criteria are NOT asserted as tested here.")
    print("=" * 70)
    return ok


# -----------------------------------------------------------------------------
# ENTRY POINT
# -----------------------------------------------------------------------------
def _in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return "ipykernel" in str(get_ipython()) or "ZMQShellCommunication" in str(get_ipython())
    except Exception:
        return False

if __name__ == "__main__":
    # When executed as a script/notebook cell, run the full interactive workflow.
    main()
elif _in_notebook():
    # In Colab/Notebook, `import`-style execution still wants to run immediately.
    main()


[setup] Installing 3 missing package(s): pypdf, pdfplumber, pytesseract
06:28:53 | INFO    | Portfolio Analyzer starting. Colab=True


INFO:PortfolioAnalyzer:Portfolio Analyzer starting. Colab=True


06:28:53 | INFO    | OCR available for scanned PDFs: False


INFO:PortfolioAnalyzer:OCR available for scanned PDFs: False



 STEP 1 / 9 : UPLOAD YOUR HOLDINGS FILE
Supported formats: .xlsx, .xls, .csv, .pdf
(The file type is checked by CONTENT, so a broker export whose name is
 mislabeled - e.g. an .xlsx that is really a table - is still accepted.)


...

INFO:PortfolioAnalyzer:Generated 1 comprehensive PDF with 72 holding section(s).



Exporting CSV/Excel companions (optional downloads)...

 EXECUTION SUMMARY
 Holdings uploaded           : 72
 Successfully identified     : 72
 Analyzed                    : 72
 Failed data lookups         : 6
 Unidentified / ambiguous    : 0
 Total invested              : ₹5,54,524.50
 Current market value        : ₹5,53,535.35
 Unrealised P/L              : -₹989.15 (-0.18%)
 Comprehensive PDF report    : 1 file (Portfolio_Analysis_Report.pdf)
 Supporting files (CSV/XLSX) : 5
 Charts                      : 4
 Output directory            : /content/Portfolio_Output
 Signal distribution (from the rule engines):
   - HOLD: 38
   - SELL: 22
   - WATCHLIST: 4
   - REVIEW: 4
   - INSUFFICIENT DATA: 2
   - ACCUMULATE: 2
 Holdings scored on the 6-step rule card: 52 (the rest are funds/unidentified - they use their own checks)

 DOWNLOAD LINKS
 Single comprehensive PDF report: /content/Portfolio_Output/Portfolio_Analysis_Report.pdf (952,968 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 [Colab] PDF download triggered. Supporting CSV/Excel files are in: /content/Portfolio_Output
